#빠른 판관비 확인을 위한 표

In [1]:
# Windows PowerShell 또는 CMD 열고
!pip install pandas openpyxl

  Using cached pandas-3.0.0-cp314-cp314-win_amd64.whl.metadata (19 kB)
  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached numpy-2.4.1-cp314-cp314-win_amd64.whl.metadata (6.6 kB)
  Using cached tzdata-2025.3-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl.metadata (2.7 kB)
Using cached pandas-3.0.0-cp314-cp314-win_amd64.whl (9.9 MB)
Using cached openpyxl-3.1.5-py2.py3-none-any.whl (250 kB)
Using cached numpy-2.4.1-cp314-cp314-win_amd64.whl (12.4 MB)
Using cached et_xmlfile-2.0.0-py3-none-any.whl (18 kB)
Using cached tzdata-2025.3-py2.py3-none-any.whl (348 kB)

   ---------------------------------------- 0/5 [tzdata]
   ---------------------------------------- 0/5 [tzdata]
   ---------------------------------------- 0/5 [tzdata]
   -------- ------------------------------- 1/5 [numpy]
   -------- ------------------------------- 1/5 [numpy]
   -------- ------------------------------- 1/5 [numpy]
   -------- ----------

# 데이터 추출 및 분석

##Target Period 에 해당 분기를 입력하세요. 

##⭐ 분석 대상 분기 선택 (예: "20254Q", "20253Q", "20252Q", "20251Q")
##target_period = "20254Q"  # 원하는 분기로 변경하세요!
    

In [6]:
import pandas as pd
from pathlib import Path
import openpyxl
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter

# ==================== Rev_Account 순서 설정 ====================
REV_ACCOUNT_ORDER = [
    '매출',
    '매출원가',
    '인건비',
    '복리후생비',
    '광고선전비',
    '지급수수료',
    '운반비',
    '경상연구개발비',
    '판매수수료',
    '여비교통비',
    '대손상각비',
    '감가상각비',
    '사용권자산상각비',
    '기타',
    '영업외수익',
    '영업외비용',
    '금융수익',
    '금융비용',
    '지분법손익',
    '법인세비용',
    '세후중단영업손익',
    '당기순이익',
    '포괄손익',
    '총포괄손익',
]

# ==================== Entity 순서 설정 ====================
ENTITY_ORDER = [
    'HQ',
    'USA',
    'Japan',
    'China',
    'Europe',
    'Asia',
    'India',
    'Mexico',
    'Oceania',
    'BWA',
    'Vietnam',
    'Turkey',
    'KOROT',
    '헬스케어',
    '삼한정공',
    'KOCP',
    '연결조정',
]

def sort_rev_accounts(accounts):
    """
    Rev_Account를 정의된 순서대로 정렬
    """
    sorted_accounts = []
    
    # 정의된 순서대로 추가
    for account in REV_ACCOUNT_ORDER:
        if account in accounts:
            sorted_accounts.append(account)
    
    # 정의되지 않은 계정은 마지막에 알파벳순으로 추가
    remaining = sorted([acc for acc in accounts if acc not in REV_ACCOUNT_ORDER])
    sorted_accounts.extend(remaining)
    
    return sorted_accounts

def sort_entities(entities):
    """
    Entity를 정의된 순서대로 정렬
    """
    sorted_entities = []
    
    # 정의된 순서대로 추가
    for entity in ENTITY_ORDER:
        if entity in entities:
            sorted_entities.append(entity)
    
    # 정의되지 않은 Entity는 마지막에 알파벳순으로 추가
    remaining = sorted([ent for ent in entities if ent not in ENTITY_ORDER])
    sorted_entities.extend(remaining)
    
    return sorted_entities

# ==================== Entity 이름 매핑 설정 ====================
ENTITY_NAME_MAPPING = {
    'BIOSPACE LATIN AMERICA S DE RL DE CV(*)': 'Mexico',
    'BWA': 'BWA',
    'Biospace Co.,Ltd.': 'China',
    'Biospace Inc. DBA INBODY': 'USA',
    'InBody India Pvt. Ltd': 'India',
    'InBody Japan Inc.': 'Japan',
    'InBody Oceania ': 'Oceania',
    '㈜코르트': 'KOROT',
    'Inbody Europe B.V.': 'Europe',
    '㈜인바디헬스케어': '헬스케어',
    '(주)삼한정공': '삼한정공',
    'INBODY TURKEY MEDİKAL TİCARET LİMİTED ŞİRKETİ': 'Turkey',
    'INBODY ASIA SDN. BHD.': 'Asia',
    '주식회사 인바디': 'HQ',
    '케이오씨피 프로젝트 제2호 벤처투자조합': 'KOCP',
    'InBody Vietnam': 'Vietnam',
    'ADJ': '연결조정',
}

def rename_entity(original_name):
    """
    Entity 이름을 짧은 이름으로 변환
    매핑에 없으면 원본 이름 반환
    """
    return ENTITY_NAME_MAPPING.get(original_name, original_name)

# ================================================================

def parse_period(period_str):
    """Period 문자열 파싱"""
    # 문자열이 아니거나 비어있는 경우 처리
    if not period_str or not isinstance(period_str, str):
        return None, None
    
    # "전체", "합계" 등 특수 period는 None 반환
    if period_str in ['전체', '합계', 'Total']:
        return None, None
    
    # 일반적인 period 형식이 아닌 경우 (길이가 5가 아닌 경우)
    if len(period_str) < 5:
        return None, None
    
    try:
        year = int(period_str[:4])
        quarter = int(period_str[4])
        return year, quarter
    except (ValueError, IndexError):
        return None, None

def get_previous_quarter(period_str):
    """이전 분기 계산"""
    year, quarter = parse_period(period_str)
    if year is None or quarter is None:
        return None
    if quarter == 1:
        return f"{year-1}4Q"
    else:
        return f"{year}{quarter-1}Q"

def get_yoy_quarter(period_str):
    """전년 동분기 계산"""
    year, quarter = parse_period(period_str)
    if year is None or quarter is None:
        return None
    return f"{year-1}{quarter}Q"

def format_period_display(period_str):
    """Period 포맷팅"""
    if not period_str:
        return "Unknown"
    year, quarter = parse_period(period_str)
    if year is None or quarter is None:
        return period_str
    return f"{year}_{quarter}Q"

def check_missing_exchange_rates(merged_pl_file):
    """
    환율이 누락된 데이터를 확인하고 경고 (자동 수정 안 함!)
    """
    print("\n" + "="*60)
    print("[환율 체크] Exchange_Rate 누락 확인")
    print("="*60)
    
    df = pd.read_excel(merged_pl_file)
    
    # Exchange_Rate가 0이거나 NaN인 행 찾기 (KRW 제외)
    non_krw_mask = df['Currency'] != 'KRW'
    missing_rate_mask = ((df['Exchange_Rate'].isna()) | (df['Exchange_Rate'] == 0)) & non_krw_mask
    missing_count = missing_rate_mask.sum()
    
    if missing_count == 0:
        print("✓ 모든 외화 데이터에 환율이 설정되어 있습니다.")
        return df
    
    print(f"\n⚠️  환율 누락 발견: {missing_count}개 행")
    
    # 통화별 누락 현황
    missing_df = df[missing_rate_mask]
    currency_counts = missing_df.groupby('Currency').size()
    
    print("\n통화별 누락 현황:")
    for currency, count in currency_counts.items():
        print(f"  ❌ {currency}: {count}개 행 - Exchange_Rate 누락!")
    
    # Entity별 누락 현황
    print("\nEntity별 누락 현황:")
    entity_currency = missing_df.groupby(['Entity', 'Currency']).size()
    for (entity, currency), count in entity_currency.items():
        print(f"  - {entity} ({currency}): {count}개 행")
    
    print("\n" + "="*60)
    print("⚠️  환율 누락 데이터가 있습니다!")
    print("="*60)
    print("\n해결 방법:")
    print("1. 원본 파일(final_financial_data_*.xlsx)에서 Exchange_Rate 컬럼 확인")
    print("2. 재무팀에 정확한 환율 문의")
    print("3. 각 통화별로 적절한 환율 입력")
    print(f"4. 스크립트 재실행")
    
    print("\n참고 - 환율 입력 위치:")
    for currency in currency_counts.keys():
        print(f"  - {currency}: 원본 파일의 PL 시트 > Exchange_Rate 컬럼")
    
    return df

def concat_pl_sheets(folder_path, output_file, file_pattern='final_financial_data'):
    """PL 시트 병합"""
    print("="*60)
    print("[Step 1] PL 시트 병합 시작")
    print("="*60)
    
    folder = Path(folder_path)
    all_files = list(folder.glob('*.xlsx'))
    all_files = [f for f in all_files if not f.name.startswith('~')]
    excel_files = [f for f in all_files if file_pattern in f.name]
    
    print(f"\n파일명에 '{file_pattern}' 포함된 파일 {len(excel_files)}개 발견:")
    for f in sorted(excel_files):
        print(f"  - {f.name}")
    
    if not excel_files:
        print("\n❌ 처리할 파일이 없습니다.")
        return None
    
    pl_sheets = []
    
    for file_path in sorted(excel_files):
        try:
            xls = pd.ExcelFile(file_path)
            if 'PL' in xls.sheet_names:
                df = pd.read_excel(file_path, sheet_name='PL')
                pl_sheets.append(df)
                print(f"✓ {file_path.name} - PL 시트 로드 완료 ({len(df)} 행)")
            else:
                print(f"⚠️  {file_path.name} - PL 시트 없음")
        except Exception as e:
            print(f"✗ {file_path.name} - 처리 실패: {str(e)}")
    
    if pl_sheets:
        print(f"\n병합 중...")
        merged_df = pd.concat(pl_sheets, ignore_index=True)
        print(f"\n✓ 병합 완료!")
        print(f"  - 병합된 시트 개수: {len(pl_sheets)}")
        print(f"  - 총 데이터 행 수: {len(merged_df)}")
        merged_df.to_excel(output_file, index=False)
        print(f"\n✓ 저장 완료: {output_file}")
        return merged_df
    else:
        print(f"\n❌ 병합할 PL 시트가 없습니다.")
        return None

def create_pivot_sheet(merged_pl_file):
    """Pivot 시트 생성 (Entity 이름 변경 적용)"""
    print("\n" + "="*60)
    print("[Step 2] Pivot 시트 생성 시작")
    print("="*60)
    
    print(f"\n파일 로드 중: {merged_pl_file}")
    df = pd.read_excel(merged_pl_file)
    print(f"✓ 데이터 로드 완료: {len(df)} 행")
    
    required_cols = ['Entity', 'Period', 'Rev_Account', 'Amount(KRW)']
    missing_cols = [col for col in required_cols if col not in df.columns]
    
    if missing_cols:
        print(f"\n❌ 필요한 컬럼이 없습니다: {missing_cols}")
        return None
    
    # Entity 이름 변경
    print(f"\n✓ Entity 이름 변경 적용 중...")
    df['Entity'] = df['Entity'].apply(rename_entity)
    
    print(f"\n✓ Period 유니크 값: {sorted(df['Period'].unique())}")
    print(f"✓ Entity 유니크 값: {df['Entity'].nunique()}개")
    
    print(f"\n피벗테이블 생성 중...")
    pivot_df = df.pivot_table(
        index='Rev_Account',
        columns=['Period', 'Entity'],
        values='Amount(KRW)',
        aggfunc='sum',
        fill_value=0
    )
    
    print(f"✓ 피벗테이블 생성 완료")
    
    for period in pivot_df.columns.get_level_values('Period').unique():
        period_cols = [col for col in pivot_df.columns if col[0] == period]
        pivot_df[(period, '합계')] = pivot_df[period_cols].sum(axis=1)
    
    pivot_df[('전체', '합계')] = pivot_df.sum(axis=1)
    pivot_df = pivot_df.reset_index()
    
    print(f"\n저장 중... (Pivot 시트)")
    
    try:
        wb = openpyxl.load_workbook(merged_pl_file)
        if 'Pivot' in wb.sheetnames:
            del wb['Pivot']
        ws = wb.create_sheet('Pivot')
    except FileNotFoundError:
        wb = openpyxl.Workbook()
        ws = wb.active
        ws.title = 'Pivot'
    
    periods = []
    entities_by_period = {}
    
    for col in pivot_df.columns:
        if col == 'Rev_Account':
            continue
        period, entity = col
        if period not in periods:
            periods.append(period)
            entities_by_period[period] = []
        if entity not in entities_by_period[period]:
            entities_by_period[period].append(entity)
    
    # ⭐ Entity 순서 정렬
    for period in periods:
        entities_by_period[period] = sort_entities(entities_by_period[period])
    
    # ⭐ Rev_Account 순서 정렬
    rev_accounts = sort_rev_accounts(pivot_df['Rev_Account'].tolist())
    
    header_fill = PatternFill(start_color='4472C4', end_color='4472C4', fill_type='solid')
    header_font = Font(bold=True, color='FFFFFF')
    subheader_fill = PatternFill(start_color='D9E1F2', end_color='D9E1F2', fill_type='solid')
    subheader_font = Font(bold=True)
    period_fill_acc = PatternFill(start_color='4472C4', end_color='4472C4', fill_type='solid')
    period_fill_qtd = PatternFill(start_color='70AD47', end_color='70AD47', fill_type='solid')
    period_font = Font(bold=True, size=12, color='FFFFFF')
    
    current_row = 1
    prev_period_data = None
    
    for period_idx, period in enumerate(periods):
        entities = entities_by_period[period]
        
        # 누적 섹션
        ws.merge_cells(start_row=current_row, start_column=1, end_row=current_row, end_column=len(entities) + 2)
        period_cell = ws.cell(row=current_row, column=1)
        period_cell.value = f"{period} (누적)"
        period_cell.fill = period_fill_acc
        period_cell.font = period_font
        period_cell.alignment = Alignment(horizontal='center', vertical='center')
        current_row += 1
        
        ws.cell(row=current_row, column=1, value='Rev_Account').fill = header_fill
        ws.cell(row=current_row, column=1).font = header_font
        ws.cell(row=current_row, column=1).alignment = Alignment(horizontal='center', vertical='center')
        
        for col_idx, entity in enumerate(entities, start=2):
            cell = ws.cell(row=current_row, column=col_idx)
            cell.value = entity
            cell.fill = subheader_fill
            cell.font = subheader_font
            cell.alignment = Alignment(horizontal='center', vertical='center')
        current_row += 1
        
        current_period_data = {}
        
        for rev_account in rev_accounts:
            ws.cell(row=current_row, column=1, value=rev_account)
            
            for col_idx, entity in enumerate(entities, start=2):
                col_tuple = (period, entity)
                row_data = pivot_df[pivot_df['Rev_Account'] == rev_account]
                if not row_data.empty and col_tuple in pivot_df.columns:
                    value = row_data.iloc[0][col_tuple]
                    ws.cell(row=current_row, column=col_idx, value=value)
                    
                    if rev_account not in current_period_data:
                        current_period_data[rev_account] = {}
                    current_period_data[rev_account][entity] = value
                else:
                    ws.cell(row=current_row, column=col_idx, value=0)
                    if rev_account not in current_period_data:
                        current_period_data[rev_account] = {}
                    current_period_data[rev_account][entity] = 0
            
            current_row += 1
        
        # 분기별 섹션
        if period_idx > 0 and prev_period_data is not None:
            current_row += 1
            
            # ⭐ 1Q 체크: 1Q는 분기별 = 누적
            year, quarter = parse_period(period)
            
            # period가 "전체" 등 특수 값이면 분기별 섹션 생성 안 함
            if year is None or quarter is None:
                prev_period_data = current_period_data
                continue
            
            is_first_quarter = (quarter == 1)
            
            ws.merge_cells(start_row=current_row, start_column=1, end_row=current_row, end_column=len(entities) + 2)
            period_cell = ws.cell(row=current_row, column=1)
            period_cell.value = f"{period} (분기별)"
            period_cell.fill = period_fill_qtd
            period_cell.font = period_font
            period_cell.alignment = Alignment(horizontal='center', vertical='center')
            current_row += 1
            
            ws.cell(row=current_row, column=1, value='Rev_Account').fill = header_fill
            ws.cell(row=current_row, column=1).font = header_font
            ws.cell(row=current_row, column=1).alignment = Alignment(horizontal='center', vertical='center')
            
            for col_idx, entity in enumerate(entities, start=2):
                cell = ws.cell(row=current_row, column=col_idx)
                cell.value = entity
                cell.fill = subheader_fill
                cell.font = subheader_font
                cell.alignment = Alignment(horizontal='center', vertical='center')
            current_row += 1
            
            for rev_account in rev_accounts:
                ws.cell(row=current_row, column=1, value=rev_account)
                
                for col_idx, entity in enumerate(entities, start=2):
                    current_value = current_period_data.get(rev_account, {}).get(entity, 0)
                    
                    if is_first_quarter:
                        # ⭐ 1Q: 분기별 = 누적
                        qtd_value = current_value
                    else:
                        # 2Q, 3Q, 4Q: 현재 누적 - 이전 누적
                        prev_value = prev_period_data.get(rev_account, {}).get(entity, 0)
                        qtd_value = current_value - prev_value
                    
                    ws.cell(row=current_row, column=col_idx, value=qtd_value)
                
                current_row += 1
        
        prev_period_data = current_period_data
        
        if period_idx < len(periods) - 1:
            current_row += 2
    
    ws.column_dimensions['A'].width = 25
    max_entities = max(len(entities_by_period[p]) for p in periods)
    for col_idx in range(2, max_entities + 2):
        ws.column_dimensions[get_column_letter(col_idx)].width = 18
    
    wb.save(merged_pl_file)
    
    total_sections = len(periods) + (len(periods) - 1)
    print(f"✓ Pivot 시트 추가 완료 (총 {total_sections}개 섹션)")
    
    return pivot_df

def create_sga_sheet(merged_pl_file):
    """
    SG&A 시트 생성 (매출총이익, 판관비, 영업이익 계산)
    """
    print("\n" + "="*60)
    print("[Step 3] SG&A 시트 생성 시작")
    print("="*60)
    
    wb = openpyxl.load_workbook(merged_pl_file)
    
    if 'Pivot' not in wb.sheetnames:
        print("❌ Pivot 시트가 없습니다.")
        return None
    
    pivot_ws = wb['Pivot']
    
    if 'SG&A' in wb.sheetnames:
        del wb['SG&A']
    sga_ws = wb.create_sheet('SG&A')
    
    target_accounts = [
        '매출', '매출원가', '매출총이익', '판관비',
        '인건비', '복리후생비', '광고선전비', '지급수수료', '운반비',
        '경상연구개발비', '판매수수료', '여비교통비','대손상각비','감가상각비', '사용권자산상각비','기타', '영업이익'
    ]
    
    calculated_accounts = ['매출총이익', '판관비', '영업이익']
    
    header_fill = PatternFill(start_color='4472C4', end_color='4472C4', fill_type='solid')
    header_font = Font(bold=True, color='FFFFFF')
    subheader_fill = PatternFill(start_color='D9E1F2', end_color='D9E1F2', fill_type='solid')
    subheader_font = Font(bold=True)
    period_fill_acc = PatternFill(start_color='4472C4', end_color='4472C4', fill_type='solid')
    period_fill_qtd = PatternFill(start_color='70AD47', end_color='70AD47', fill_type='solid')
    period_font = Font(bold=True, size=12, color='FFFFFF')
    calc_fill = PatternFill(start_color='FFF2CC', end_color='FFF2CC', fill_type='solid')
    calc_font = Font(bold=True)
    
    current_row = 1
    sga_row = 1
    max_row = pivot_ws.max_row
    max_col = pivot_ws.max_column
    section_count = 0
    
    while current_row <= max_row:
        cell = pivot_ws.cell(row=current_row, column=1)
        
        if cell.value and isinstance(cell.value, str) and ('누적' in cell.value or '분기별' in cell.value):
            section_count += 1
            period_header = cell.value
            is_qtd = '분기별' in period_header
            
            sga_ws.merge_cells(start_row=sga_row, start_column=1, end_row=sga_row, end_column=max_col)
            sga_cell = sga_ws.cell(row=sga_row, column=1)
            sga_cell.value = period_header
            sga_cell.fill = period_fill_qtd if is_qtd else period_fill_acc
            sga_cell.font = period_font
            sga_cell.alignment = Alignment(horizontal='center', vertical='center')
            sga_row += 1
            current_row += 1
            
            for col in range(1, max_col + 1):
                src_cell = pivot_ws.cell(row=current_row, column=col)
                dst_cell = sga_ws.cell(row=sga_row, column=col)
                dst_cell.value = src_cell.value
                if col == 1:
                    dst_cell.fill = header_fill
                    dst_cell.font = header_font
                else:
                    dst_cell.fill = subheader_fill
                    dst_cell.font = subheader_font
                dst_cell.alignment = Alignment(horizontal='center', vertical='center')
            
            sga_row += 1
            current_row += 1
            
            section_data = {}
            while current_row <= max_row:
                rev_account_cell = pivot_ws.cell(row=current_row, column=1)
                rev_account = rev_account_cell.value
                
                if not rev_account or (isinstance(rev_account, str) and ('누적' in rev_account or '분기별' in rev_account)):
                    break
                
                row_data = [rev_account]
                for col in range(2, max_col + 1):
                    value = pivot_ws.cell(row=current_row, column=col).value
                    row_data.append(value if value is not None else 0)
                
                section_data[rev_account] = row_data
                current_row += 1
            
            for account in target_accounts:
                if account in calculated_accounts:
                    if account == '매출총이익':
                        if '매출' in section_data and '매출원가' in section_data:
                            sga_ws.cell(row=sga_row, column=1, value=account)
                            sga_ws.cell(row=sga_row, column=1).fill = calc_fill
                            sga_ws.cell(row=sga_row, column=1).font = calc_font
                            for col_idx in range(2, max_col + 1):
                                revenue = section_data['매출'][col_idx - 1]
                                cogs = section_data['매출원가'][col_idx - 1]
                                
                                try:
                                    revenue_val = float(revenue) if revenue and revenue != '' else 0
                                except (ValueError, TypeError):
                                    revenue_val = 0
                                
                                try:
                                    cogs_val = float(cogs) if cogs and cogs != '' else 0
                                except (ValueError, TypeError):
                                    cogs_val = 0
                                
                                gross_profit = revenue_val - cogs_val
                                sga_ws.cell(row=sga_row, column=col_idx, value=gross_profit).fill = calc_fill
                            sga_row += 1
                    
                    elif account == '판관비':
                        sga_components = [ '인건비', '복리후생비', '광고선전비', '지급수수료', '운반비', 
                        '경상연구개발비', '판매수수료', '여비교통비','대손상각비','감가상각비', '사용권자산상각비','기타']
                        sga_ws.cell(row=sga_row, column=1, value=account)
                        sga_ws.cell(row=sga_row, column=1).fill = calc_fill
                        sga_ws.cell(row=sga_row, column=1).font = calc_font
                        for col_idx in range(2, max_col + 1):
                            sga_total = 0
                            for c in sga_components:
                                if c in section_data and section_data[c][col_idx - 1]:
                                    try:
                                        value = float(section_data[c][col_idx - 1])
                                        sga_total += value
                                    except (ValueError, TypeError):
                                        pass
                            sga_ws.cell(row=sga_row, column=col_idx, value=sga_total).fill = calc_fill
                        sga_row += 1
                    
                    elif account == '영업이익':
                        sga_ws.cell(row=sga_row, column=1, value=account)
                        sga_ws.cell(row=sga_row, column=1).fill = calc_fill
                        sga_ws.cell(row=sga_row, column=1).font = calc_font
                        gross_profit_row = None
                        sga_expense_row = None
                        for search_row in range(sga_row - 1, 0, -1):
                            cell_value = sga_ws.cell(row=search_row, column=1).value
                            if cell_value == '매출총이익':
                                gross_profit_row = search_row
                            elif cell_value == '판관비':
                                sga_expense_row = search_row
                            if gross_profit_row and sga_expense_row:
                                break
                        if gross_profit_row and sga_expense_row:
                            for col_idx in range(2, max_col + 1):
                                gp = sga_ws.cell(row=gross_profit_row, column=col_idx).value
                                sga_exp = sga_ws.cell(row=sga_expense_row, column=col_idx).value
                                
                                try:
                                    gp_val = float(gp) if gp and gp != '' else 0
                                except (ValueError, TypeError):
                                    gp_val = 0
                                
                                try:
                                    sga_val = float(sga_exp) if sga_exp and sga_exp != '' else 0
                                except (ValueError, TypeError):
                                    sga_val = 0
                                
                                operating_income = gp_val - sga_val
                                sga_ws.cell(row=sga_row, column=col_idx, value=operating_income).fill = calc_fill
                        sga_row += 1
                else:
                    if account in section_data:
                        row_data = section_data[account]
                        for col_idx, value in enumerate(row_data, start=1):
                            sga_ws.cell(row=sga_row, column=col_idx, value=value)
                        sga_row += 1
            
            if current_row < max_row:
                while current_row <= max_row and not pivot_ws.cell(row=current_row, column=1).value:
                    sga_row += 1
                    current_row += 1
        else:
            current_row += 1
    
    sga_ws.column_dimensions['A'].width = 25
    for col_idx in range(2, max_col + 1):
        sga_ws.column_dimensions[get_column_letter(col_idx)].width = 18
    
    wb.save(merged_pl_file)
    
    print(f"✓ SG&A 시트 생성 완료 ({section_count}개 섹션)")
    
    return True

def create_analysis_horizontal(merged_pl_file, target_period="20254Q"):
    """
    Analysis 시트 생성 - 가로 배치 (QoQ | YoY(Q) | YoY(Y))
    전체 합계 + 각 Entity별
    ⭐ Pivot 시트 데이터 기준 (모든 계정)
    """
    print("\n" + "="*60)
    print(f"[Step 4] Analysis 시트 생성 (가로 배치)")
    print("="*60)
    
    wb = openpyxl.load_workbook(merged_pl_file)
    
    if 'Pivot' not in wb.sheetnames:
        print("❌ Pivot 시트가 없습니다.")
        return None
    
    pivot_ws = wb['Pivot']
    
    if 'Analysis' in wb.sheetnames:
        del wb['Analysis']
    analysis_ws = wb.create_sheet('Analysis')
    
    # 비교 분기 계산
    prev_quarter = get_previous_quarter(target_period)
    yoy_quarter = get_yoy_quarter(target_period)
    
    target_display = format_period_display(target_period)
    prev_display = format_period_display(prev_quarter)
    yoy_display = format_period_display(yoy_quarter)
    
    print(f"\n분석 분기:")
    print(f"  - QoQ: {prev_display} → {target_display}")
    print(f"  - YoY(Q): {yoy_display} → {target_display}")
    print(f"  - YoY(Y): {yoy_display}(누적) → {target_display}(누적)")
    
    # 스타일
    header_fill = PatternFill(start_color='4472C4', end_color='4472C4', fill_type='solid')
    header_font = Font(bold=True, color='FFFFFF')
    section_fill = PatternFill(start_color='FFC000', end_color='FFC000', fill_type='solid')
    section_font = Font(bold=True, size=11, color='FFFFFF')
    increase_fill = PatternFill(start_color='C6EFCE', end_color='C6EFCE', fill_type='solid')
    decrease_fill = PatternFill(start_color='FFC7CE', end_color='FFC7CE', fill_type='solid')
    
    # Entity 목록 (Pivot 시트에서)
    entities = []
    search_pattern = f"{target_period} (누적)"
    for row in range(1, pivot_ws.max_row + 1):
        cell = pivot_ws.cell(row=row, column=1)
        if cell.value and search_pattern in str(cell.value):
            row += 1
            for col in range(2, pivot_ws.max_column + 1):
                entity = pivot_ws.cell(row=row, column=col).value
                if entity and entity not in ['합계', None] and entity not in entities:
                    entities.append(entity)
            break
    
    # ⭐ Entity 순서 정렬
    entities = sort_entities(entities)
    
    print(f"\n발견된 Entity (정렬됨): {entities}")
    
    # 데이터 읽기 함수 (⭐ Pivot 시트에서)
    def read_section_data(section_name, entity_name='합계'):
        data = {}
        max_row = pivot_ws.max_row
        
        for row in range(1, max_row + 1):
            cell = pivot_ws.cell(row=row, column=1)
            if cell.value and isinstance(cell.value, str) and section_name in cell.value:
                row += 1
                
                target_col = None
                max_col = pivot_ws.max_column
                for col in range(1, max_col + 1):
                    if pivot_ws.cell(row=row, column=col).value == entity_name:
                        target_col = col
                        break
                
                if not target_col:
                    continue
                
                row += 1
                
                while row <= max_row:
                    rev_account = pivot_ws.cell(row=row, column=1).value
                    if not rev_account or (isinstance(rev_account, str) and ('누적' in rev_account or '분기별' in rev_account)):
                        break
                    
                    value = pivot_ws.cell(row=row, column=target_col).value
                    try:
                        data[rev_account] = float(value) if value else 0
                    except (ValueError, TypeError):
                        data[rev_account] = 0
                    
                    row += 1
                
                break
        
        return data
    
    # 가로 배치 작성 함수
    def write_horizontal_analysis(ws, start_row, entity_name, qoq_base, qoq_compare, yoyq_base, yoyq_compare, yoyy_base, yoyy_compare):
        current_row = start_row
        
        # Entity 헤더
        ws.merge_cells(start_row=current_row, start_column=1, end_row=current_row, end_column=13)
        entity_cell = ws.cell(row=current_row, column=1)
        entity_cell.value = f"[{entity_name}]"
        entity_cell.fill = section_fill
        entity_cell.font = section_font
        entity_cell.alignment = Alignment(horizontal='center', vertical='center')
        current_row += 1
        
        # 컬럼 헤더 (분기 정보 포함)
        headers = [
            'Rev_Account',
            f'{prev_display}', f'{target_display}', 'QoQ_증감액', 'QoQ_증감률(%)',
            f'{yoy_display}', f'{target_display}', 'YoY(Q)_증감액', 'YoY(Q)_증감률(%)',
            f'{yoy_display}(누적)', f'{target_display}(누적)', 'YoY(Y)_증감액', 'YoY(Y)_증감률(%)'
        ]
        
        for col_idx, header in enumerate(headers, start=1):
            cell = ws.cell(row=current_row, column=col_idx)
            cell.value = header
            cell.fill = header_fill
            cell.font = header_font
            cell.alignment = Alignment(horizontal='center', vertical='center')
        current_row += 1
        
        # ⭐ 모든 계정 수집 후 정렬 (Pivot 시트의 모든 계정)
        all_accounts = set()
        all_accounts.update(qoq_base.keys())
        all_accounts.update(qoq_compare.keys())
        all_accounts.update(yoyq_base.keys())
        all_accounts.update(yoyq_compare.keys())
        all_accounts.update(yoyy_base.keys())
        all_accounts.update(yoyy_compare.keys())
        all_accounts = sort_rev_accounts(list(all_accounts))
        
        # 데이터 행
        for account in all_accounts:
            col = 1
            ws.cell(row=current_row, column=col, value=account)
            col += 1
            
            # QoQ
            qoq_b = qoq_base.get(account, 0)
            qoq_c = qoq_compare.get(account, 0)
            qoq_change = qoq_c - qoq_b
            qoq_rate = (qoq_change / qoq_b * 100) if qoq_b != 0 else (0 if qoq_change == 0 else float('inf'))
            
            ws.cell(row=current_row, column=col, value=qoq_b); col += 1
            ws.cell(row=current_row, column=col, value=qoq_c); col += 1
            
            change_cell = ws.cell(row=current_row, column=col, value=qoq_change)
            if qoq_change > 0:
                change_cell.fill = increase_fill
            elif qoq_change < 0:
                change_cell.fill = decrease_fill
            col += 1
            
            if qoq_rate != float('inf'):
                rate_cell = ws.cell(row=current_row, column=col, value=qoq_rate)
                rate_cell.number_format = '0.00'
                if qoq_rate > 0:
                    rate_cell.fill = increase_fill
                elif qoq_rate < 0:
                    rate_cell.fill = decrease_fill
            else:
                ws.cell(row=current_row, column=col, value='N/A')
            col += 1
            
            # YoY(Q)
            yoyq_b = yoyq_base.get(account, 0)
            yoyq_c = yoyq_compare.get(account, 0)
            yoyq_change = yoyq_c - yoyq_b
            yoyq_rate = (yoyq_change / yoyq_b * 100) if yoyq_b != 0 else (0 if yoyq_change == 0 else float('inf'))
            
            ws.cell(row=current_row, column=col, value=yoyq_b); col += 1
            ws.cell(row=current_row, column=col, value=yoyq_c); col += 1
            
            change_cell = ws.cell(row=current_row, column=col, value=yoyq_change)
            if yoyq_change > 0:
                change_cell.fill = increase_fill
            elif yoyq_change < 0:
                change_cell.fill = decrease_fill
            col += 1
            
            if yoyq_rate != float('inf'):
                rate_cell = ws.cell(row=current_row, column=col, value=yoyq_rate)
                rate_cell.number_format = '0.00'
                if yoyq_rate > 0:
                    rate_cell.fill = increase_fill
                elif yoyq_rate < 0:
                    rate_cell.fill = decrease_fill
            else:
                ws.cell(row=current_row, column=col, value='N/A')
            col += 1
            
            # YoY(Y)
            yoyy_b = yoyy_base.get(account, 0)
            yoyy_c = yoyy_compare.get(account, 0)
            yoyy_change = yoyy_c - yoyy_b
            yoyy_rate = (yoyy_change / yoyy_b * 100) if yoyy_b != 0 else (0 if yoyy_change == 0 else float('inf'))
            
            ws.cell(row=current_row, column=col, value=yoyy_b); col += 1
            ws.cell(row=current_row, column=col, value=yoyy_c); col += 1
            
            change_cell = ws.cell(row=current_row, column=col, value=yoyy_change)
            if yoyy_change > 0:
                change_cell.fill = increase_fill
            elif yoyy_change < 0:
                change_cell.fill = decrease_fill
            col += 1
            
            if yoyy_rate != float('inf'):
                rate_cell = ws.cell(row=current_row, column=col, value=yoyy_rate)
                rate_cell.number_format = '0.00'
                if yoyy_rate > 0:
                    rate_cell.fill = increase_fill
                elif yoyy_rate < 0:
                    rate_cell.fill = decrease_fill
            else:
                ws.cell(row=current_row, column=col, value='N/A')
            
            current_row += 1
        
        return current_row
    
    # Analysis 시트 작성
    current_row = 1
    
    # 전체 합계
    print("\n✓ 전체 합계 분석 (Pivot 시트 기준)")
    qoq_prev_total = read_section_data(f"{prev_quarter} (분기별)", "합계")
    qoq_target_total = read_section_data(f"{target_period} (분기별)", "합계")
    yoyq_base_total = read_section_data(f"{yoy_quarter} (분기별)", "합계")
    yoyq_target_total = qoq_target_total
    yoyy_base_total = read_section_data(f"{yoy_quarter} (누적)", "합계")
    yoyy_target_total = read_section_data(f"{target_period} (누적)", "합계")
    
    current_row = write_horizontal_analysis(
        analysis_ws, current_row, "전체 합계",
        qoq_prev_total, qoq_target_total,
        yoyq_base_total, yoyq_target_total,
        yoyy_base_total, yoyy_target_total
    )
    current_row += 2
    
    # Entity별
    for entity in entities:
        print(f"  - {entity}")
        qoq_prev_entity = read_section_data(f"{prev_quarter} (분기별)", entity)
        qoq_target_entity = read_section_data(f"{target_period} (분기별)", entity)
        yoyq_base_entity = read_section_data(f"{yoy_quarter} (분기별)", entity)
        yoyq_target_entity = qoq_target_entity
        yoyy_base_entity = read_section_data(f"{yoy_quarter} (누적)", entity)
        yoyy_target_entity = read_section_data(f"{target_period} (누적)", entity)
        
        current_row = write_horizontal_analysis(
            analysis_ws, current_row, entity,
            qoq_prev_entity, qoq_target_entity,
            yoyq_base_entity, yoyq_target_entity,
            yoyy_base_entity, yoyy_target_entity
        )
        current_row += 2
    
    # 컬럼 너비
    analysis_ws.column_dimensions['A'].width = 25
    for col_idx in range(2, 14):
        analysis_ws.column_dimensions[get_column_letter(col_idx)].width = 15
    
    wb.save(merged_pl_file)
    
    total_tables = 1 + len(entities)
    print(f"\n✓ Analysis 시트 생성 완료 ({total_tables}개 테이블, Pivot 시트 기준)")
    
    return True

def create_analysis_sga_horizontal(merged_pl_file, target_period="20254Q"):
    """
    Analysis_SG&A 시트 생성 - 가로 배치 (QoQ | YoY(Q) | YoY(Y))
    전체 합계 + 각 Entity별
    """
    print("\n" + "="*60)
    print(f"[Step 5] Analysis_SG&A 시트 생성 (가로 배치)")
    print("="*60)
    
    wb = openpyxl.load_workbook(merged_pl_file)
    
    if 'SG&A' not in wb.sheetnames:
        print("❌ SG&A 시트가 없습니다.")
        return None
    
    sga_ws = wb['SG&A']
    
    if 'Analysis_SG&A' in wb.sheetnames:
        del wb['Analysis_SG&A']
    analysis_sga_ws = wb.create_sheet('Analysis_SG&A')
    
    # 비교 분기 계산
    prev_quarter = get_previous_quarter(target_period)
    yoy_quarter = get_yoy_quarter(target_period)
    
    target_display = format_period_display(target_period)
    prev_display = format_period_display(prev_quarter)
    yoy_display = format_period_display(yoy_quarter)
    
    print(f"\n분석 분기:")
    print(f"  - QoQ: {prev_display} → {target_display}")
    print(f"  - YoY(Q): {yoy_display} → {target_display}")
    print(f"  - YoY(Y): {yoy_display}(누적) → {target_display}(누적)")
    
    # 스타일
    header_fill = PatternFill(start_color='4472C4', end_color='4472C4', fill_type='solid')
    header_font = Font(bold=True, color='FFFFFF')
    section_fill = PatternFill(start_color='FFC000', end_color='FFC000', fill_type='solid')
    section_font = Font(bold=True, size=11, color='FFFFFF')
    increase_fill = PatternFill(start_color='C6EFCE', end_color='C6EFCE', fill_type='solid')
    decrease_fill = PatternFill(start_color='FFC7CE', end_color='FFC7CE', fill_type='solid')
    
    # SG&A 계정 목록
    sga_accounts = [
        '매출', '매출원가', '매출총이익', '판관비',
        '인건비', '복리후생비', '광고선전비', '지급수수료', '운반비',
        '경상연구개발비', '판매수수료', '여비교통비','대손상각비','감가상각비', '사용권자산상각비','기타', '영업이익'
    ]
    
    # Entity 목록
    entities = []
    search_pattern = f"{target_period} (누적)"
    for row in range(1, sga_ws.max_row + 1):
        cell = sga_ws.cell(row=row, column=1)
        if cell.value and search_pattern in str(cell.value):
            row += 1
            for col in range(2, sga_ws.max_column + 1):
                entity = sga_ws.cell(row=row, column=col).value
                if entity and entity not in ['합계', None] and entity not in entities:
                    entities.append(entity)
            break
    
    # ⭐ Entity 순서 정렬
    entities = sort_entities(entities)
    
    print(f"\n발견된 Entity (정렬됨): {entities}")
    
    # 데이터 읽기 함수 (SG&A 시트에서)
    def read_section_data(section_name, entity_name='합계'):
        data = {}
        max_row = sga_ws.max_row
        
        for row in range(1, max_row + 1):
            cell = sga_ws.cell(row=row, column=1)
            if cell.value and isinstance(cell.value, str) and section_name in cell.value:
                row += 1
                
                target_col = None
                max_col = sga_ws.max_column
                for col in range(1, max_col + 1):
                    if sga_ws.cell(row=row, column=col).value == entity_name:
                        target_col = col
                        break
                
                if not target_col:
                    continue
                
                row += 1
                
                while row <= max_row:
                    rev_account = sga_ws.cell(row=row, column=1).value
                    if not rev_account or (isinstance(rev_account, str) and ('누적' in rev_account or '분기별' in rev_account)):
                        break
                    
                    value = sga_ws.cell(row=row, column=target_col).value
                    try:
                        data[rev_account] = float(value) if value else 0
                    except (ValueError, TypeError):
                        data[rev_account] = 0
                    
                    row += 1
                
                break
        
        return data
    
    # 가로 배치 작성 함수
    def write_horizontal_sga_analysis(ws, start_row, entity_name, qoq_base, qoq_compare, yoyq_base, yoyq_compare, yoyy_base, yoyy_compare):
        current_row = start_row
        
        # Entity 헤더
        ws.merge_cells(start_row=current_row, start_column=1, end_row=current_row, end_column=13)
        entity_cell = ws.cell(row=current_row, column=1)
        entity_cell.value = f"[{entity_name}]"
        entity_cell.fill = section_fill
        entity_cell.font = section_font
        entity_cell.alignment = Alignment(horizontal='center', vertical='center')
        current_row += 1
        
        # 컬럼 헤더 (분기 정보 포함)
        headers = [
            'Rev_Account',
            f'{prev_display}', f'{target_display}', 'QoQ_증감액', 'QoQ_증감률(%)',
            f'{yoy_display}', f'{target_display}', 'YoY(Q)_증감액', 'YoY(Q)_증감률(%)',
            f'{yoy_display}(누적)', f'{target_display}(누적)', 'YoY(Y)_증감액', 'YoY(Y)_증감률(%)'
        ]
        
        for col_idx, header in enumerate(headers, start=1):
            cell = ws.cell(row=current_row, column=col_idx)
            cell.value = header
            cell.fill = header_fill
            cell.font = header_font
            cell.alignment = Alignment(horizontal='center', vertical='center')
        current_row += 1
        
        # 데이터 행 (SG&A 계정만)
        for account in sga_accounts:
            col = 1
            ws.cell(row=current_row, column=col, value=account)
            col += 1
            
            # QoQ
            qoq_b = qoq_base.get(account, 0)
            qoq_c = qoq_compare.get(account, 0)
            qoq_change = qoq_c - qoq_b
            qoq_rate = (qoq_change / qoq_b * 100) if qoq_b != 0 else (0 if qoq_change == 0 else float('inf'))
            
            ws.cell(row=current_row, column=col, value=qoq_b); col += 1
            ws.cell(row=current_row, column=col, value=qoq_c); col += 1
            
            change_cell = ws.cell(row=current_row, column=col, value=qoq_change)
            if qoq_change > 0:
                change_cell.fill = increase_fill
            elif qoq_change < 0:
                change_cell.fill = decrease_fill
            col += 1
            
            if qoq_rate != float('inf'):
                rate_cell = ws.cell(row=current_row, column=col, value=qoq_rate)
                rate_cell.number_format = '0.00'
                if qoq_rate > 0:
                    rate_cell.fill = increase_fill
                elif qoq_rate < 0:
                    rate_cell.fill = decrease_fill
            else:
                ws.cell(row=current_row, column=col, value='N/A')
            col += 1
            
            # YoY(Q)
            yoyq_b = yoyq_base.get(account, 0)
            yoyq_c = yoyq_compare.get(account, 0)
            yoyq_change = yoyq_c - yoyq_b
            yoyq_rate = (yoyq_change / yoyq_b * 100) if yoyq_b != 0 else (0 if yoyq_change == 0 else float('inf'))
            
            ws.cell(row=current_row, column=col, value=yoyq_b); col += 1
            ws.cell(row=current_row, column=col, value=yoyq_c); col += 1
            
            change_cell = ws.cell(row=current_row, column=col, value=yoyq_change)
            if yoyq_change > 0:
                change_cell.fill = increase_fill
            elif yoyq_change < 0:
                change_cell.fill = decrease_fill
            col += 1
            
            if yoyq_rate != float('inf'):
                rate_cell = ws.cell(row=current_row, column=col, value=yoyq_rate)
                rate_cell.number_format = '0.00'
                if yoyq_rate > 0:
                    rate_cell.fill = increase_fill
                elif yoyq_rate < 0:
                    rate_cell.fill = decrease_fill
            else:
                ws.cell(row=current_row, column=col, value='N/A')
            col += 1
            
            # YoY(Y)
            yoyy_b = yoyy_base.get(account, 0)
            yoyy_c = yoyy_compare.get(account, 0)
            yoyy_change = yoyy_c - yoyy_b
            yoyy_rate = (yoyy_change / yoyy_b * 100) if yoyy_b != 0 else (0 if yoyy_change == 0 else float('inf'))
            
            ws.cell(row=current_row, column=col, value=yoyy_b); col += 1
            ws.cell(row=current_row, column=col, value=yoyy_c); col += 1
            
            change_cell = ws.cell(row=current_row, column=col, value=yoyy_change)
            if yoyy_change > 0:
                change_cell.fill = increase_fill
            elif yoyy_change < 0:
                change_cell.fill = decrease_fill
            col += 1
            
            if yoyy_rate != float('inf'):
                rate_cell = ws.cell(row=current_row, column=col, value=yoyy_rate)
                rate_cell.number_format = '0.00'
                if yoyy_rate > 0:
                    rate_cell.fill = increase_fill
                elif yoyy_rate < 0:
                    rate_cell.fill = decrease_fill
            else:
                ws.cell(row=current_row, column=col, value='N/A')
            
            current_row += 1
        
        return current_row
    
    # Analysis_SG&A 시트 작성
    current_row = 1
    
    # 전체 합계
    print("\n✓ 전체 합계 분석")
    qoq_prev_total = read_section_data(f"{prev_quarter} (분기별)", "합계")
    qoq_target_total = read_section_data(f"{target_period} (분기별)", "합계")
    yoyq_base_total = read_section_data(f"{yoy_quarter} (분기별)", "합계")
    yoyq_target_total = qoq_target_total
    yoyy_base_total = read_section_data(f"{yoy_quarter} (누적)", "합계")
    yoyy_target_total = read_section_data(f"{target_period} (누적)", "합계")
    
    current_row = write_horizontal_sga_analysis(
        analysis_sga_ws, current_row, "전체 합계",
        qoq_prev_total, qoq_target_total,
        yoyq_base_total, yoyq_target_total,
        yoyy_base_total, yoyy_target_total
    )
    current_row += 2
    
    # Entity별
    for entity in entities:
        print(f"  - {entity}")
        qoq_prev_entity = read_section_data(f"{prev_quarter} (분기별)", entity)
        qoq_target_entity = read_section_data(f"{target_period} (분기별)", entity)
        yoyq_base_entity = read_section_data(f"{yoy_quarter} (분기별)", entity)
        yoyq_target_entity = qoq_target_entity
        yoyy_base_entity = read_section_data(f"{yoy_quarter} (누적)", entity)
        yoyy_target_entity = read_section_data(f"{target_period} (누적)", entity)
        
        current_row = write_horizontal_sga_analysis(
            analysis_sga_ws, current_row, entity,
            qoq_prev_entity, qoq_target_entity,
            yoyq_base_entity, yoyq_target_entity,
            yoyy_base_entity, yoyy_target_entity
        )
        current_row += 2
    
    # 컬럼 너비
    analysis_sga_ws.column_dimensions['A'].width = 25
    for col_idx in range(2, 14):
        analysis_sga_ws.column_dimensions[get_column_letter(col_idx)].width = 15
    
    wb.save(merged_pl_file)
    
    total_tables = 1 + len(entities)
    print(f"\n✓ Analysis_SG&A 시트 생성 완료 ({total_tables}개 테이블)")
    
    return True

def create_sheet_total_pl(merged_pl_file, target_period="20254Q"):
    """
    Sheet_total PL 시트 생성 - Entity가 컬럼으로 배치
    QoQ, YoY(Q), YoY(Y) 각각 4개 테이블 (Base, Compare, 증감액, 증감률)
    """
    print("\n" + "="*60)
    print(f"[Step 6] Sheet_total PL 시트 생성")
    print("="*60)
    
    wb = openpyxl.load_workbook(merged_pl_file)
    
    if 'SG&A' not in wb.sheetnames:
        print("❌ SG&A 시트가 없습니다.")
        return None
    
    sga_ws = wb['SG&A']
    
    if 'Sheet_total PL' in wb.sheetnames:
        del wb['Sheet_total PL']
    total_pl_ws = wb.create_sheet('Sheet_total PL')
    
    # 비교 분기 계산
    prev_quarter = get_previous_quarter(target_period)
    yoy_quarter = get_yoy_quarter(target_period)
    
    target_display = format_period_display(target_period)
    prev_display = format_period_display(prev_quarter)
    yoy_display = format_period_display(yoy_quarter)
    
    print(f"\n분석 분기:")
    print(f"  - QoQ: {prev_display} → {target_display}")
    print(f"  - YoY(Q): {yoy_display} → {target_display}")
    print(f"  - YoY(Y): {yoy_display}(누적) → {target_display}(누적)")
    
    # 스타일
    header_fill = PatternFill(start_color='4472C4', end_color='4472C4', fill_type='solid')
    header_font = Font(bold=True, color='FFFFFF')
    section_fill = PatternFill(start_color='FFC000', end_color='FFC000', fill_type='solid')
    section_font = Font(bold=True, size=11, color='FFFFFF')
    increase_fill = PatternFill(start_color='C6EFCE', end_color='C6EFCE', fill_type='solid')
    decrease_fill = PatternFill(start_color='FFC7CE', end_color='FFC7CE', fill_type='solid')
    
    # Entity 목록
    entities = []
    search_pattern = f"{target_period} (누적)"
    for row in range(1, sga_ws.max_row + 1):
        cell = sga_ws.cell(row=row, column=1)
        if cell.value and search_pattern in str(cell.value):
            row += 1
            for col in range(2, sga_ws.max_column + 1):
                entity = sga_ws.cell(row=row, column=col).value
                # 합계만 제외, 연결조정 포함
                if entity and entity not in entities:
                    entities.append(entity)
            break
    
    # ⭐ Entity 순서 정렬
    entities = sort_entities(entities)
    
    print(f"\nEntity (정렬됨): {entities}")
    
    # 데이터 읽기
    def read_all_entities_data(section_name):
        """모든 Entity의 데이터를 읽어서 딕셔너리로 반환"""
        entity_data = {}
        
        for entity in entities:
            data = {}
            max_row = sga_ws.max_row
            
            for row in range(1, max_row + 1):
                cell = sga_ws.cell(row=row, column=1)
                if cell.value and isinstance(cell.value, str) and section_name in cell.value:
                    row += 1
                    
                    target_col = None
                    max_col = sga_ws.max_column
                    for col in range(1, max_col + 1):
                        if sga_ws.cell(row=row, column=col).value == entity:
                            target_col = col
                            break
                    
                    if not target_col:
                        continue
                    
                    row += 1
                    
                    while row <= max_row:
                        rev_account = sga_ws.cell(row=row, column=1).value
                        if not rev_account or (isinstance(rev_account, str) and ('누적' in rev_account or '분기별' in rev_account)):
                            break
                        
                        value = sga_ws.cell(row=row, column=target_col).value
                        try:
                            data[rev_account] = float(value) if value else 0
                        except (ValueError, TypeError):
                            data[rev_account] = 0
                        
                        row += 1
                    
                    break
            
            entity_data[entity] = data
        
        return entity_data
    
    # 테이블 작성 함수
    def write_entity_table(ws, start_row, title, entity_data_dict, value_type='value'):
        """
        Entity를 컬럼으로 하는 테이블 작성
        value_type: 'value', 'change', 'rate'
        """
        current_row = start_row
        
        # 타이틀
        ws.merge_cells(start_row=current_row, start_column=1, end_row=current_row, end_column=len(entities) + 1)
        title_cell = ws.cell(row=current_row, column=1)
        title_cell.value = title
        title_cell.fill = section_fill
        title_cell.font = section_font
        title_cell.alignment = Alignment(horizontal='center', vertical='center')
        current_row += 1
        
        # 헤더
        ws.cell(row=current_row, column=1, value='Rev_Account').fill = header_fill
        ws.cell(row=current_row, column=1).font = header_font
        ws.cell(row=current_row, column=1).alignment = Alignment(horizontal='center', vertical='center')
        
        for col_idx, entity in enumerate(entities, start=2):
            cell = ws.cell(row=current_row, column=col_idx)
            cell.value = entity
            cell.fill = header_fill
            cell.font = header_font
            cell.alignment = Alignment(horizontal='center', vertical='center')
        current_row += 1
        
        # ⭐ Analysis_SG&A와 동일한 계정 순서 사용
        sga_accounts = [
             '매출', '매출원가', '매출총이익', '판관비',
             '인건비', '복리후생비', '광고선전비', '지급수수료', '운반비',
             '경상연구개발비', '판매수수료', '여비교통비','대손상각비','감가상각비', '사용권자산상각비','기타', '영업이익'
        ]
        
        # 데이터 행 - 고정된 SG&A 계정 순서로
        for account in sga_accounts:
            ws.cell(row=current_row, column=1, value=account)
            
            for col_idx, entity in enumerate(entities, start=2):
                value = entity_data_dict.get(entity, {}).get(account, 0)
                cell = ws.cell(row=current_row, column=col_idx, value=value)
                
                # 증감액/증감률이면 색상 적용
                if value_type in ['change', 'rate']:
                    if value > 0:
                        cell.fill = increase_fill
                    elif value < 0:
                        cell.fill = decrease_fill
                
                # 증감률이면 포맷 적용
                if value_type == 'rate' and value != float('inf'):
                    cell.number_format = '0.00'
            
            current_row += 1
        
        return current_row
    
    # 증감 계산 함수
    def calculate_changes(base_data, compare_data):
        """증감액과 증감률 계산"""
        change_data = {}
        rate_data = {}
        
        for entity in entities:
            base = base_data.get(entity, {})
            compare = compare_data.get(entity, {})
            
            entity_change = {}
            entity_rate = {}
            
            all_accounts = set()
            all_accounts.update(base.keys())
            all_accounts.update(compare.keys())
            
            for account in all_accounts:
                base_val = base.get(account, 0)
                compare_val = compare.get(account, 0)
                change = compare_val - base_val
                
                if base_val != 0:
                    rate = (change / base_val) * 100
                else:
                    rate = 0 if change == 0 else float('inf')
                
                entity_change[account] = change
                entity_rate[account] = rate
            
            change_data[entity] = entity_change
            rate_data[entity] = entity_rate
        
        return change_data, rate_data
    
    # Sheet_total PL 작성
    current_row = 1
    
    # QoQ 분석
    print("\n✓ QoQ 테이블 생성")
    qoq_prev = read_all_entities_data(f"{prev_quarter} (분기별)")
    qoq_target = read_all_entities_data(f"{target_period} (분기별)")
    qoq_change, qoq_rate = calculate_changes(qoq_prev, qoq_target)
    
    current_row = write_entity_table(total_pl_ws, current_row, 
                                     f"QoQ 분석: {prev_display}", qoq_prev, 'value')
    current_row += 1
    
    current_row = write_entity_table(total_pl_ws, current_row, 
                                     f"QoQ 분석: {target_display}", qoq_target, 'value')
    current_row += 1
    
    current_row = write_entity_table(total_pl_ws, current_row, 
                                     f"QoQ 분석: 증감액 ({prev_display} → {target_display})", qoq_change, 'change')
    current_row += 1
    
    current_row = write_entity_table(total_pl_ws, current_row, 
                                     f"QoQ 분석: 증감률(%) ({prev_display} → {target_display})", qoq_rate, 'rate')
    current_row += 2
    
    # YoY(Q) 분석
    print("✓ YoY(Q) 테이블 생성")
    yoyq_base = read_all_entities_data(f"{yoy_quarter} (분기별)")
    yoyq_target = qoq_target  # 동일
    yoyq_change, yoyq_rate = calculate_changes(yoyq_base, yoyq_target)
    
    current_row = write_entity_table(total_pl_ws, current_row, 
                                     f"YoY(Q) 분석: {yoy_display}", yoyq_base, 'value')
    current_row += 1
    
    current_row = write_entity_table(total_pl_ws, current_row, 
                                     f"YoY(Q) 분석: {target_display}", yoyq_target, 'value')
    current_row += 1
    
    current_row = write_entity_table(total_pl_ws, current_row, 
                                     f"YoY(Q) 분석: 증감액 ({yoy_display} → {target_display})", yoyq_change, 'change')
    current_row += 1
    
    current_row = write_entity_table(total_pl_ws, current_row, 
                                     f"YoY(Q) 분석: 증감률(%) ({yoy_display} → {target_display})", yoyq_rate, 'rate')
    current_row += 2
    
    # YoY(Y) 분석
    print("✓ YoY(Y) 테이블 생성")
    yoyy_base = read_all_entities_data(f"{yoy_quarter} (누적)")
    yoyy_target = read_all_entities_data(f"{target_period} (누적)")
    yoyy_change, yoyy_rate = calculate_changes(yoyy_base, yoyy_target)
    
    current_row = write_entity_table(total_pl_ws, current_row, 
                                     f"YoY(Y) 분석: {yoy_display}(누적)", yoyy_base, 'value')
    current_row += 1
    
    current_row = write_entity_table(total_pl_ws, current_row, 
                                     f"YoY(Y) 분석: {target_display}(누적)", yoyy_target, 'value')
    current_row += 1
    
    current_row = write_entity_table(total_pl_ws, current_row, 
                                     f"YoY(Y) 분석: 증감액 ({yoy_display} → {target_display})", yoyy_change, 'change')
    current_row += 1
    
    current_row = write_entity_table(total_pl_ws, current_row, 
                                     f"YoY(Y) 분석: 증감률(%) ({yoy_display} → {target_display})", yoyy_rate, 'rate')
    
    # 컬럼 너비
    total_pl_ws.column_dimensions['A'].width = 25
    for col_idx in range(2, len(entities) + 2):
        total_pl_ws.column_dimensions[get_column_letter(col_idx)].width = 15
    
    wb.save(merged_pl_file)
    
    print(f"\n✓ Sheet_total PL 시트 생성 완료 (12개 테이블)")
    
    return True

# 실행
if __name__ == "__main__":
    folder_path = r"C:\Users\user\InBody Co., Ltd\회계팀 - General\★★Accounting Folder★★vF\1. 재무결산\FY2025\2025_12_4Q\2. 연결재무제표\Income statement"
    merged_pl_file = r"C:\Users\user\InBody Co., Ltd\회계팀 - General\★★Accounting Folder★★vF\1. 재무결산\FY2025\2025_12_4Q\2. 연결재무제표\Income statement\InBody_PL_Analysis.xlsx"
    file_pattern = 'final_financial_data'
    
    # ⭐ 분석 대상 분기 - 데이터에 있는 분기로 설정
    # 사용 가능: 20243Q, 20244Q, 20251Q, 20252Q, 20253Q
    target_period = "20254Q"  # 2025년 3분기 (가장 최신)
    
    print("="*60)
    print("PL 분석 (가로 배치 - QoQ | YoY(Q) | YoY(Y))")
    print(f"분석 기준 분기: {format_period_display(target_period)}")
    print("="*60)
    
    print("\n📋 Entity 이름 매핑:")
    for original, renamed in ENTITY_NAME_MAPPING.items():
        print(f"  {original:40s} → {renamed}")
    print()
    
    # Step 1: PL 시트 병합
    merged_df = concat_pl_sheets(folder_path, merged_pl_file, file_pattern)
    
    if merged_df is not None:
        # Step 1.5: 환율 누락 체크 (자동 수정 안 함, 경고만)
        merged_df = check_missing_exchange_rates(merged_pl_file)
        
        # Step 2: Pivot 시트 생성
        pivot_df = create_pivot_sheet(merged_pl_file)
        
        if pivot_df is not None:
            # Step 3: SG&A 시트 생성
            sga_result = create_sga_sheet(merged_pl_file)
            
            if sga_result:
                # Step 4: Analysis 시트 생성 (가로 배치)
                analysis_result = create_analysis_horizontal(merged_pl_file, target_period)
                
                if analysis_result:
                    # Step 5: Analysis_SG&A 시트 생성 (가로 배치)
                    analysis_sga_result = create_analysis_sga_horizontal(merged_pl_file, target_period)
                    
                    if analysis_sga_result:
                        # Step 6: Sheet_total PL 시트 생성
                        total_pl_result = create_sheet_total_pl(merged_pl_file, target_period)
                        
                        if total_pl_result:
                            print("\n" + "="*60)
                            print("✅ 모든 작업 완료!")
                            print("="*60)
                            print(f"\n생성된 파일: {merged_pl_file}")
                            print(f"  - 분석 기준 분기: {format_period_display(target_period)}")
                            print(f"  - Entity 이름: 짧은 이름으로 변경")
                            print(f"  - Sheet1: 병합된 PL 데이터")
                            print(f"  - Pivot: Entity별 누적 + 분기별")
                            print(f"  - SG&A: 매출총이익, 판관비, 영업이익 계산")
                            print(f"  - Analysis: QoQ | YoY(Q) | YoY(Y) 가로 배치 (전체 + Entity별)")
                            print(f"  - Analysis_SG&A: QoQ | YoY(Q) | YoY(Y) 가로 배치 (전체 + Entity별)")
                            print(f"  - Sheet_total PL: Entity 가로 배치 (QoQ, YoY(Q), YoY(Y) 각 4개 테이블)")
                        else:
                            print("\n❌ Step 6 (Sheet_total PL 생성) 실패")
                    else:
                        print("\n❌ Step 5 (Analysis_SG&A 생성) 실패")
                else:
                    print("\n❌ Step 4 (Analysis 생성) 실패")
            else:
                print("\n❌ Step 3 (SG&A 생성) 실패")
        else:
            print("\n❌ Step 2 (Pivot 생성) 실패")
    else:
        print("\n❌ Step 1 (PL 병합) 실패")

PL 분석 (가로 배치 - QoQ | YoY(Q) | YoY(Y))
분석 기준 분기: 2025_4Q

📋 Entity 이름 매핑:
  BIOSPACE LATIN AMERICA S DE RL DE CV(*)  → Mexico
  BWA                                      → BWA
  Biospace Co.,Ltd.                        → China
  Biospace Inc. DBA INBODY                 → USA
  InBody India Pvt. Ltd                    → India
  InBody Japan Inc.                        → Japan
  InBody Oceania                           → Oceania
  ㈜코르트                                     → KOROT
  Inbody Europe B.V.                       → Europe
  ㈜인바디헬스케어                                 → 헬스케어
  (주)삼한정공                                  → 삼한정공
  INBODY TURKEY MEDİKAL TİCARET LİMİTED ŞİRKETİ → Turkey
  INBODY ASIA SDN. BHD.                    → Asia
  주식회사 인바디                                 → HQ
  케이오씨피 프로젝트 제2호 벤처투자조합                    → KOCP
  InBody Vietnam                           → Vietnam
  ADJ                                      → 연결조정

[Step 1] PL 시트 병합 시작

파일명에 'final_financial_data' 포함된 파일 6개

In [1]:
import pandas as pd
from pathlib import Path
import openpyxl
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter


# ==================== Rev_Account 순서 설정 ====================
REV_ACCOUNT_ORDER = [
    '매출',
    '매출원가',
    '인건비',
    '복리후생비',
    '광고선전비',
    '지급수수료',
    '운반비',
    '경상연구개발비',
    '판매수수료',
    '여비교통비',
    '대손상각비',
    '감가상각비',
    '사용권자산상각비',
    '기타',
    '영업외수익',
    '영업외비용',
    '금융수익',
    '금융비용',
    '지분법손익',
    '법인세비용',
    '세후중단영업손익',
    '당기순이익',
    '포괄손익',
    '총포괄손익',
]

# ==================== Entity 순서 설정 ====================
ENTITY_ORDER = [
    'HQ',
    'USA',
    'Japan',
    'China',
    'Europe',
    'Asia',
    'India',
    'Mexico',
    'Oceania',
    'BWA',
    'Vietnam',
    'Turkey',
    'KOROT',
    '헬스케어',
    '삼한정공',
    'KOCP',
    '연결조정',
]

def sort_rev_accounts(accounts):
    """Rev_Account를 정의된 순서대로 정렬"""
    sorted_accounts = []
    for account in REV_ACCOUNT_ORDER:
        if account in accounts:
            sorted_accounts.append(account)
    remaining = sorted([acc for acc in accounts if acc not in REV_ACCOUNT_ORDER])
    sorted_accounts.extend(remaining)
    return sorted_accounts

def sort_entities(entities):
    """Entity를 정의된 순서대로 정렬"""
    sorted_entities = []
    for entity in ENTITY_ORDER:
        if entity in entities:
            sorted_entities.append(entity)
    remaining = sorted([ent for ent in entities if ent not in ENTITY_ORDER])
    sorted_entities.extend(remaining)
    return sorted_entities

# ==================== Entity 이름 매핑 ====================
# 원본 이름 → 짧은 이름으로 변경
ENTITY_NAME_MAPPING = {
    'BIOSPACE LATIN AMERICA S DE RL DE CV(*)': 'Mexico',
    'BWA': 'BWA',
    'Biospace Co.,Ltd.': 'China',
    'Biospace Inc. DBA INBODY': 'USA',
    'InBody India Pvt. Ltd': 'India',
    'InBody Japan Inc.': 'Japan',
    'InBody Oceania ': 'Oceania',
    '㈜코르트': 'KOROT',
    'Inbody Europe B.V.': 'Europe',
    '㈜인바디헬스케어': '헬스케어',
    '(주)삼한정공': '삼한정공',
    'INBODY TURKEY MEDİKAL TİCARET LİMİTED ŞİRKETİ': 'Turkey',
    'INBODY ASIA SDN. BHD.': 'Asia',
    '주식회사 인바디': 'HQ',
    '케이오씨피 프로젝트 제2호 벤처투자조합': 'KOCP',
    'InBody Vietnam': 'Vietnam',
    'ADJ': '연결조정',
}

def rename_entity(original_name):
    """
    Entity 이름을 짧은 이름으로 변환
    매핑에 없으면 원본 이름 반환
    """
    return ENTITY_NAME_MAPPING.get(original_name, original_name)

def parse_period(period_str):
    """Period 문자열 파싱"""
    if not period_str or not isinstance(period_str, str):
        return None, None
    if period_str in ['전체', '합계', 'Total']:
        return None, None
    if len(period_str) < 5:
        return None, None
    try:
        year = int(period_str[:4])
        quarter = int(period_str[4])
        return year, quarter
    except (ValueError, IndexError):
        return None, None

def get_previous_quarter(period_str):
    """이전 분기 계산"""
    year, quarter = parse_period(period_str)
    if year is None or quarter is None:
        return None
    if quarter == 1:
        return f"{year-1}4Q"
    else:
        return f"{year}{quarter-1}Q"

def get_yoy_quarter(period_str):
    """전년 동분기 계산"""
    year, quarter = parse_period(period_str)
    if year is None or quarter is None:
        return None
    return f"{year-1}{quarter}Q"

def format_period_display(period_str):
    """Period 포맷팅"""
    if not period_str:
        return "Unknown"
    year, quarter = parse_period(period_str)
    if year is None or quarter is None:
        return period_str
    return f"{year}_{quarter}Q"

def create_pivot_sheet_fcy(merged_pl_file):
    """Pivot(FCY) 시트 생성 - Amount 기준"""
    print("\n" + "="*60)
    print("[Step 2-FCY] Pivot(FCY) 시트 생성 시작")
    print("="*60)
    
    print(f"\n파일 로드 중: {merged_pl_file}")
    df = pd.read_excel(merged_pl_file)
    print(f"✓ 데이터 로드 완료: {len(df)} 행")
    
    required_cols = ['Entity', 'Period', 'Rev_Account', 'Amount']
    missing_cols = [col for col in required_cols if col not in df.columns]
    
    if missing_cols:
        print(f"\n❌ 필요한 컬럼이 없습니다: {missing_cols}")
        return None
    
    # Entity 이름 변경
    print(f"\n✓ Entity 이름 변경 적용 중...")
    df['Entity'] = df['Entity'].apply(rename_entity)
    
    print(f"\n✓ Period 유니크 값: {sorted(df['Period'].unique())}")
    print(f"✓ Entity 유니크 값: {df['Entity'].nunique()}개")
    
    print(f"\n피벗테이블 생성 중 (Amount 기준 - FCY)...")
    pivot_df = df.pivot_table(
        index='Rev_Account',
        columns=['Period', 'Entity'],
        values='Amount',  # ⭐ Amount (외화)
        aggfunc='sum',
        fill_value=0
    )
    
    print(f"✓ 피벗테이블 생성 완료")
    
    for period in pivot_df.columns.get_level_values('Period').unique():
        period_cols = [col for col in pivot_df.columns if col[0] == period]
        pivot_df[(period, '합계')] = pivot_df[period_cols].sum(axis=1)
    
    pivot_df[('전체', '합계')] = pivot_df.sum(axis=1)
    pivot_df = pivot_df.reset_index()
    
    print(f"\n저장 중... (Pivot(FCY) 시트)")
    
    wb = openpyxl.load_workbook(merged_pl_file)
    if 'Pivot(FCY)' in wb.sheetnames:
        del wb['Pivot(FCY)']
    ws = wb.create_sheet('Pivot(FCY)')
    
    periods = []
    entities_by_period = {}
    
    for col in pivot_df.columns:
        if col == 'Rev_Account':
            continue
        period, entity = col
        if period not in periods:
            periods.append(period)
            entities_by_period[period] = []
        if entity not in entities_by_period[period]:
            entities_by_period[period].append(entity)
    
    # Entity 순서 정렬
    for period in periods:
        entities_by_period[period] = sort_entities(entities_by_period[period])
    
    # Rev_Account 순서 정렬
    rev_accounts = sort_rev_accounts(pivot_df['Rev_Account'].tolist())
    
    header_fill = PatternFill(start_color='4472C4', end_color='4472C4', fill_type='solid')
    header_font = Font(bold=True, color='FFFFFF')
    subheader_fill = PatternFill(start_color='D9E1F2', end_color='D9E1F2', fill_type='solid')
    subheader_font = Font(bold=True)
    period_fill_acc = PatternFill(start_color='4472C4', end_color='4472C4', fill_type='solid')
    period_fill_qtd = PatternFill(start_color='70AD47', end_color='70AD47', fill_type='solid')
    period_font = Font(bold=True, size=12, color='FFFFFF')
    
    current_row = 1
    prev_period_data = None
    
    for period_idx, period in enumerate(periods):
        entities = entities_by_period[period]
        
        # 누적 섹션
        ws.merge_cells(start_row=current_row, start_column=1, end_row=current_row, end_column=len(entities) + 2)
        period_cell = ws.cell(row=current_row, column=1)
        period_cell.value = f"{period} (누적)"
        period_cell.fill = period_fill_acc
        period_cell.font = period_font
        period_cell.alignment = Alignment(horizontal='center', vertical='center')
        current_row += 1
        
        ws.cell(row=current_row, column=1, value='Rev_Account').fill = header_fill
        ws.cell(row=current_row, column=1).font = header_font
        ws.cell(row=current_row, column=1).alignment = Alignment(horizontal='center', vertical='center')
        
        for col_idx, entity in enumerate(entities, start=2):
            cell = ws.cell(row=current_row, column=col_idx)
            cell.value = entity
            cell.fill = subheader_fill
            cell.font = subheader_font
            cell.alignment = Alignment(horizontal='center', vertical='center')
        current_row += 1
        
        current_period_data = {}
        
        for rev_account in rev_accounts:
            ws.cell(row=current_row, column=1, value=rev_account)
            
            for col_idx, entity in enumerate(entities, start=2):
                col_tuple = (period, entity)
                row_data = pivot_df[pivot_df['Rev_Account'] == rev_account]
                if not row_data.empty and col_tuple in pivot_df.columns:
                    value = row_data.iloc[0][col_tuple]
                    ws.cell(row=current_row, column=col_idx, value=value)
                    
                    if rev_account not in current_period_data:
                        current_period_data[rev_account] = {}
                    current_period_data[rev_account][entity] = value
                else:
                    ws.cell(row=current_row, column=col_idx, value=0)
                    if rev_account not in current_period_data:
                        current_period_data[rev_account] = {}
                    current_period_data[rev_account][entity] = 0
            
            current_row += 1
        
        # 분기별 섹션
        if period_idx > 0 and prev_period_data is not None:
            current_row += 1
            
            year, quarter = parse_period(period)
            
            if year is None or quarter is None:
                prev_period_data = current_period_data
                continue
            
            is_first_quarter = (quarter == 1)
            
            ws.merge_cells(start_row=current_row, start_column=1, end_row=current_row, end_column=len(entities) + 2)
            period_cell = ws.cell(row=current_row, column=1)
            period_cell.value = f"{period} (분기별)"
            period_cell.fill = period_fill_qtd
            period_cell.font = period_font
            period_cell.alignment = Alignment(horizontal='center', vertical='center')
            current_row += 1
            
            ws.cell(row=current_row, column=1, value='Rev_Account').fill = header_fill
            ws.cell(row=current_row, column=1).font = header_font
            ws.cell(row=current_row, column=1).alignment = Alignment(horizontal='center', vertical='center')
            
            for col_idx, entity in enumerate(entities, start=2):
                cell = ws.cell(row=current_row, column=col_idx)
                cell.value = entity
                cell.fill = subheader_fill
                cell.font = subheader_font
                cell.alignment = Alignment(horizontal='center', vertical='center')
            current_row += 1
            
            for rev_account in rev_accounts:
                ws.cell(row=current_row, column=1, value=rev_account)
                
                for col_idx, entity in enumerate(entities, start=2):
                    current_value = current_period_data.get(rev_account, {}).get(entity, 0)
                    
                    if is_first_quarter:
                        qtd_value = current_value
                    else:
                        prev_value = prev_period_data.get(rev_account, {}).get(entity, 0)
                        qtd_value = current_value - prev_value
                    
                    ws.cell(row=current_row, column=col_idx, value=qtd_value)
                
                current_row += 1
        
        prev_period_data = current_period_data
        
        if period_idx < len(periods) - 1:
            current_row += 2
    
    ws.column_dimensions['A'].width = 25
    max_entities = max(len(entities_by_period[p]) for p in periods)
    for col_idx in range(2, max_entities + 2):
        ws.column_dimensions[get_column_letter(col_idx)].width = 18
    
    wb.save(merged_pl_file)
    
    total_sections = len(periods) + (len(periods) - 1)
    print(f"✓ Pivot(FCY) 시트 추가 완료 (총 {total_sections}개 섹션)")
    
    return pivot_df

def create_sga_sheet_fcy(merged_pl_file):
    """SG&A(FCY) 시트 생성 - Pivot(FCY) 기반"""
    print("\n" + "="*60)
    print("[Step 3-FCY] SG&A(FCY) 시트 생성 시작")
    print("="*60)
    
    wb = openpyxl.load_workbook(merged_pl_file)
    
    if 'Pivot(FCY)' not in wb.sheetnames:
        print("❌ Pivot(FCY) 시트가 없습니다.")
        return None
    
    pivot_ws = wb['Pivot(FCY)']
    
    if 'SG&A(FCY)' in wb.sheetnames:
        del wb['SG&A(FCY)']
    sga_ws = wb.create_sheet('SG&A(FCY)')
    
    target_accounts = [
        '매출', '매출원가', '매출총이익', '판관비',
        '인건비', '복리후생비', '광고선전비', '지급수수료', '운반비',
        '경상연구개발비', '판매수수료', '여비교통비','대손상각비','감가상각비', '사용권자산상각비','기타', '영업이익'
    ]
    
    calculated_accounts = ['매출총이익', '판관비', '영업이익']
    
    header_fill = PatternFill(start_color='4472C4', end_color='4472C4', fill_type='solid')
    header_font = Font(bold=True, color='FFFFFF')
    subheader_fill = PatternFill(start_color='D9E1F2', end_color='D9E1F2', fill_type='solid')
    subheader_font = Font(bold=True)
    period_fill_acc = PatternFill(start_color='4472C4', end_color='4472C4', fill_type='solid')
    period_fill_qtd = PatternFill(start_color='70AD47', end_color='70AD47', fill_type='solid')
    period_font = Font(bold=True, size=12, color='FFFFFF')
    calc_fill = PatternFill(start_color='FFF2CC', end_color='FFF2CC', fill_type='solid')
    calc_font = Font(bold=True)
    
    current_row = 1
    sga_row = 1
    max_row = pivot_ws.max_row
    max_col = pivot_ws.max_column
    section_count = 0
    
    while current_row <= max_row:
        cell = pivot_ws.cell(row=current_row, column=1)
        
        if cell.value and isinstance(cell.value, str) and ('누적' in cell.value or '분기별' in cell.value):
            section_count += 1
            period_header = cell.value
            is_qtd = '분기별' in period_header
            
            sga_ws.merge_cells(start_row=sga_row, start_column=1, end_row=sga_row, end_column=max_col)
            sga_cell = sga_ws.cell(row=sga_row, column=1)
            sga_cell.value = period_header
            sga_cell.fill = period_fill_qtd if is_qtd else period_fill_acc
            sga_cell.font = period_font
            sga_cell.alignment = Alignment(horizontal='center', vertical='center')
            sga_row += 1
            current_row += 1
            
            for col in range(1, max_col + 1):
                src_cell = pivot_ws.cell(row=current_row, column=col)
                dst_cell = sga_ws.cell(row=sga_row, column=col)
                dst_cell.value = src_cell.value
                if col == 1:
                    dst_cell.fill = header_fill
                    dst_cell.font = header_font
                else:
                    dst_cell.fill = subheader_fill
                    dst_cell.font = subheader_font
                dst_cell.alignment = Alignment(horizontal='center', vertical='center')
            
            sga_row += 1
            current_row += 1
            
            section_data = {}
            while current_row <= max_row:
                rev_account_cell = pivot_ws.cell(row=current_row, column=1)
                rev_account = rev_account_cell.value
                
                if not rev_account or (isinstance(rev_account, str) and ('누적' in rev_account or '분기별' in rev_account)):
                    break
                
                row_data = [rev_account]
                for col in range(2, max_col + 1):
                    value = pivot_ws.cell(row=current_row, column=col).value
                    row_data.append(value if value is not None else 0)
                
                section_data[rev_account] = row_data
                current_row += 1
            
            for account in target_accounts:
                if account in calculated_accounts:
                    if account == '매출총이익':
                        if '매출' in section_data and '매출원가' in section_data:
                            sga_ws.cell(row=sga_row, column=1, value=account)
                            sga_ws.cell(row=sga_row, column=1).fill = calc_fill
                            sga_ws.cell(row=sga_row, column=1).font = calc_font
                            for col_idx in range(2, max_col + 1):
                                revenue = section_data['매출'][col_idx - 1]
                                cogs = section_data['매출원가'][col_idx - 1]
                                
                                try:
                                    revenue_val = float(revenue) if revenue and revenue != '' else 0
                                except (ValueError, TypeError):
                                    revenue_val = 0
                                
                                try:
                                    cogs_val = float(cogs) if cogs and cogs != '' else 0
                                except (ValueError, TypeError):
                                    cogs_val = 0
                                
                                gross_profit = revenue_val - cogs_val
                                sga_ws.cell(row=sga_row, column=col_idx, value=gross_profit).fill = calc_fill
                            sga_row += 1
                    
                    elif account == '판관비':
                        sga_components = ['인건비', '복리후생비', '광고선전비', '지급수수료', '운반비', 
                        '경상연구개발비', '판매수수료', '여비교통비','대손상각비','감가상각비', '사용권자산상각비','기타']
                        sga_ws.cell(row=sga_row, column=1, value=account)
                        sga_ws.cell(row=sga_row, column=1).fill = calc_fill
                        sga_ws.cell(row=sga_row, column=1).font = calc_font
                        for col_idx in range(2, max_col + 1):
                            sga_total = 0
                            for c in sga_components:
                                if c in section_data and section_data[c][col_idx - 1]:
                                    try:
                                        value = float(section_data[c][col_idx - 1])
                                        sga_total += value
                                    except (ValueError, TypeError):
                                        pass
                            sga_ws.cell(row=sga_row, column=col_idx, value=sga_total).fill = calc_fill
                        sga_row += 1
                    
                    elif account == '영업이익':
                        sga_ws.cell(row=sga_row, column=1, value=account)
                        sga_ws.cell(row=sga_row, column=1).fill = calc_fill
                        sga_ws.cell(row=sga_row, column=1).font = calc_font
                        gross_profit_row = None
                        sga_expense_row = None
                        for search_row in range(sga_row - 1, 0, -1):
                            cell_value = sga_ws.cell(row=search_row, column=1).value
                            if cell_value == '매출총이익':
                                gross_profit_row = search_row
                            elif cell_value == '판관비':
                                sga_expense_row = search_row
                            if gross_profit_row and sga_expense_row:
                                break
                        if gross_profit_row and sga_expense_row:
                            for col_idx in range(2, max_col + 1):
                                gp = sga_ws.cell(row=gross_profit_row, column=col_idx).value
                                sga_exp = sga_ws.cell(row=sga_expense_row, column=col_idx).value
                                
                                try:
                                    gp_val = float(gp) if gp and gp != '' else 0
                                except (ValueError, TypeError):
                                    gp_val = 0
                                
                                try:
                                    sga_val = float(sga_exp) if sga_exp and sga_exp != '' else 0
                                except (ValueError, TypeError):
                                    sga_val = 0
                                
                                operating_income = gp_val - sga_val
                                sga_ws.cell(row=sga_row, column=col_idx, value=operating_income).fill = calc_fill
                        sga_row += 1
                else:
                    if account in section_data:
                        row_data = section_data[account]
                        for col_idx, value in enumerate(row_data, start=1):
                            sga_ws.cell(row=sga_row, column=col_idx, value=value)
                        sga_row += 1
            
            if current_row < max_row:
                while current_row <= max_row and not pivot_ws.cell(row=current_row, column=1).value:
                    sga_row += 1
                    current_row += 1
        else:
            current_row += 1
    
    sga_ws.column_dimensions['A'].width = 25
    for col_idx in range(2, max_col + 1):
        sga_ws.column_dimensions[get_column_letter(col_idx)].width = 18
    
    wb.save(merged_pl_file)
    
    print(f"✓ SG&A(FCY) 시트 생성 완료 ({section_count}개 섹션)")
    
    return True

def create_analysis_horizontal_fcy(merged_pl_file, target_period="20254Q"):
    """Analysis(FCY) 시트 생성 - Pivot(FCY) 기반"""
    print("\n" + "="*60)
    print(f"[Step 4-FCY] Analysis(FCY) 시트 생성 (가로 배치)")
    print("="*60)
    
    wb = openpyxl.load_workbook(merged_pl_file)
    
    if 'Pivot(FCY)' not in wb.sheetnames:
        print("❌ Pivot(FCY) 시트가 없습니다.")
        return None
    
    pivot_ws = wb['Pivot(FCY)']
    
    if 'Analysis(FCY)' in wb.sheetnames:
        del wb['Analysis(FCY)']
    analysis_ws = wb.create_sheet('Analysis(FCY)')
    
    prev_quarter = get_previous_quarter(target_period)
    yoy_quarter = get_yoy_quarter(target_period)
    
    target_display = format_period_display(target_period)
    prev_display = format_period_display(prev_quarter)
    yoy_display = format_period_display(yoy_quarter)
    
    print(f"\n분석 분기:")
    print(f"  - QoQ: {prev_display} → {target_display}")
    print(f"  - YoY(Q): {yoy_display} → {target_display}")
    print(f"  - YoY(Y): {yoy_display}(누적) → {target_display}(누적)")
    
    header_fill = PatternFill(start_color='4472C4', end_color='4472C4', fill_type='solid')
    header_font = Font(bold=True, color='FFFFFF')
    section_fill = PatternFill(start_color='FFC000', end_color='FFC000', fill_type='solid')
    section_font = Font(bold=True, size=11, color='FFFFFF')
    increase_fill = PatternFill(start_color='C6EFCE', end_color='C6EFCE', fill_type='solid')
    decrease_fill = PatternFill(start_color='FFC7CE', end_color='FFC7CE', fill_type='solid')
    
    # Entity 목록
    entities = []
    search_pattern = f"{target_period} (누적)"
    for row in range(1, pivot_ws.max_row + 1):
        cell = pivot_ws.cell(row=row, column=1)
        if cell.value and search_pattern in str(cell.value):
            row += 1
            for col in range(2, pivot_ws.max_column + 1):
                entity = pivot_ws.cell(row=row, column=col).value
                if entity and entity not in ['합계', None] and entity not in entities:
                    entities.append(entity)
            break
    
    entities = sort_entities(entities)
    print(f"\n발견된 Entity (정렬됨): {entities}")
    
    def read_section_data(section_name, entity_name='합계'):
        data = {}
        max_row = pivot_ws.max_row
        
        for row in range(1, max_row + 1):
            cell = pivot_ws.cell(row=row, column=1)
            if cell.value and isinstance(cell.value, str) and section_name in cell.value:
                row += 1
                
                target_col = None
                max_col = pivot_ws.max_column
                for col in range(1, max_col + 1):
                    if pivot_ws.cell(row=row, column=col).value == entity_name:
                        target_col = col
                        break
                
                if not target_col:
                    continue
                
                row += 1
                
                while row <= max_row:
                    rev_account = pivot_ws.cell(row=row, column=1).value
                    if not rev_account or (isinstance(rev_account, str) and ('누적' in rev_account or '분기별' in rev_account)):
                        break
                    
                    value = pivot_ws.cell(row=row, column=target_col).value
                    try:
                        data[rev_account] = float(value) if value else 0
                    except (ValueError, TypeError):
                        data[rev_account] = 0
                    
                    row += 1
                
                break
        
        return data
    
    def write_horizontal_analysis(ws, start_row, entity_name, qoq_base, qoq_compare, yoyq_base, yoyq_compare, yoyy_base, yoyy_compare):
        current_row = start_row
        
        ws.merge_cells(start_row=current_row, start_column=1, end_row=current_row, end_column=13)
        entity_cell = ws.cell(row=current_row, column=1)
        entity_cell.value = f"[{entity_name}]"
        entity_cell.fill = section_fill
        entity_cell.font = section_font
        entity_cell.alignment = Alignment(horizontal='center', vertical='center')
        current_row += 1
        
        headers = [
            'Rev_Account',
            f'{prev_display}', f'{target_display}', 'QoQ_증감액', 'QoQ_증감률(%)',
            f'{yoy_display}', f'{target_display}', 'YoY(Q)_증감액', 'YoY(Q)_증감률(%)',
            f'{yoy_display}(누적)', f'{target_display}(누적)', 'YoY(Y)_증감액', 'YoY(Y)_증감률(%)'
        ]
        
        for col_idx, header in enumerate(headers, start=1):
            cell = ws.cell(row=current_row, column=col_idx)
            cell.value = header
            cell.fill = header_fill
            cell.font = header_font
            cell.alignment = Alignment(horizontal='center', vertical='center')
        current_row += 1
        
        all_accounts = set()
        all_accounts.update(qoq_base.keys())
        all_accounts.update(qoq_compare.keys())
        all_accounts.update(yoyq_base.keys())
        all_accounts.update(yoyq_compare.keys())
        all_accounts.update(yoyy_base.keys())
        all_accounts.update(yoyy_compare.keys())
        all_accounts = sort_rev_accounts(list(all_accounts))
        
        for account in all_accounts:
            col = 1
            ws.cell(row=current_row, column=col, value=account)
            col += 1
            
            # QoQ
            qoq_b = qoq_base.get(account, 0)
            qoq_c = qoq_compare.get(account, 0)
            qoq_change = qoq_c - qoq_b
            qoq_rate = (qoq_change / qoq_b * 100) if qoq_b != 0 else (0 if qoq_change == 0 else float('inf'))
            
            ws.cell(row=current_row, column=col, value=qoq_b); col += 1
            ws.cell(row=current_row, column=col, value=qoq_c); col += 1
            
            change_cell = ws.cell(row=current_row, column=col, value=qoq_change)
            if qoq_change > 0:
                change_cell.fill = increase_fill
            elif qoq_change < 0:
                change_cell.fill = decrease_fill
            col += 1
            
            if qoq_rate != float('inf'):
                rate_cell = ws.cell(row=current_row, column=col, value=qoq_rate)
                rate_cell.number_format = '0.00'
                if qoq_rate > 0:
                    rate_cell.fill = increase_fill
                elif qoq_rate < 0:
                    rate_cell.fill = decrease_fill
            else:
                ws.cell(row=current_row, column=col, value='N/A')
            col += 1
            
            # YoY(Q)
            yoyq_b = yoyq_base.get(account, 0)
            yoyq_c = yoyq_compare.get(account, 0)
            yoyq_change = yoyq_c - yoyq_b
            yoyq_rate = (yoyq_change / yoyq_b * 100) if yoyq_b != 0 else (0 if yoyq_change == 0 else float('inf'))
            
            ws.cell(row=current_row, column=col, value=yoyq_b); col += 1
            ws.cell(row=current_row, column=col, value=yoyq_c); col += 1
            
            change_cell = ws.cell(row=current_row, column=col, value=yoyq_change)
            if yoyq_change > 0:
                change_cell.fill = increase_fill
            elif yoyq_change < 0:
                change_cell.fill = decrease_fill
            col += 1
            
            if yoyq_rate != float('inf'):
                rate_cell = ws.cell(row=current_row, column=col, value=yoyq_rate)
                rate_cell.number_format = '0.00'
                if yoyq_rate > 0:
                    rate_cell.fill = increase_fill
                elif yoyq_rate < 0:
                    rate_cell.fill = decrease_fill
            else:
                ws.cell(row=current_row, column=col, value='N/A')
            col += 1
            
            # YoY(Y)
            yoyy_b = yoyy_base.get(account, 0)
            yoyy_c = yoyy_compare.get(account, 0)
            yoyy_change = yoyy_c - yoyy_b
            yoyy_rate = (yoyy_change / yoyy_b * 100) if yoyy_b != 0 else (0 if yoyy_change == 0 else float('inf'))
            
            ws.cell(row=current_row, column=col, value=yoyy_b); col += 1
            ws.cell(row=current_row, column=col, value=yoyy_c); col += 1
            
            change_cell = ws.cell(row=current_row, column=col, value=yoyy_change)
            if yoyy_change > 0:
                change_cell.fill = increase_fill
            elif yoyy_change < 0:
                change_cell.fill = decrease_fill
            col += 1
            
            if yoyy_rate != float('inf'):
                rate_cell = ws.cell(row=current_row, column=col, value=yoyy_rate)
                rate_cell.number_format = '0.00'
                if yoyy_rate > 0:
                    rate_cell.fill = increase_fill
                elif yoyy_rate < 0:
                    rate_cell.fill = decrease_fill
            else:
                ws.cell(row=current_row, column=col, value='N/A')
            
            current_row += 1
        
        return current_row
    
    current_row = 1
    
    print("\n✓ 전체 합계 분석 (Pivot(FCY) 기준)")
    qoq_prev_total = read_section_data(f"{prev_quarter} (분기별)", "합계")
    qoq_target_total = read_section_data(f"{target_period} (분기별)", "합계")
    yoyq_base_total = read_section_data(f"{yoy_quarter} (분기별)", "합계")
    yoyq_target_total = qoq_target_total
    yoyy_base_total = read_section_data(f"{yoy_quarter} (누적)", "합계")
    yoyy_target_total = read_section_data(f"{target_period} (누적)", "합계")
    
    current_row = write_horizontal_analysis(
        analysis_ws, current_row, "전체 합계",
        qoq_prev_total, qoq_target_total,
        yoyq_base_total, yoyq_target_total,
        yoyy_base_total, yoyy_target_total
    )
    current_row += 2
    
    for entity in entities:
        print(f"  - {entity}")
        qoq_prev_entity = read_section_data(f"{prev_quarter} (분기별)", entity)
        qoq_target_entity = read_section_data(f"{target_period} (분기별)", entity)
        yoyq_base_entity = read_section_data(f"{yoy_quarter} (분기별)", entity)
        yoyq_target_entity = qoq_target_entity
        yoyy_base_entity = read_section_data(f"{yoy_quarter} (누적)", entity)
        yoyy_target_entity = read_section_data(f"{target_period} (누적)", entity)
        
        current_row = write_horizontal_analysis(
            analysis_ws, current_row, entity,
            qoq_prev_entity, qoq_target_entity,
            yoyq_base_entity, yoyq_target_entity,
            yoyy_base_entity, yoyy_target_entity
        )
        current_row += 2
    
    analysis_ws.column_dimensions['A'].width = 25
    for col_idx in range(2, 14):
        analysis_ws.column_dimensions[get_column_letter(col_idx)].width = 15
    
    wb.save(merged_pl_file)
    
    total_tables = 1 + len(entities)
    print(f"\n✓ Analysis(FCY) 시트 생성 완료 ({total_tables}개 테이블, Pivot(FCY) 기준)")
    
    return True

def create_analysis_sga_horizontal_fcy(merged_pl_file, target_period="20254Q"):
    """Analysis_SG&A(FCY) 시트 생성 - SG&A(FCY) 기반"""
    print("\n" + "="*60)
    print(f"[Step 5-FCY] Analysis_SG&A(FCY) 시트 생성 (가로 배치)")
    print("="*60)
    
    wb = openpyxl.load_workbook(merged_pl_file)
    
    if 'SG&A(FCY)' not in wb.sheetnames:
        print("❌ SG&A(FCY) 시트가 없습니다.")
        return None
    
    sga_ws = wb['SG&A(FCY)']
    
    if 'Analysis_SG&A(FCY)' in wb.sheetnames:
        del wb['Analysis_SG&A(FCY)']
    analysis_sga_ws = wb.create_sheet('Analysis_SG&A(FCY)')
    
    prev_quarter = get_previous_quarter(target_period)
    yoy_quarter = get_yoy_quarter(target_period)
    
    target_display = format_period_display(target_period)
    prev_display = format_period_display(prev_quarter)
    yoy_display = format_period_display(yoy_quarter)
    
    print(f"\n분석 분기:")
    print(f"  - QoQ: {prev_display} → {target_display}")
    print(f"  - YoY(Q): {yoy_display} → {target_display}")
    print(f"  - YoY(Y): {yoy_display}(누적) → {target_display}(누적)")
    
    header_fill = PatternFill(start_color='4472C4', end_color='4472C4', fill_type='solid')
    header_font = Font(bold=True, color='FFFFFF')
    section_fill = PatternFill(start_color='FFC000', end_color='FFC000', fill_type='solid')
    section_font = Font(bold=True, size=11, color='FFFFFF')
    increase_fill = PatternFill(start_color='C6EFCE', end_color='C6EFCE', fill_type='solid')
    decrease_fill = PatternFill(start_color='FFC7CE', end_color='FFC7CE', fill_type='solid')
    
    sga_accounts = [
        '매출', '매출원가', '매출총이익', '판관비',
        '인건비', '복리후생비', '광고선전비', '지급수수료', '운반비',
        '경상연구개발비', '판매수수료', '여비교통비','대손상각비','감가상각비', '사용권자산상각비','기타', '영업이익'
    ]
    
    # Entity 목록
    entities = []
    search_pattern = f"{target_period} (누적)"
    for row in range(1, sga_ws.max_row + 1):
        cell = sga_ws.cell(row=row, column=1)
        if cell.value and search_pattern in str(cell.value):
            row += 1
            for col in range(2, sga_ws.max_column + 1):
                entity = sga_ws.cell(row=row, column=col).value
                if entity and entity not in ['합계', None] and entity not in entities:
                    entities.append(entity)
            break
    
    entities = sort_entities(entities)
    print(f"\n발견된 Entity (정렬됨): {entities}")
    
    def read_section_data(section_name, entity_name='합계'):
        data = {}
        max_row = sga_ws.max_row
        
        for row in range(1, max_row + 1):
            cell = sga_ws.cell(row=row, column=1)
            if cell.value and isinstance(cell.value, str) and section_name in cell.value:
                row += 1
                
                target_col = None
                max_col = sga_ws.max_column
                for col in range(1, max_col + 1):
                    if sga_ws.cell(row=row, column=col).value == entity_name:
                        target_col = col
                        break
                
                if not target_col:
                    continue
                
                row += 1
                
                while row <= max_row:
                    rev_account = sga_ws.cell(row=row, column=1).value
                    if not rev_account or (isinstance(rev_account, str) and ('누적' in rev_account or '분기별' in rev_account)):
                        break
                    
                    value = sga_ws.cell(row=row, column=target_col).value
                    try:
                        data[rev_account] = float(value) if value else 0
                    except (ValueError, TypeError):
                        data[rev_account] = 0
                    
                    row += 1
                
                break
        
        return data
    
    def write_horizontal_sga_analysis(ws, start_row, entity_name, qoq_base, qoq_compare, yoyq_base, yoyq_compare, yoyy_base, yoyy_compare):
        current_row = start_row
        
        ws.merge_cells(start_row=current_row, start_column=1, end_row=current_row, end_column=13)
        entity_cell = ws.cell(row=current_row, column=1)
        entity_cell.value = f"[{entity_name}]"
        entity_cell.fill = section_fill
        entity_cell.font = section_font
        entity_cell.alignment = Alignment(horizontal='center', vertical='center')
        current_row += 1
        
        headers = [
            'Rev_Account',
            f'{prev_display}', f'{target_display}', 'QoQ_증감액', 'QoQ_증감률(%)',
            f'{yoy_display}', f'{target_display}', 'YoY(Q)_증감액', 'YoY(Q)_증감률(%)',
            f'{yoy_display}(누적)', f'{target_display}(누적)', 'YoY(Y)_증감액', 'YoY(Y)_증감률(%)'
        ]
        
        for col_idx, header in enumerate(headers, start=1):
            cell = ws.cell(row=current_row, column=col_idx)
            cell.value = header
            cell.fill = header_fill
            cell.font = header_font
            cell.alignment = Alignment(horizontal='center', vertical='center')
        current_row += 1
        
        for account in sga_accounts:
            col = 1
            ws.cell(row=current_row, column=col, value=account)
            col += 1
            
            # QoQ
            qoq_b = qoq_base.get(account, 0)
            qoq_c = qoq_compare.get(account, 0)
            qoq_change = qoq_c - qoq_b
            qoq_rate = (qoq_change / qoq_b * 100) if qoq_b != 0 else (0 if qoq_change == 0 else float('inf'))
            
            ws.cell(row=current_row, column=col, value=qoq_b); col += 1
            ws.cell(row=current_row, column=col, value=qoq_c); col += 1
            
            change_cell = ws.cell(row=current_row, column=col, value=qoq_change)
            if qoq_change > 0:
                change_cell.fill = increase_fill
            elif qoq_change < 0:
                change_cell.fill = decrease_fill
            col += 1
            
            if qoq_rate != float('inf'):
                rate_cell = ws.cell(row=current_row, column=col, value=qoq_rate)
                rate_cell.number_format = '0.00'
                if qoq_rate > 0:
                    rate_cell.fill = increase_fill
                elif qoq_rate < 0:
                    rate_cell.fill = decrease_fill
            else:
                ws.cell(row=current_row, column=col, value='N/A')
            col += 1
            
            # YoY(Q)
            yoyq_b = yoyq_base.get(account, 0)
            yoyq_c = yoyq_compare.get(account, 0)
            yoyq_change = yoyq_c - yoyq_b
            yoyq_rate = (yoyq_change / yoyq_b * 100) if yoyq_b != 0 else (0 if yoyq_change == 0 else float('inf'))
            
            ws.cell(row=current_row, column=col, value=yoyq_b); col += 1
            ws.cell(row=current_row, column=col, value=yoyq_c); col += 1
            
            change_cell = ws.cell(row=current_row, column=col, value=yoyq_change)
            if yoyq_change > 0:
                change_cell.fill = increase_fill
            elif yoyq_change < 0:
                change_cell.fill = decrease_fill
            col += 1
            
            if yoyq_rate != float('inf'):
                rate_cell = ws.cell(row=current_row, column=col, value=yoyq_rate)
                rate_cell.number_format = '0.00'
                if yoyq_rate > 0:
                    rate_cell.fill = increase_fill
                elif yoyq_rate < 0:
                    rate_cell.fill = decrease_fill
            else:
                ws.cell(row=current_row, column=col, value='N/A')
            col += 1
            
            # YoY(Y)
            yoyy_b = yoyy_base.get(account, 0)
            yoyy_c = yoyy_compare.get(account, 0)
            yoyy_change = yoyy_c - yoyy_b
            yoyy_rate = (yoyy_change / yoyy_b * 100) if yoyy_b != 0 else (0 if yoyy_change == 0 else float('inf'))
            
            ws.cell(row=current_row, column=col, value=yoyy_b); col += 1
            ws.cell(row=current_row, column=col, value=yoyy_c); col += 1
            
            change_cell = ws.cell(row=current_row, column=col, value=yoyy_change)
            if yoyy_change > 0:
                change_cell.fill = increase_fill
            elif yoyy_change < 0:
                change_cell.fill = decrease_fill
            col += 1
            
            if yoyy_rate != float('inf'):
                rate_cell = ws.cell(row=current_row, column=col, value=yoyy_rate)
                rate_cell.number_format = '0.00'
                if yoyy_rate > 0:
                    rate_cell.fill = increase_fill
                elif yoyy_rate < 0:
                    rate_cell.fill = decrease_fill
            else:
                ws.cell(row=current_row, column=col, value='N/A')
            
            current_row += 1
        
        return current_row
    
    current_row = 1
    
    print("\n✓ 전체 합계 분석")
    qoq_prev_total = read_section_data(f"{prev_quarter} (분기별)", "합계")
    qoq_target_total = read_section_data(f"{target_period} (분기별)", "합계")
    yoyq_base_total = read_section_data(f"{yoy_quarter} (분기별)", "합계")
    yoyq_target_total = qoq_target_total
    yoyy_base_total = read_section_data(f"{yoy_quarter} (누적)", "합계")
    yoyy_target_total = read_section_data(f"{target_period} (누적)", "합계")
    
    current_row = write_horizontal_sga_analysis(
        analysis_sga_ws, current_row, "전체 합계",
        qoq_prev_total, qoq_target_total,
        yoyq_base_total, yoyq_target_total,
        yoyy_base_total, yoyy_target_total
    )
    current_row += 2
    
    for entity in entities:
        print(f"  - {entity}")
        qoq_prev_entity = read_section_data(f"{prev_quarter} (분기별)", entity)
        qoq_target_entity = read_section_data(f"{target_period} (분기별)", entity)
        yoyq_base_entity = read_section_data(f"{yoy_quarter} (분기별)", entity)
        yoyq_target_entity = qoq_target_entity
        yoyy_base_entity = read_section_data(f"{yoy_quarter} (누적)", entity)
        yoyy_target_entity = read_section_data(f"{target_period} (누적)", entity)
        
        current_row = write_horizontal_sga_analysis(
            analysis_sga_ws, current_row, entity,
            qoq_prev_entity, qoq_target_entity,
            yoyq_base_entity, yoyq_target_entity,
            yoyy_base_entity, yoyy_target_entity
        )
        current_row += 2
    
    analysis_sga_ws.column_dimensions['A'].width = 25
    for col_idx in range(2, 14):
        analysis_sga_ws.column_dimensions[get_column_letter(col_idx)].width = 15
    
    wb.save(merged_pl_file)
    
    total_tables = 1 + len(entities)
    print(f"\n✓ Analysis_SG&A(FCY) 시트 생성 완료 ({total_tables}개 테이블)")
    
    return True

def create_sheet_total_pl_fcy(merged_pl_file, target_period="20254Q"):
    """Sheet_total PL(FCY) 시트 생성 - SG&A(FCY) 기반"""
    print("\n" + "="*60)
    print(f"[Step 6-FCY] Sheet_total PL(FCY) 시트 생성")
    print("="*60)
    
    wb = openpyxl.load_workbook(merged_pl_file)
    
    if 'SG&A(FCY)' not in wb.sheetnames:
        print("❌ SG&A(FCY) 시트가 없습니다.")
        return None
    
    sga_ws = wb['SG&A(FCY)']
    
    if 'Sheet_total PL(FCY)' in wb.sheetnames:
        del wb['Sheet_total PL(FCY)']
    total_pl_ws = wb.create_sheet('Sheet_total PL(FCY)')
    
    prev_quarter = get_previous_quarter(target_period)
    yoy_quarter = get_yoy_quarter(target_period)
    
    target_display = format_period_display(target_period)
    prev_display = format_period_display(prev_quarter)
    yoy_display = format_period_display(yoy_quarter)
    
    print(f"\n분석 분기:")
    print(f"  - QoQ: {prev_display} → {target_display}")
    print(f"  - YoY(Q): {yoy_display} → {target_display}")
    print(f"  - YoY(Y): {yoy_display}(누적) → {target_display}(누적)")
    
    header_fill = PatternFill(start_color='4472C4', end_color='4472C4', fill_type='solid')
    header_font = Font(bold=True, color='FFFFFF')
    section_fill = PatternFill(start_color='FFC000', end_color='FFC000', fill_type='solid')
    section_font = Font(bold=True, size=11, color='FFFFFF')
    increase_fill = PatternFill(start_color='C6EFCE', end_color='C6EFCE', fill_type='solid')
    decrease_fill = PatternFill(start_color='FFC7CE', end_color='FFC7CE', fill_type='solid')
    
    # Entity 목록
    entities = []
    search_pattern = f"{target_period} (누적)"
    for row in range(1, sga_ws.max_row + 1):
        cell = sga_ws.cell(row=row, column=1)
        if cell.value and search_pattern in str(cell.value):
            row += 1
            for col in range(2, sga_ws.max_column + 1):
                entity = sga_ws.cell(row=row, column=col).value
                if entity and entity not in entities:
                    entities.append(entity)
            break
    
    entities = sort_entities(entities)
    print(f"\nEntity (정렬됨): {entities}")
    
    def read_all_entities_data(section_name):
        entity_data = {}
        
        for entity in entities:
            data = {}
            max_row = sga_ws.max_row
            
            for row in range(1, max_row + 1):
                cell = sga_ws.cell(row=row, column=1)
                if cell.value and isinstance(cell.value, str) and section_name in cell.value:
                    row += 1
                    
                    target_col = None
                    max_col = sga_ws.max_column
                    for col in range(1, max_col + 1):
                        if sga_ws.cell(row=row, column=col).value == entity:
                            target_col = col
                            break
                    
                    if not target_col:
                        continue
                    
                    row += 1
                    
                    while row <= max_row:
                        rev_account = sga_ws.cell(row=row, column=1).value
                        if not rev_account or (isinstance(rev_account, str) and ('누적' in rev_account or '분기별' in rev_account)):
                            break
                        
                        value = sga_ws.cell(row=row, column=target_col).value
                        try:
                            data[rev_account] = float(value) if value else 0
                        except (ValueError, TypeError):
                            data[rev_account] = 0
                        
                        row += 1
                    
                    break
            
            entity_data[entity] = data
        
        return entity_data
    
    def write_entity_table(ws, start_row, title, entity_data_dict, value_type='value'):
        current_row = start_row
        
        ws.merge_cells(start_row=current_row, start_column=1, end_row=current_row, end_column=len(entities) + 1)
        title_cell = ws.cell(row=current_row, column=1)
        title_cell.value = title
        title_cell.fill = section_fill
        title_cell.font = section_font
        title_cell.alignment = Alignment(horizontal='center', vertical='center')
        current_row += 1
        
        ws.cell(row=current_row, column=1, value='Rev_Account').fill = header_fill
        ws.cell(row=current_row, column=1).font = header_font
        ws.cell(row=current_row, column=1).alignment = Alignment(horizontal='center', vertical='center')
        
        for col_idx, entity in enumerate(entities, start=2):
            cell = ws.cell(row=current_row, column=col_idx)
            cell.value = entity
            cell.fill = header_fill
            cell.font = header_font
            cell.alignment = Alignment(horizontal='center', vertical='center')
        current_row += 1
        
        sga_accounts = [
            '매출', '매출원가', '매출총이익', '판관비',
            '인건비', '복리후생비', '광고선전비', '지급수수료', '운반비',
            '경상연구개발비', '판매수수료', '여비교통비','대손상각비','감가상각비', '사용권자산상각비','기타', '영업이익'
        ]
        
        for account in sga_accounts:
            ws.cell(row=current_row, column=1, value=account)
            
            for col_idx, entity in enumerate(entities, start=2):
                value = entity_data_dict.get(entity, {}).get(account, 0)
                cell = ws.cell(row=current_row, column=col_idx, value=value)
                
                if value_type in ['change', 'rate']:
                    if value > 0:
                        cell.fill = increase_fill
                    elif value < 0:
                        cell.fill = decrease_fill
                
                if value_type == 'rate' and value != float('inf'):
                    cell.number_format = '0.00'
            
            current_row += 1
        
        return current_row
    
    def calculate_changes(base_data, compare_data):
        change_data = {}
        rate_data = {}
        
        for entity in entities:
            base = base_data.get(entity, {})
            compare = compare_data.get(entity, {})
            
            entity_change = {}
            entity_rate = {}
            
            all_accounts = set()
            all_accounts.update(base.keys())
            all_accounts.update(compare.keys())
            
            for account in all_accounts:
                base_val = base.get(account, 0)
                compare_val = compare.get(account, 0)
                change = compare_val - base_val
                
                if base_val != 0:
                    rate = (change / base_val) * 100
                else:
                    rate = 0 if change == 0 else float('inf')
                
                entity_change[account] = change
                entity_rate[account] = rate
            
            change_data[entity] = entity_change
            rate_data[entity] = entity_rate
        
        return change_data, rate_data
    
    current_row = 1
    
    # QoQ 분석
    print("\n✓ QoQ 테이블 생성")
    qoq_prev = read_all_entities_data(f"{prev_quarter} (분기별)")
    qoq_target = read_all_entities_data(f"{target_period} (분기별)")
    qoq_change, qoq_rate = calculate_changes(qoq_prev, qoq_target)
    
    current_row = write_entity_table(total_pl_ws, current_row, 
                                     f"QoQ 분석: {prev_display}", qoq_prev, 'value')
    current_row += 1
    
    current_row = write_entity_table(total_pl_ws, current_row, 
                                     f"QoQ 분석: {target_display}", qoq_target, 'value')
    current_row += 1
    
    current_row = write_entity_table(total_pl_ws, current_row, 
                                     f"QoQ 분석: 증감액 ({prev_display} → {target_display})", qoq_change, 'change')
    current_row += 1
    
    current_row = write_entity_table(total_pl_ws, current_row, 
                                     f"QoQ 분석: 증감률(%) ({prev_display} → {target_display})", qoq_rate, 'rate')
    current_row += 2
    
    # YoY(Q) 분석
    print("✓ YoY(Q) 테이블 생성")
    yoyq_base = read_all_entities_data(f"{yoy_quarter} (분기별)")
    yoyq_target = qoq_target
    yoyq_change, yoyq_rate = calculate_changes(yoyq_base, yoyq_target)
    
    current_row = write_entity_table(total_pl_ws, current_row, 
                                     f"YoY(Q) 분석: {yoy_display}", yoyq_base, 'value')
    current_row += 1
    
    current_row = write_entity_table(total_pl_ws, current_row, 
                                     f"YoY(Q) 분석: {target_display}", yoyq_target, 'value')
    current_row += 1
    
    current_row = write_entity_table(total_pl_ws, current_row, 
                                     f"YoY(Q) 분석: 증감액 ({yoy_display} → {target_display})", yoyq_change, 'change')
    current_row += 1
    
    current_row = write_entity_table(total_pl_ws, current_row, 
                                     f"YoY(Q) 분석: 증감률(%) ({yoy_display} → {target_display})", yoyq_rate, 'rate')
    current_row += 2
    
    # YoY(Y) 분석
    print("✓ YoY(Y) 테이블 생성")
    yoyy_base = read_all_entities_data(f"{yoy_quarter} (누적)")
    yoyy_target = read_all_entities_data(f"{target_period} (누적)")
    yoyy_change, yoyy_rate = calculate_changes(yoyy_base, yoyy_target)
    
    current_row = write_entity_table(total_pl_ws, current_row, 
                                     f"YoY(Y) 분석: {yoy_display}(누적)", yoyy_base, 'value')
    current_row += 1
    
    current_row = write_entity_table(total_pl_ws, current_row, 
                                     f"YoY(Y) 분석: {target_display}(누적)", yoyy_target, 'value')
    current_row += 1
    
    current_row = write_entity_table(total_pl_ws, current_row, 
                                     f"YoY(Y) 분석: 증감액 ({yoy_display} → {target_display})", yoyy_change, 'change')
    current_row += 1
    
    current_row = write_entity_table(total_pl_ws, current_row, 
                                     f"YoY(Y) 분석: 증감률(%) ({yoy_display} → {target_display})", yoyy_rate, 'rate')
    
    total_pl_ws.column_dimensions['A'].width = 25
    for col_idx in range(2, len(entities) + 2):
        total_pl_ws.column_dimensions[get_column_letter(col_idx)].width = 15
    
    wb.save(merged_pl_file)
    
    print(f"\n✓ Sheet_total PL(FCY) 시트 생성 완료 (12개 테이블)")
    
    return True

# 실행
if __name__ == "__main__":
    merged_pl_file = r"C:\Users\user\InBody Co., Ltd\회계팀 - General\★★Accounting Folder★★vF\1. 재무결산\FY2025\2025_12_4Q\2. 연결재무제표\Income statement\InBody_PL_Analysis.xlsx"
    target_period = "20254Q"
    
    print("="*60)
    print("FCY (외화) 시트 생성")
    print(f"분석 기준 분기: {format_period_display(target_period)}")
    print("="*60)
    
    # Step 2-FCY: Pivot(FCY)
    pivot_df_fcy = create_pivot_sheet_fcy(merged_pl_file)
    
    if pivot_df_fcy is not None:
        # Step 3-FCY: SG&A(FCY)
        sga_result_fcy = create_sga_sheet_fcy(merged_pl_file)
        
        if sga_result_fcy:
            # Step 4-FCY: Analysis(FCY)
            analysis_result_fcy = create_analysis_horizontal_fcy(merged_pl_file, target_period)
            
            if analysis_result_fcy:
                # Step 5-FCY: Analysis_SG&A(FCY)
                analysis_sga_result_fcy = create_analysis_sga_horizontal_fcy(merged_pl_file, target_period)
                
                if analysis_sga_result_fcy:
                    # Step 6-FCY: Sheet_total PL(FCY)
                    total_pl_result_fcy = create_sheet_total_pl_fcy(merged_pl_file, target_period)
                    
                    if total_pl_result_fcy:
                        print("\n" + "="*60)
                        print("✅ FCY 시트 생성 완료!")
                        print("="*60)
                        print(f"\n생성된 파일: {merged_pl_file}")
                        print(f"\n【FCY 기준 시트】")
                        print(f"  - Pivot(FCY): Entity별 누적 + 분기별 (외화)")
                        print(f"  - SG&A(FCY): 매출총이익, 판관비, 영업이익 계산 (외화)")
                        print(f"  - Analysis(FCY): QoQ | YoY(Q) | YoY(Y) 가로 배치 (외화)")
                        print(f"  - Analysis_SG&A(FCY): QoQ | YoY(Q) | YoY(Y) 가로 배치 (외화)")
                        print(f"  - Sheet_total PL(FCY): Entity 가로 배치 (외화)")
                    else:
                        print("\n❌ Step 6-FCY (Sheet_total PL(FCY) 생성) 실패")
                else:
                    print("\n❌ Step 5-FCY (Analysis_SG&A(FCY) 생성) 실패")
            else:
                print("\n❌ Step 4-FCY (Analysis(FCY) 생성) 실패")
        else:
            print("\n❌ Step 3-FCY (SG&A(FCY) 생성) 실패")
    else:
        print("\n❌ Step 2-FCY (Pivot(FCY) 생성) 실패")

FCY (외화) 시트 생성
분석 기준 분기: 2025_4Q

[Step 2-FCY] Pivot(FCY) 시트 생성 시작

파일 로드 중: C:\Users\user\InBody Co., Ltd\회계팀 - General\★★Accounting Folder★★vF\1. 재무결산\FY2025\2025_12_4Q\2. 연결재무제표\Income statement\InBody_PL_Analysis.xlsx
✓ 데이터 로드 완료: 9604 행

✓ Entity 이름 변경 적용 중...

✓ Period 유니크 값: ['20243Q', '20244Q', '20251Q', '20252Q', '20253Q', '20254Q']
✓ Entity 유니크 값: 17개

피벗테이블 생성 중 (Amount 기준 - FCY)...
✓ 피벗테이블 생성 완료

저장 중... (Pivot(FCY) 시트)
✓ Pivot(FCY) 시트 추가 완료 (총 15개 섹션)

[Step 3-FCY] SG&A(FCY) 시트 생성 시작
✓ SG&A(FCY) 시트 생성 완료 (14개 섹션)

[Step 4-FCY] Analysis(FCY) 시트 생성 (가로 배치)

분석 분기:
  - QoQ: 2025_3Q → 2025_4Q
  - YoY(Q): 2024_4Q → 2025_4Q
  - YoY(Y): 2024_4Q(누적) → 2025_4Q(누적)

발견된 Entity (정렬됨): ['HQ', 'USA', 'Japan', 'China', 'Europe', 'Asia', 'India', 'Mexico', 'Oceania', 'BWA', 'Vietnam', 'Turkey', 'KOROT', '헬스케어', '삼한정공', 'KOCP', '연결조정']

✓ 전체 합계 분석 (Pivot(FCY) 기준)
  - HQ
  - USA
  - Japan
  - China
  - Europe
  - Asia
  - India
  - Mexico
  - Oceania
  - BWA
  - Vietnam
  - Turkey
  - KORO

# 데이터 변환

##신규 분기때마다 직전분기 코드를 복사해서 PERIOD 를 수정해주세요
##파일 위치는 해당 폴더 위치를 복사해서 붙여 주세요.


In [5]:
import pandas as pd
import os
from pathlib import Path

# ==================== 여기를 수정하세요! ====================
# 기간 설정
PERIOD = "20254Q"  # 예: "2025_4Q", "2025_12월", "FY2025" 등

# Entity별 통화 및 환율 매핑 (BS: 기말환율, PL: 평균환율)
ENTITY_CURRENCY_MAP = {
    'Biospace Inc. DBA INBODY': {
        'currency': 'USD', 
        'bs_rate': 1434.90,  # BS 기말환율
        'pl_rate': 1422.22   # PL 평균환율
    },
    'InBody Japan Inc.': {
        'currency': 'JPY', 
        'bs_rate': 9.1763,
        'pl_rate': 9.5079
    },
    'Biospace Co.,Ltd.': {
        'currency': 'CNH', 
        'bs_rate': 204.76,
        'pl_rate': 197.78
    },
    'InBody Oceania ': {
        'currency': 'AUD', 
        'bs_rate': 961.02,
        'pl_rate': 916.97
    },
    'BIOSPACE LATIN AMERICA S DE RL DE CV(*)': {
        'currency': 'MXN', 
        'bs_rate': 79.77,
        'pl_rate': 74.2
    },
    'BWA': {
        'currency': 'USD', 
        'bs_rate': 1434.90,
        'pl_rate': 1422.22
    },
    '㈜인바디헬스케어': {
        'currency': 'KRW', 
        'bs_rate': 1.0,
        'pl_rate': 1.0
    },
    'InBody Vietnam': {
        'currency': 'VND', 
        'bs_rate': 0.0546,
        'pl_rate': 0.0547
    },
    'INBODY ASIA SDN. BHD.': {
        'currency': 'MYR', 
        'bs_rate': 354.56,
        'pl_rate': 332.18
    },
    'InBody India Pvt. Ltd': {
        'currency': 'INR', 
        'bs_rate': 15.98,
        'pl_rate': 16.32
    },
    'Inbody Europe B.V.': {
        'currency': 'EUR', 
        'bs_rate': 1685.72,
        'pl_rate': 1607.46
    },
    '㈜코르트': {
        'currency': 'KRW', 
        'bs_rate': 1.0,
        'pl_rate': 1.0
    },
    'INBODY TURKEY MEDİKAL TİCARET LİMİTED ŞİRKETİ': {
        'currency': 'TRY', 
        'bs_rate': 33.40,
        'pl_rate': 36.05
    },
    '(주)삼한정공': {
        'currency': 'KRW', 
        'bs_rate': 1.0,
        'pl_rate': 1.0
    },
    '주식회사 인바디': {
        'currency': 'KRW', 
        'bs_rate': 1.0,
        'pl_rate': 1.0
    },
    '케이오씨피 프로젝트 제2호 벤처투자조합': {
        'currency': 'KRW', 
        'bs_rate': 1.0,
        'pl_rate': 1.0
    },
    'ADJ': {
        'currency': 'KRW', 
        'bs_rate': 1.0,
        'pl_rate': 1.0
    },
}
# ============================================================

def preprocess_sheet(df, sheet_type):
    """데이터 전처리: H~J → K, M~O → P로 이동"""
    # H~J 열 데이터를 K로 이동 (0-indexed: 7,8,9 → 10)
    for col in [7, 8, 9]:  # H, I, J
        mask = df[col].notna()
        df.loc[mask, 10] = df.loc[mask, col]  # K
    
    # M~O 열 데이터를 P로 이동 (0-indexed: 12,13,14 → 15)
    for col in [12, 13, 14]:  # M, N, O
        mask = df[col].notna()
        df.loc[mask, 15] = df.loc[mask, col]  # P
    
    return df

def extract_data_from_file(file_path, entity_currency_map):
    """단일 엑셀 파일에서 BS, IS 데이터 추출"""
    all_data = []
    
    try:
        # BS 시트 처리
        bs_df = pd.read_excel(file_path, sheet_name='BS', header=None)
        entity = bs_df.iloc[0, 6]  # G1 (0-indexed: row 0, col 6)
        
        # Entity별 통화 정보 가져오기 (ENTITY_CURRENCY_MAP에서)
        currency = entity_currency_map.get(entity, {}).get('currency', 'Unknown')
        
        # 전처리
        bs_df = preprocess_sheet(bs_df, 'BS')
        
        # 6행(index 5)을 컬럼명으로, 7~231행(index 6~230) 데이터 추출
        bs_data = bs_df.iloc[6:231, [6, 10, 15, 22]]  # G, K, P, W (0-indexed: 6, 10, 15, 22)
        bs_data.columns = bs_df.iloc[5, [6, 10, 15, 22]].values  # 6행을 컬럼명으로
        
        bs_data['Entity'] = entity
        bs_data['Statement'] = 'BS'
        bs_data['Currency'] = currency
        bs_data['Period'] = PERIOD
        
        # 공란 제거 (모든 값이 NaN인 행 제거)
        bs_data = bs_data.dropna(how='all', subset=bs_data.columns[:-4])  # Entity, Statement, Currency, Period 제외
        
        # K열(두 번째 열)에 값이 없는 행 삭제
        k_col_name = bs_data.columns[1]  # K열 컬럼명
        bs_data = bs_data[bs_data[k_col_name].notna()]
        
        # W열만 숫자형으로 변환 (공란을 0으로, G/K/P는 텍스트 유지)
        w_col_name = bs_data.columns[3]  # W열 컬럼명
        bs_data[w_col_name] = pd.to_numeric(bs_data[w_col_name], errors='coerce').fillna(0)
        
        all_data.append(bs_data)
        
        # IS 시트 처리
        is_df = pd.read_excel(file_path, sheet_name='IS', header=None)
        entity = is_df.iloc[0, 6]  # G1
        
        # Entity별 통화 정보 가져오기 (ENTITY_CURRENCY_MAP에서)
        currency = entity_currency_map.get(entity, {}).get('currency', 'Unknown')
        
        # 전처리
        is_df = preprocess_sheet(is_df, 'IS')
        
        # 6행을 컬럼명으로, 7~129행(index 6~128) 데이터 추출
        is_data = is_df.iloc[6:129, [6, 10, 15, 22]]  # G, K, P, W
        is_data.columns = is_df.iloc[5, [6, 10, 15, 22]].values
        
        is_data['Entity'] = entity
        is_data['Statement'] = 'IS'
        is_data['Currency'] = currency
        is_data['Period'] = PERIOD
        
        # 공란 제거
        is_data = is_data.dropna(how='all', subset=is_data.columns[:-4])  # Entity, Statement, Currency, Period 제외
        
        # K열(두 번째 열)에 값이 없는 행 삭제
        k_col_name = is_data.columns[1]  # K열 컬럼명
        is_data = is_data[is_data[k_col_name].notna()]
        
        # W열만 숫자형으로 변환 (공란을 0으로, G/K/P는 텍스트 유지)
        w_col_name = is_data.columns[3]  # W열 컬럼명
        is_data[w_col_name] = pd.to_numeric(is_data[w_col_name], errors='coerce').fillna(0)
        
        all_data.append(is_data)
        
        print(f"✓ {os.path.basename(file_path)} - {entity} ({currency}) 처리 완료")
        
    except Exception as e:
        print(f"✗ {os.path.basename(file_path)} 처리 실패: {str(e)}")
    
    return all_data

def consolidate_financial_data(input_folder, output_file, entity_currency_map):
    """여러 엑셀 파일을 통합"""
    all_results = []
    
    # 폴더 내 모든 엑셀 파일 처리
    excel_files = list(Path(input_folder).glob('*.xlsx'))
    excel_files = [f for f in excel_files if not f.name.startswith('~')]  # 임시파일 제외
    
    print(f"총 {len(excel_files)}개 파일 발견\n")
    
    for file_path in sorted(excel_files):
        data_list = extract_data_from_file(file_path, entity_currency_map)
        all_results.extend(data_list)
    
    # 모든 데이터 통합
    if all_results:
        consolidated_df = pd.concat(all_results, ignore_index=True)
        
        # 컬럼 순서 조정 (Entity, Statement, Currency, Period를 앞으로)
        cols = ['Entity', 'Statement', 'Currency', 'Period'] + [col for col in consolidated_df.columns if col not in ['Entity', 'Statement', 'Currency', 'Period']]
        consolidated_df = consolidated_df[cols]
        
        # 결과 저장
        consolidated_df.to_excel(output_file, index=False)
        
        print(f"\n{'='*60}")
        print(f"[Step 1] 통합 완료!")
        print(f"총 {len(consolidated_df)} 행의 데이터")
        print(f"저장 위치: {output_file}")
        print(f"Entity 개수: {consolidated_df['Entity'].nunique()}")
        print(f"{'='*60}")
        
        # 요약 정보
        print("\n[Entity별 데이터 건수]")
        print(consolidated_df.groupby(['Entity', 'Statement', 'Currency', 'Period']).size())
        
        # Unknown Currency 확인
        unknown_currency = consolidated_df[consolidated_df['Currency'] == 'Unknown']['Entity'].unique()
        if len(unknown_currency) > 0:
            print(f"\n⚠️  통화 정보가 없는 Entity: {list(unknown_currency)}")
            print("   → ENTITY_CURRENCY_MAP에 추가해주세요!")
        
        return consolidated_df
    else:
        print("처리된 데이터가 없습니다.")
        return None

def process_final_file(input_file, output_file, entity_currency_map):
    """
    consolidated_financial_data.xlsx를 읽어서 
    Amount, Exchange_Rate, Amount(KRW) 추가 및 PL 시트 생성
    """
    print("\n" + "="*60)
    print("[Step 2] 통화 및 환율 적용 중...")
    print("="*60)
    
    # 데이터 읽기
    df = pd.read_excel(input_file)
    
    print(f"✓ 데이터 로드 완료: {len(df)} 행")
    print(f"✓ 전체 컬럼: {list(df.columns)}")
    
    # 1. Amount 열 추가 - 각 행의 Currency에 해당하는 컬럼 값
    # 통화 컬럼 매핑 (Currency와 실제 컬럼명이 다를 수 있음)
    통화_컬럼_매핑 = {
        'AUD': 'AUD',
        'KRW': 'KRW',
        'JPY': 'JPY',
        'USD': 'USD',
        'CNY': 'CNH',  # Currency는 CNY지만 컬럼명은 CNH
        'CNH': 'CNH',  # CNH도 허용
        'MXN': 'MXN',
        'EUR': 'EUR',
        'VND': 'VND',
        'MYR': 'MYR',
        'INR': 'INR',
        'TRY': 'TRY',
    }
    
    def get_amount(row):
        currency = row['Currency']
        col_name = 통화_컬럼_매핑.get(currency)
        
        # 매핑된 컬럼이 있고 해당 컬럼이 존재하면
        if col_name and col_name in df.columns:
            value = row[col_name]
            return 0 if pd.isna(value) else float(value)
        else:
            # 통화 컬럼이 없으면 모든 통화 컬럼 확인해서 값이 있는 것 사용
            통화_컬럼들 = ['AUD', 'KRW', 'JPY', 'USD', 'CNH', 'MXN', 'EUR', 'VND', 'MYR', 'INR']
            for col in 통화_컬럼들:
                if col in df.columns and pd.notna(row[col]) and row[col] != 0:
                    return float(row[col])
            return 0
    
    df['Amount'] = df.apply(get_amount, axis=1)
    
    print(f"✓ Amount 생성 완료")
    print(f"  - Amount 합계: {df['Amount'].sum():,.2f}")
    print(f"  - Amount > 0인 행: {(df['Amount'] > 0).sum()} / {len(df)}")
    
    # 2. Entity와 Statement별 Exchange_Rate 매핑
    def get_exchange_rate(row):
        entity = row['Entity']
        statement = row['Statement']
        entity_info = entity_currency_map.get(entity, {})
        
        # BS는 bs_rate, IS(PL)는 pl_rate 사용
        if statement == 'BS':
            return entity_info.get('bs_rate', 1.0)
        else:  # IS
            return entity_info.get('pl_rate', 1.0)
    
    df['Exchange_Rate'] = df.apply(get_exchange_rate, axis=1)
    
    # 3. Amount(KRW) 계산 (소숫점 반올림)
    df['Amount(KRW)'] = (df['Amount'] * df['Exchange_Rate']).round(0).astype('Int64')
    
    print(f"  - Amount(KRW) 합계: {df['Amount(KRW)'].sum():,.0f}")
    
    # 컬럼 순서 조정
    base_cols = ['Entity', 'Statement', 'Currency', 'Period']
    data_cols = [col for col in df.columns if col not in base_cols + ['Amount', 'Exchange_Rate', 'Amount(KRW)']]
    final_cols = base_cols + data_cols + ['Amount', 'Exchange_Rate', 'Amount(KRW)']
    
    df = df[final_cols]
    
    # Sheet1 저장
    with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
        df.to_excel(writer, sheet_name='Sheet1', index=False)
    
    print(f"\n✓ Sheet1 저장 완료: {output_file}")
    
    # 통화별 요약
    print("\n[Entity별 통화 및 총액]")
    summary = df.groupby(['Entity', 'Statement', 'Currency']).agg({
        'Amount': 'sum',
        'Amount(KRW)': 'sum',
        'Exchange_Rate': 'first'  # 환율 표시
    }).round(0)
    print(summary)
    
    print(f"\n{'='*60}")
    print(f"[Step 2] 완료!")
    print(f"총 {len(df)} 행의 데이터")
    print(f"저장 위치: {output_file}")
    print(f"{'='*60}")
    
    # BS와 PL 환율 확인
    print("\n[BS 환율 매핑 상태]")
    bs_data = df[df['Statement'] == 'BS'][['Entity', 'Currency', 'Exchange_Rate']].drop_duplicates()
    for _, row in bs_data.iterrows():
        print(f"✓ {row['Entity']} ({row['Currency']}) - {row['Exchange_Rate']} (기말환율)")
    
    print("\n[PL 환율 매핑 상태]")
    pl_data = df[df['Statement'] == 'IS'][['Entity', 'Currency', 'Exchange_Rate']].drop_duplicates()
    for _, row in pl_data.iterrows():
        print(f"✓ {row['Entity']} ({row['Currency']}) - {row['Exchange_Rate']} (평균환율)")
    
    return df

def create_pl_sheet(final_file):
    """
    final_financial_data.xlsx에 PL 시트 추가
    """
    print("\n" + "="*60)
    print("[Step 3] PL 시트 생성 중...")
    print("="*60)
    
    # final_financial_data.xlsx의 Sheet1 읽기
    df = pd.read_excel(final_file, sheet_name='Sheet1')
    
    # 1. Statement = "IS"만 필터링
    df_pl = df[df['Statement'] == 'IS'].copy()
    print(f"✓ IS 데이터 필터링: {len(df_pl)} 행")
    
    # 2. 특정 컬럼만 선택
    selected_cols = ['Entity', 'Statement', 'Currency', 'Period', 'Code', '계정과목', 'Account', 
                     'Amount', 'Exchange_Rate', 'Amount(KRW)']
    df_pl = df_pl[selected_cols]
    
    # 3. 특정 code 행 삭제
    codes_to_remove = [40000, 50000, 59000, 60000, 69000, 71000, 72000, 73000, 74000, 
                       75000, 79000, 80000, 81000, 82000, 83000, 84000]
    df_pl = df_pl[~df_pl['Code'].isin(codes_to_remove)]
    print(f"✓ 특정 Code 삭제 후: {len(df_pl)} 행")
    
    # 4. Rev_Account 컬럼 생성
    def get_rev_account(code):
        code_str = str(code)
        
        if code_str.startswith('4'):
            return '매출'
        elif code_str.startswith('5'):
            return '매출원가'
        elif code in [60001, 60002, 60003, 60004, 60005, 60006, 60034]:
            return '인건비'
        elif code == 60007:
            return '복리후생비'
        elif code == 60008:
            return '여비교통비'
        elif code == 60015:
            return '광고선전비'
        elif code == 60017:
            return '운반비'
        elif code == 60018:
            return '지급수수료'
        elif code == 60019:
            return '경상연구개발비'
        elif code == 60027:
            return '판매수수료'
        elif code in [60021, 60029]:
            return '감가상각비'
        elif code == 60033:
            return '사용권자산상각비'
        elif code == 60022:
            return '대손상각비'
        elif code_str.startswith('60'):
            return '기타'
        elif code_str.startswith('71'):
            return '영업외수익'
        elif code_str.startswith('72'):
            return '영업외비용'
        elif code_str.startswith('73'):
            return '금융수익'
        elif code_str.startswith('74'):
            return '금융비용'
        elif code_str.startswith('75'):
            return '지분법손익'
        elif code_str.startswith('76'):
            return '순화폐성손익익'
        elif code_str.startswith('80'):
            return '법인세비용'
        elif code_str.startswith('81'):
            return '세후중단영업손익'
        elif code_str.startswith('82'):
            return '당기순이익'
        elif code_str.startswith('83'):
            return '포괄손익'
        elif code_str.startswith('84'):
            return '총포괄손익'
        else:
            return '기타'
    
    df_pl['Rev_Account'] = df_pl['Code'].apply(get_rev_account)
    
    # 컬럼 순서 조정 (Rev_Account를 Account 다음에)
    final_cols = ['Entity', 'Statement', 'Currency', 'Period', 'Code', '계정과목', 'Account', 
                  'Rev_Account', 'Amount', 'Exchange_Rate', 'Amount(KRW)']
    df_pl = df_pl[final_cols]
    
    # PL 시트 추가
    with pd.ExcelWriter(final_file, engine='openpyxl', mode='a') as writer:
        df_pl.to_excel(writer, sheet_name='PL', index=False)
    
    print(f"✓ PL 시트 저장 완료: {final_file}")
    
    # Rev_Account별 요약
    print("\n[Rev_Account별 집계]")
    summary = df_pl.groupby('Rev_Account').agg({
        'Amount(KRW)': 'sum',
        'Code': 'count'
    }).round(0)
    summary.columns = ['Amount(KRW)', '행수']
    summary = summary.sort_values('Amount(KRW)', ascending=False)
    print(summary)
    
    print(f"\n{'='*60}")
    print(f"[Step 3] 완료!")
    print(f"{'='*60}")
    
    return df_pl

# 실행
if __name__ == "__main__":
    # 경로 설정
    input_folder = r"C:\Users\user\InBody Co., Ltd\회계팀 - General\★★Accounting Folder★★vF\1. 재무결산\FY2025\2025_12_4Q\2. 연결재무제표\3. Final PKG"
    consolidated_file = r"C:\Users\user\InBody Co., Ltd\회계팀 - General\★★Accounting Folder★★vF\1. 재무결산\FY2025\2025_12_4Q\2. 연결재무제표\Income statement\consolidated_financial_data_25Q4.xlsx"
    final_file = r"C:\Users\user\InBody Co., Ltd\회계팀 - General\★★Accounting Folder★★vF\1. 재무결산\FY2025\2025_12_4Q\2. 연결재무제표\Income statement\final_financial_data_25Q4.xlsx"
    
    print("="*60)
    print("해외법인 재무제표 데이터 통합 시작")
    print("="*60)
    
    # Step 1: 데이터 통합
    result_df = consolidate_financial_data(input_folder, consolidated_file, ENTITY_CURRENCY_MAP)
    
    if result_df is not None:
        # Step 2: 환율 적용
        final_df = process_final_file(consolidated_file, final_file, ENTITY_CURRENCY_MAP)
        
        if final_df is not None:
            # Step 3: PL 시트 생성
            pl_df = create_pl_sheet(final_file)
            
            print("\n" + "="*60)
            print("✅ 모든 작업이 완료되었습니다!")
            print("="*60)
            print(f"\n생성된 파일:")
            print(f"  1. {consolidated_file}")
            print(f"  2. {final_file} (Sheet1 + PL)")
        else:
            print("\n❌ Step 2 실패")
    else:
        print("\n❌ Step 1 실패")


해외법인 재무제표 데이터 통합 시작
총 17개 파일 발견



C:\Users\user\AppData\Local\Temp\ipykernel_8588\4269645831.py:104: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['<Hide>' '계정과목' '매출액' '매출원가' '매출총이익' '판매비와관리비' '영업이익' '기타수익' '기타비용'
 '금융수익' '금융비용' '지분법손익' '법인세비용차감전이익' '법인세비용' '세후중단영업손익' '당기순이익' '포괄손익'
 '총포괄손익']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[mask, 10] = df.loc[mask, col]  # K
C:\Users\user\AppData\Local\Temp\ipykernel_8588\4269645831.py:109: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['Account' 'Sales' 'Cost of Goods Sold ' 'Gross Profit'
 'Selling and Administration Expense' 'Operating Income' 'Other Revenue'
 'Other Expense' 'Financial Revenue' 'Financial Expense'
 'Gain and Loss on Equity Method' 'Net Income before Corporate Income Tax'
 'Corporate Income Tax' 'Gain and Loss from discontinued operations '
 'Net Income ' '

✓ 1. Inbody K-IFRS Package_HQ_2025_4Q_Draft(260107).xlsx - 주식회사 인바디 (KRW) 처리 완료


C:\Users\user\AppData\Local\Temp\ipykernel_8588\4269645831.py:104: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['<Hide>' '계정과목' '매출액' '매출원가' '매출총이익' '판매비와관리비' '영업이익' '기타수익' '기타비용'
 '금융수익' '금융비용' '지분법손익' '법인세비용차감전이익' '법인세비용' '세후중단영업손익' '당기순이익' '포괄손익'
 '총포괄손익']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[mask, 10] = df.loc[mask, col]  # K
C:\Users\user\AppData\Local\Temp\ipykernel_8588\4269645831.py:109: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['Account' 'Sales' 'Cost of Goods Sold ' 'Gross Profit'
 'Selling and Administration Expense' 'Operating Income' 'Other Revenue'
 'Other Expense' 'Financial Revenue' 'Financial Expense'
 'Gain and Loss on Equity Method' 'Net Income before Corporate Income Tax'
 'Corporate Income Tax' 'Gain and Loss from discontinued operations '
 'Net Income ' '

✓ 10. Inbody K-IFRS Package_KOROT_2025_4Q_Final(250129).xlsx - ㈜코르트 (KRW) 처리 완료


...

✓ 7. Inbody K-IFRS Package_Europe_2025_4Q_Draft(260204).xlsx - Inbody Europe B.V. (EUR) 처리 완료


C:\Users\user\AppData\Local\Temp\ipykernel_8588\4269645831.py:104: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['<Hide>' '계정과목' '매출액' '매출원가' '매출총이익' '판매비와관리비' '영업이익' '기타수익' '기타비용'
 '금융수익' '금융비용' '지분법손익' '법인세비용차감전이익' '법인세비용' '세후중단영업손익' '당기순이익' '포괄손익'
 '총포괄손익']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[mask, 10] = df.loc[mask, col]  # K
C:\Users\user\AppData\Local\Temp\ipykernel_8588\4269645831.py:109: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['Account' 'Sales' 'Cost of Goods Sold ' 'Gross Profit'
 'Selling and Administration Expense' 'Operating Income' 'Other Revenue'
 'Other Expense' 'Financial Revenue' 'Financial Expense'
 'Gain and Loss on Equity Method' 'Net Income before Corporate Income Tax'
 'Corporate Income Tax' 'Gain and Loss from discontinued operations '
 'Net Income ' '

✓ 8. Inbody K-IFRS Package_삼한정공_2025_4Q_Draft(260128).xlsx - (주)삼한정공 (KRW) 처리 완료


C:\Users\user\AppData\Local\Temp\ipykernel_8588\4269645831.py:104: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['<Hide>' '계정과목' '매출액' '매출원가' '매출총이익' '판매비와관리비' '영업이익' '기타수익' '기타비용'
 '금융수익' '금융비용' '지분법손익' '법인세비용차감전이익' '법인세비용' '세후중단영업손익' '당기순이익' '포괄손익'
 '총포괄손익']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[mask, 10] = df.loc[mask, col]  # K
C:\Users\user\AppData\Local\Temp\ipykernel_8588\4269645831.py:109: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['Account' 'Sales' 'Cost of Goods Sold ' 'Gross Profit'
 'Selling and Administration Expense' 'Operating Income' 'Other Revenue'
 'Other Expense' 'Financial Revenue' 'Financial Expense'
 'Gain and Loss on Equity Method' 'Net Income before Corporate Income Tax'
 'Corporate Income Tax' 'Gain and Loss from discontinued operations '
 'Net Income ' '

✓ 9. Inbody K-IFRS Package_KOCP_2025_4Q_Draft(260129).xlsx - 케이오씨피 프로젝트 제2호 벤처투자조합 (KRW) 처리 완료

[Step 1] 통합 완료!
총 5269 행의 데이터
저장 위치: C:\Users\user\InBody Co., Ltd\회계팀 - General\★★Accounting Folder★★vF\1. 재무결산\FY2025\2025_12_4Q\2. 연결재무제표\Income statement\consolidated_financial_data_25Q4.xlsx
Entity 개수: 17

[Entity별 데이터 건수]
Entity                                         Statement  Currency  Period
(주)삼한정공                                        BS         KRW       20254Q    195
                                               IS         KRW       20254Q    115
ADJ                                            BS         KRW       20254Q    195
                                               IS         KRW       20254Q    115
BIOSPACE LATIN AMERICA S DE RL DE CV(*)        BS         MXN       20254Q    195
                                               IS         MXN       20254Q    115
BWA                                            BS         USD       20254Q    195
                           

In [4]:
import pandas as pd
import os
from pathlib import Path

# ==================== 여기를 수정하세요! ====================
# 기간 설정
PERIOD = "20253Q"  

# Entity별 통화 및 환율 매핑 (BS: 기말환율, PL: 평균환율)
ENTITY_CURRENCY_MAP = {
    'Biospace Inc. DBA INBODY': {
        'currency': 'USD', 
        'bs_rate': 1402.20,
        'pl_rate': 1412.79
    },
    'InBody Japan Inc.': {
        'currency': 'JPY', 
        'bs_rate': 9.4373,
        'pl_rate': 9.5426
    },
    'Biospace Co.,Ltd.': {
        'currency': 'CNH', 
        'bs_rate': 196.82,
        'pl_rate': 195.52
    },
    'InBody Oceania ': {
        'currency': 'AUD', 
        'bs_rate': 922.37,
        'pl_rate': 905.42
    },
    'BIOSPACE LATIN AMERICA S DE RL DE CV(*)': {
        'currency': 'MXN', 
        'bs_rate': 76.34,
        'pl_rate': 72.53
    },
    'BWA': {
        'currency': 'USD', 
        'bs_rate': 1402.20,
        'pl_rate': 1412.79
    },
    '㈜인바디헬스케어': {
        'currency': 'KRW', 
        'bs_rate': 1.0,
        'pl_rate': 1.0
    },
    'InBody Vietnam': {
        'currency': 'VND', 
        'bs_rate': 0.0531,
        'pl_rate': 0.0546
    },
    'INBODY ASIA SDN. BHD.': {
        'currency': 'MYR', 
        'bs_rate': 332.67,
        'pl_rate': 326.49
    },
    'InBody India Pvt. Ltd': {
        'currency': 'INR', 
        'bs_rate': 15.81,
        'pl_rate': 16.33
    },
    'Inbody Europe B.V.': {
        'currency': 'EUR', 
        'bs_rate': 1644.50,
        'pl_rate': 1580.96
    },
    '㈜코르트': {
        'currency': 'KRW', 
        'bs_rate': 1.0,
        'pl_rate': 1.0
    },
    'INBODY TURKEY MEDİKAL TİCARET LİMİTED ŞİRKETİ': {
        'currency': 'TRY', 
        'bs_rate': 33.40,
        'pl_rate': 36.05
    },
    '(주)삼한정공': {
        'currency': 'KRW', 
        'bs_rate': 1.0,
        'pl_rate': 1.0
    },
    '주식회사 인바디': {
        'currency': 'KRW', 
        'bs_rate': 1.0,
        'pl_rate': 1.0
    },
    '케이오씨피 프로젝트 제2호 벤처투자조합': {
        'currency': 'KRW', 
        'bs_rate': 1.0,
        'pl_rate': 1.0
    },
    'ADJ': {
        'currency': 'KRW', 
        'bs_rate': 1.0,
        'pl_rate': 1.0
    },
}
# ============================================================

def preprocess_sheet(df, sheet_type):
    """데이터 전처리: H~J → K, M~O → P로 이동"""
    # H~J 열 데이터를 K로 이동 (0-indexed: 7,8,9 → 10)
    for col in [7, 8, 9]:  # H, I, J
        mask = df[col].notna()
        df.loc[mask, 10] = df.loc[mask, col]  # K
    
    # M~O 열 데이터를 P로 이동 (0-indexed: 12,13,14 → 15)
    for col in [12, 13, 14]:  # M, N, O
        mask = df[col].notna()
        df.loc[mask, 15] = df.loc[mask, col]  # P
    
    return df

def extract_data_from_file(file_path, entity_currency_map):
    """단일 엑셀 파일에서 BS, IS 데이터 추출"""
    all_data = []
    
    try:
        # BS 시트 처리
        bs_df = pd.read_excel(file_path, sheet_name='BS', header=None)
        entity = bs_df.iloc[0, 6]  # G1 (0-indexed: row 0, col 6)
        
        # Entity별 통화 정보 가져오기 (ENTITY_CURRENCY_MAP에서)
        currency = entity_currency_map.get(entity, {}).get('currency', 'Unknown')
        
        # 전처리
        bs_df = preprocess_sheet(bs_df, 'BS')
        
        # 6행(index 5)을 컬럼명으로, 7~231행(index 6~230) 데이터 추출
        bs_data = bs_df.iloc[6:231, [6, 10, 15, 22]]  # G, K, P, W (0-indexed: 6, 10, 15, 22)
        bs_data.columns = bs_df.iloc[5, [6, 10, 15, 22]].values  # 6행을 컬럼명으로
        
        bs_data['Entity'] = entity
        bs_data['Statement'] = 'BS'
        bs_data['Currency'] = currency
        bs_data['Period'] = PERIOD
        
        # 공란 제거 (모든 값이 NaN인 행 제거)
        bs_data = bs_data.dropna(how='all', subset=bs_data.columns[:-4])  # Entity, Statement, Currency, Period 제외
        
        # K열(두 번째 열)에 값이 없는 행 삭제
        k_col_name = bs_data.columns[1]  # K열 컬럼명
        bs_data = bs_data[bs_data[k_col_name].notna()]
        
        # W열만 숫자형으로 변환 (공란을 0으로, G/K/P는 텍스트 유지)
        w_col_name = bs_data.columns[3]  # W열 컬럼명
        bs_data[w_col_name] = pd.to_numeric(bs_data[w_col_name], errors='coerce').fillna(0)
        
        all_data.append(bs_data)
        
        # IS 시트 처리
        is_df = pd.read_excel(file_path, sheet_name='IS', header=None)
        entity = is_df.iloc[0, 6]  # G1
        
        # Entity별 통화 정보 가져오기 (ENTITY_CURRENCY_MAP에서)
        currency = entity_currency_map.get(entity, {}).get('currency', 'Unknown')
        
        # 전처리
        is_df = preprocess_sheet(is_df, 'IS')
        
        # 6행을 컬럼명으로, 7~129행(index 6~128) 데이터 추출
        is_data = is_df.iloc[6:129, [6, 10, 15, 22]]  # G, K, P, W
        is_data.columns = is_df.iloc[5, [6, 10, 15, 22]].values
        
        is_data['Entity'] = entity
        is_data['Statement'] = 'IS'
        is_data['Currency'] = currency
        is_data['Period'] = PERIOD
        
        # 공란 제거
        is_data = is_data.dropna(how='all', subset=is_data.columns[:-4])  # Entity, Statement, Currency, Period 제외
        
        # K열(두 번째 열)에 값이 없는 행 삭제
        k_col_name = is_data.columns[1]  # K열 컬럼명
        is_data = is_data[is_data[k_col_name].notna()]
        
        # W열만 숫자형으로 변환 (공란을 0으로, G/K/P는 텍스트 유지)
        w_col_name = is_data.columns[3]  # W열 컬럼명
        is_data[w_col_name] = pd.to_numeric(is_data[w_col_name], errors='coerce').fillna(0)
        
        all_data.append(is_data)
        
        print(f"✓ {os.path.basename(file_path)} - {entity} ({currency}) 처리 완료")
        
    except Exception as e:
        print(f"✗ {os.path.basename(file_path)} 처리 실패: {str(e)}")
    
    return all_data

def consolidate_financial_data(input_folder, output_file, entity_currency_map):
    """여러 엑셀 파일을 통합"""
    all_results = []
    
    # 폴더 내 모든 엑셀 파일 처리
    excel_files = list(Path(input_folder).glob('*.xlsx'))
    excel_files = [f for f in excel_files if not f.name.startswith('~')]  # 임시파일 제외
    
    print(f"총 {len(excel_files)}개 파일 발견\n")
    
    for file_path in sorted(excel_files):
        data_list = extract_data_from_file(file_path, entity_currency_map)
        all_results.extend(data_list)
    
    # 모든 데이터 통합
    if all_results:
        consolidated_df = pd.concat(all_results, ignore_index=True)
        
        # 컬럼 순서 조정 (Entity, Statement, Currency, Period를 앞으로)
        cols = ['Entity', 'Statement', 'Currency', 'Period'] + [col for col in consolidated_df.columns if col not in ['Entity', 'Statement', 'Currency', 'Period']]
        consolidated_df = consolidated_df[cols]
        
        # 결과 저장
        consolidated_df.to_excel(output_file, index=False)
        
        print(f"\n{'='*60}")
        print(f"[Step 1] 통합 완료!")
        print(f"총 {len(consolidated_df)} 행의 데이터")
        print(f"저장 위치: {output_file}")
        print(f"Entity 개수: {consolidated_df['Entity'].nunique()}")
        print(f"{'='*60}")
        
        # 요약 정보
        print("\n[Entity별 데이터 건수]")
        print(consolidated_df.groupby(['Entity', 'Statement', 'Currency', 'Period']).size())
        
        # Unknown Currency 확인
        unknown_currency = consolidated_df[consolidated_df['Currency'] == 'Unknown']['Entity'].unique()
        if len(unknown_currency) > 0:
            print(f"\n⚠️  통화 정보가 없는 Entity: {list(unknown_currency)}")
            print("   → ENTITY_CURRENCY_MAP에 추가해주세요!")
        
        return consolidated_df
    else:
        print("처리된 데이터가 없습니다.")
        return None

def process_final_file(input_file, output_file, entity_currency_map):
    """
    consolidated_financial_data.xlsx를 읽어서 
    Amount, Exchange_Rate, Amount(KRW) 추가 및 PL 시트 생성
    """
    print("\n" + "="*60)
    print("[Step 2] 통화 및 환율 적용 중...")
    print("="*60)
    
    # 데이터 읽기
    df = pd.read_excel(input_file)
    
    print(f"✓ 데이터 로드 완료: {len(df)} 행")
    print(f"✓ 전체 컬럼: {list(df.columns)}")
    
    # 1. Amount 열 추가 - 각 행의 Currency에 해당하는 컬럼 값
    # 통화 컬럼 매핑 (Currency와 실제 컬럼명이 다를 수 있음)
    통화_컬럼_매핑 = {
        'AUD': 'AUD',
        'KRW': 'KRW',
        'JPY': 'JPY',
        'USD': 'USD',
        'CNY': 'CNH',  # Currency는 CNY지만 컬럼명은 CNH
        'CNH': 'CNH',  # CNH도 허용
        'MXN': 'MXN',
        'EUR': 'EUR',
        'VND': 'VND',
        'MYR': 'MYR',
        'INR': 'INR',
        'TRY': 'TRY',
    }
    
    def get_amount(row):
        currency = row['Currency']
        col_name = 통화_컬럼_매핑.get(currency)
        
        # 매핑된 컬럼이 있고 해당 컬럼이 존재하면
        if col_name and col_name in df.columns:
            value = row[col_name]
            return 0 if pd.isna(value) else float(value)
        else:
            # 통화 컬럼이 없으면 모든 통화 컬럼 확인해서 값이 있는 것 사용
            통화_컬럼들 = ['AUD', 'KRW', 'JPY', 'USD', 'CNH', 'MXN', 'EUR', 'VND', 'MYR', 'INR', 'TRY']
            for col in 통화_컬럼들:
                if col in df.columns and pd.notna(row[col]) and row[col] != 0:
                    return float(row[col])
            return 0
    
    df['Amount'] = df.apply(get_amount, axis=1)
    
    print(f"✓ Amount 생성 완료")
    print(f"  - Amount 합계: {df['Amount'].sum():,.2f}")
    print(f"  - Amount > 0인 행: {(df['Amount'] > 0).sum()} / {len(df)}")
    
    # 2. Entity와 Statement별 Exchange_Rate 매핑
    def get_exchange_rate(row):
        entity = row['Entity']
        statement = row['Statement']
        entity_info = entity_currency_map.get(entity, {})
        
        # BS는 bs_rate, IS(PL)는 pl_rate 사용
        if statement == 'BS':
            return entity_info.get('bs_rate', 1.0)
        else:  # IS
            return entity_info.get('pl_rate', 1.0)
    
    df['Exchange_Rate'] = df.apply(get_exchange_rate, axis=1)
    
    # 3. Amount(KRW) 계산 (소숫점 반올림)
    df['Amount(KRW)'] = (df['Amount'] * df['Exchange_Rate']).round(0).astype('Int64')
    
    print(f"  - Amount(KRW) 합계: {df['Amount(KRW)'].sum():,.0f}")
    
    # 컬럼 순서 조정
    base_cols = ['Entity', 'Statement', 'Currency', 'Period']
    data_cols = [col for col in df.columns if col not in base_cols + ['Amount', 'Exchange_Rate', 'Amount(KRW)']]
    final_cols = base_cols + data_cols + ['Amount', 'Exchange_Rate', 'Amount(KRW)']
    
    df = df[final_cols]
    
    # Sheet1 저장
    with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
        df.to_excel(writer, sheet_name='Sheet1', index=False)
    
    print(f"\n✓ Sheet1 저장 완료: {output_file}")
    
    # 통화별 요약
    print("\n[Entity별 통화 및 총액]")
    summary = df.groupby(['Entity', 'Statement', 'Currency']).agg({
        'Amount': 'sum',
        'Amount(KRW)': 'sum',
        'Exchange_Rate': 'first'  # 환율 표시
    }).round(0)
    print(summary)
    
    print(f"\n{'='*60}")
    print(f"[Step 2] 완료!")
    print(f"총 {len(df)} 행의 데이터")
    print(f"저장 위치: {output_file}")
    print(f"{'='*60}")
    
    # BS와 PL 환율 확인
    print("\n[BS 환율 매핑 상태]")
    bs_data = df[df['Statement'] == 'BS'][['Entity', 'Currency', 'Exchange_Rate']].drop_duplicates()
    for _, row in bs_data.iterrows():
        print(f"✓ {row['Entity']} ({row['Currency']}) - {row['Exchange_Rate']} (기말환율)")
    
    print("\n[PL 환율 매핑 상태]")
    pl_data = df[df['Statement'] == 'IS'][['Entity', 'Currency', 'Exchange_Rate']].drop_duplicates()
    for _, row in pl_data.iterrows():
        print(f"✓ {row['Entity']} ({row['Currency']}) - {row['Exchange_Rate']} (평균환율)")
    
    return df

def create_pl_sheet(final_file):
    """
    final_financial_data.xlsx에 PL 시트 추가
    """
    print("\n" + "="*60)
    print("[Step 3] PL 시트 생성 중...")
    print("="*60)
    
    # final_financial_data.xlsx의 Sheet1 읽기
    df = pd.read_excel(final_file, sheet_name='Sheet1')
    
    # 1. Statement = "IS"만 필터링
    df_pl = df[df['Statement'] == 'IS'].copy()
    print(f"✓ IS 데이터 필터링: {len(df_pl)} 행")
    
    # 2. 특정 컬럼만 선택
    selected_cols = ['Entity', 'Statement', 'Currency', 'Period', 'Code', '계정과목', 'Account', 
                     'Amount', 'Exchange_Rate', 'Amount(KRW)']
    df_pl = df_pl[selected_cols]
    
    # 3. 특정 code 행 삭제
    codes_to_remove = [40000, 50000, 59000, 60000, 69000, 71000, 72000, 73000, 74000, 
                       75000, 79000, 80000, 81000, 82000, 83000, 84000]
    df_pl = df_pl[~df_pl['Code'].isin(codes_to_remove)]
    print(f"✓ 특정 Code 삭제 후: {len(df_pl)} 행")
    
    # 4. Rev_Account 컬럼 생성
    def get_rev_account(code):
        code_str = str(code)
        
        if code_str.startswith('4'):
            return '매출'
        elif code_str.startswith('5'):
            return '매출원가'
        elif code in [60001, 60002, 60003, 60004, 60005, 60006, 60034]:
            return '인건비'
        elif code == 60007:
            return '복리후생비'
        elif code == 60008:
            return '여비교통비'
        elif code == 60015:
            return '광고선전비'
        elif code == 60017:
            return '운반비'
        elif code == 60018:
            return '지급수수료'
        elif code == 60019:
            return '경상연구개발비'
        elif code == 60027:
            return '판매수수료'
        elif code in [60021, 60029]:
            return '감가상각비'
        elif code == 60033:
            return '사용권자산상각비'
        elif code == 60022:
            return '대손상각비'
        elif code_str.startswith('60'):
            return '기타'
        elif code_str.startswith('71'):
            return '영업외수익'
        elif code_str.startswith('72'):
            return '영업외비용'
        elif code_str.startswith('73'):
            return '금융수익'
        elif code_str.startswith('74'):
            return '금융비용'
        elif code_str.startswith('75'):
            return '지분법손익'
        elif code_str.startswith('80'):
            return '법인세비용'
        elif code_str.startswith('81'):
            return '세후중단영업손익'
        elif code_str.startswith('82'):
            return '당기순이익'
        elif code_str.startswith('83'):
            return '포괄손익'
        elif code_str.startswith('84'):
            return '총포괄손익'
        else:
            return '기타'
    
    df_pl['Rev_Account'] = df_pl['Code'].apply(get_rev_account)
    
    # 컬럼 순서 조정 (Rev_Account를 Account 다음에)
    final_cols = ['Entity', 'Statement', 'Currency', 'Period', 'Code', '계정과목', 'Account', 
                  'Rev_Account', 'Amount', 'Exchange_Rate', 'Amount(KRW)']
    df_pl = df_pl[final_cols]
    
    # PL 시트 추가
    with pd.ExcelWriter(final_file, engine='openpyxl', mode='a') as writer:
        df_pl.to_excel(writer, sheet_name='PL', index=False)
    
    print(f"✓ PL 시트 저장 완료: {final_file}")
    
    # Rev_Account별 요약
    print("\n[Rev_Account별 집계]")
    summary = df_pl.groupby('Rev_Account').agg({
        'Amount(KRW)': 'sum',
        'Code': 'count'
    }).round(0)
    summary.columns = ['Amount(KRW)', '행수']
    summary = summary.sort_values('Amount(KRW)', ascending=False)
    print(summary)
    
    print(f"\n{'='*60}")
    print(f"[Step 3] 완료!")
    print(f"{'='*60}")
    
    return df_pl

# 실행
if __name__ == "__main__":
    # 경로 설정
    input_folder = r"C:\Users\user\InBody Co., Ltd\회계팀 - General\★★Accounting Folder★★vF\1. 재무결산\FY2025\2025_09_3Q\2. 연결재무제표\3. Final PKG"
    consolidated_file = r"C:\Users\user\InBody Co., Ltd\회계팀 - General\★★Accounting Folder★★vF\1. 재무결산\FY2025\2025_12_4Q\2. 연결재무제표\Income statement\consolidated_financial_data_25Q3.xlsx"
    final_file = r"C:\Users\user\InBody Co., Ltd\회계팀 - General\★★Accounting Folder★★vF\1. 재무결산\FY2025\2025_12_4Q\2. 연결재무제표\Income statement\final_financial_data_25Q3.xlsx"
    
    print("="*60)
    print("해외법인 재무제표 데이터 통합 시작")
    print("="*60)
    
    # Step 1: 데이터 통합
    result_df = consolidate_financial_data(input_folder, consolidated_file, ENTITY_CURRENCY_MAP)
    
    if result_df is not None:
        # Step 2: 환율 적용
        final_df = process_final_file(consolidated_file, final_file, ENTITY_CURRENCY_MAP)
        
        if final_df is not None:
            # Step 3: PL 시트 생성
            pl_df = create_pl_sheet(final_file)
            
            print("\n" + "="*60)
            print("✅ 모든 작업이 완료되었습니다!")
            print("="*60)
            print(f"\n생성된 파일:")
            print(f"  1. {consolidated_file}")
            print(f"  2. {final_file} (Sheet1 + PL)")
        else:
            print("\n❌ Step 2 실패")
    else:
        print("\n❌ Step 1 실패")

해외법인 재무제표 데이터 통합 시작
총 16개 파일 발견



C:\Users\user\AppData\Local\Temp\ipykernel_21576\55255598.py:104: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['<Hide>' '계정과목' '매출액' '매출원가' '매출총이익' '판매비와관리비' '영업이익' '기타수익' '기타비용'
 '금융수익' '금융비용' '지분법손익' '법인세비용차감전이익' '법인세비용' '세후중단영업손익' '당기순이익' '포괄손익'
 '총포괄손익']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[mask, 10] = df.loc[mask, col]  # K
C:\Users\user\AppData\Local\Temp\ipykernel_21576\55255598.py:109: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['Account' 'Sales' 'Cost of Goods Sold ' 'Gross Profit'
 'Selling and Administration Expense' 'Operating Income' 'Other Revenue'
 'Other Expense' 'Financial Revenue' 'Financial Expense'
 'Gain and Loss on Equity Method' 'Net Income before Corporate Income Tax'
 'Corporate Income Tax' 'Gain and Loss from discontinued operations '
 'Net Income ' 'Ot

✓ 1. Inbody K-IFRS Package_HQ_2025_3Q_Final(251024).xlsx - 주식회사 인바디 (KRW) 처리 완료


C:\Users\user\AppData\Local\Temp\ipykernel_21576\55255598.py:104: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['<Hide>' '계정과목' '매출액' '매출원가' '매출총이익' '판매비와관리비' '영업이익' '기타수익' '기타비용'
 '금융수익' '금융비용' '지분법손익' '법인세비용차감전이익' '법인세비용' '세후중단영업손익' '당기순이익' '포괄손익'
 '총포괄손익']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[mask, 10] = df.loc[mask, col]  # K
C:\Users\user\AppData\Local\Temp\ipykernel_21576\55255598.py:109: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['Account' 'Sales' 'Cost of Goods Sold ' 'Gross Profit'
 'Selling and Administration Expense' 'Operating Income' 'Other Revenue'
 'Other Expense' 'Financial Revenue' 'Financial Expense'
 'Gain and Loss on Equity Method' 'Net Income before Corporate Income Tax'
 'Corporate Income Tax' 'Gain and Loss from discontinued operations '
 'Net Income ' 'Ot

✓ 10. Inbody K-IFRS Package_KOROT_2025_3Q_Final(251030_2).xlsx - ㈜코르트 (KRW) 처리 완료


...

✓ 7. Inbody K-IFRS Package_Europe_2025_3Q_Final(251023).xlsx - Inbody Europe B.V. (EUR) 처리 완료


C:\Users\user\AppData\Local\Temp\ipykernel_21576\55255598.py:104: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['<Hide>' '계정과목' '매출액' '매출원가' '매출총이익' '판매비와관리비' '영업이익' '기타수익' '기타비용'
 '금융수익' '금융비용' '지분법손익' '법인세비용차감전이익' '법인세비용' '세후중단영업손익' '당기순이익' '포괄손익'
 '총포괄손익']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[mask, 10] = df.loc[mask, col]  # K
C:\Users\user\AppData\Local\Temp\ipykernel_21576\55255598.py:109: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['Account' 'Sales' 'Cost of Goods Sold ' 'Gross Profit'
 'Selling and Administration Expense' 'Operating Income' 'Other Revenue'
 'Other Expense' 'Financial Revenue' 'Financial Expense'
 'Gain and Loss on Equity Method' 'Net Income before Corporate Income Tax'
 'Corporate Income Tax' 'Gain and Loss from discontinued operations '
 'Net Income ' 'Ot

✓ 8. Inbody K-IFRS Package_삼한정공_2025_3Q_Final(251023).xlsx - (주)삼한정공 (KRW) 처리 완료


C:\Users\user\AppData\Local\Temp\ipykernel_21576\55255598.py:104: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['<Hide>' '계정과목' '매출액' '매출원가' '매출총이익' '판매비와관리비' '영업이익' '기타수익' '기타비용'
 '금융수익' '금융비용' '지분법손익' '법인세비용차감전이익' '법인세비용' '세후중단영업손익' '당기순이익' '포괄손익'
 '총포괄손익']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[mask, 10] = df.loc[mask, col]  # K
C:\Users\user\AppData\Local\Temp\ipykernel_21576\55255598.py:109: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['Account' 'Sales' 'Cost of Goods Sold ' 'Gross Profit'
 'Selling and Administration Expense' 'Operating Income' 'Other Revenue'
 'Other Expense' 'Financial Revenue' 'Financial Expense'
 'Gain and Loss on Equity Method' 'Net Income before Corporate Income Tax'
 'Corporate Income Tax' 'Gain and Loss from discontinued operations '
 'Net Income ' 'Ot

✓ 9. Inbody K-IFRS Package_KOCP_2025_3Q_Final(250925).xlsx - 케이오씨피 프로젝트 제2호 벤처투자조합 (KRW) 처리 완료

[Step 1] 통합 완료!
총 4960 행의 데이터
저장 위치: C:\Users\user\InBody Co., Ltd\회계팀 - General\★★Accounting Folder★★vF\1. 재무결산\FY2025\2025_12_4Q\2. 연결재무제표\Income statement\consolidated_financial_data_25Q3.xlsx
Entity 개수: 16

[Entity별 데이터 건수]
Entity                                   Statement  Currency  Period
(주)삼한정공                                  BS         KRW       20253Q    195
                                         IS         KRW       20253Q    115
ADJ                                      BS         KRW       20253Q    195
                                         IS         KRW       20253Q    115
BIOSPACE LATIN AMERICA S DE RL DE CV(*)  BS         MXN       20253Q    195
                                         IS         MXN       20253Q    115
BWA                                      BS         USD       20253Q    195
                                         IS         USD       20253Q    115

In [5]:
import pandas as pd
import os
from pathlib import Path

# ==================== 여기를 수정하세요! ====================
# 기간 설정
PERIOD = "20252Q"  # 예: "2025_4Q", "2025_12월", "FY2025" 등

# Entity별 통화 및 환율 매핑 (BS: 기말환율, PL: 평균환율)

ENTITY_CURRENCY_MAP = {
    'Biospace Inc. DBA INBODY': {
        'currency': 'USD', 
        'bs_rate': 1356.40,
        'pl_rate': 1427.94
    },
    'InBody Japan Inc.': {
        'currency': 'JPY', 
        'bs_rate': 9.3898,
        'pl_rate': 9.6226
    },
    'Biospace Co.,Ltd.': {
        'currency': 'CNH', 
        'bs_rate': 189.16,
        'pl_rate': 196.61
    },
    'InBody Oceania ': {
        'currency': 'AUD', 
        'bs_rate': 886.61,
        'pl_rate': 904.85
    },
    'BIOSPACE LATIN AMERICA S DE RL DE CV(*)': {
        'currency': 'MXN', 
        'bs_rate': 72.11,
        'pl_rate': 71.52
    },
    'BWA': {
        'currency': 'USD', 
        'bs_rate': 1356.40,
        'pl_rate': 1427.94
    },
    '㈜인바디헬스케어': {
        'currency': 'KRW', 
        'bs_rate': 1.0,
        'pl_rate': 1.0
    },
    'InBody Vietnam': {
        'currency': 'VND', 
        'bs_rate': 0.0520,
        'pl_rate': 0.0556
    },
    'INBODY ASIA SDN. BHD.': {
        'currency': 'MYR', 
        'bs_rate': 320.74,
        'pl_rate': 325.74
    },
    'InBody India Pvt. Ltd': {
        'currency': 'INR', 
        'bs_rate': 15.87,
        'pl_rate': 16.59
    },
    'Inbody Europe B.V.': {
        'currency': 'EUR', 
        'bs_rate': 1591.80,
        'pl_rate': 1559.87
    },
    '㈜코르트': {
        'currency': 'KRW', 
        'bs_rate': 1.0,
        'pl_rate': 1.0
    },
    'INBODY TURKEY MEDİKAL TİCARET LİMİTED ŞİRKETİ': {
        'currency': 'TRY', 
        'bs_rate': 33.40,
        'pl_rate': 36.05
    },
    '(주)삼한정공': {
        'currency': 'KRW', 
        'bs_rate': 1.0,
        'pl_rate': 1.0
    },
    '주식회사 인바디': {
        'currency': 'KRW', 
        'bs_rate': 1.0,
        'pl_rate': 1.0
    },
    '케이오씨피 프로젝트 제2호 벤처투자조합': {
        'currency': 'KRW', 
        'bs_rate': 1.0,
        'pl_rate': 1.0
    },
    'ADJ': {
        'currency': 'KRW', 
        'bs_rate': 1.0,
        'pl_rate': 1.0
    },
}
# ============================================================

def preprocess_sheet(df, sheet_type):
    """데이터 전처리: H~J → K, M~O → P로 이동"""
    # H~J 열 데이터를 K로 이동 (0-indexed: 7,8,9 → 10)
    for col in [7, 8, 9]:  # H, I, J
        mask = df[col].notna()
        df.loc[mask, 10] = df.loc[mask, col]  # K
    
    # M~O 열 데이터를 P로 이동 (0-indexed: 12,13,14 → 15)
    for col in [12, 13, 14]:  # M, N, O
        mask = df[col].notna()
        df.loc[mask, 15] = df.loc[mask, col]  # P
    
    return df

def extract_data_from_file(file_path, entity_currency_map):
    """단일 엑셀 파일에서 BS, IS 데이터 추출"""
    all_data = []
    
    try:
        # BS 시트 처리
        bs_df = pd.read_excel(file_path, sheet_name='BS', header=None)
        entity = bs_df.iloc[0, 6]  # G1 (0-indexed: row 0, col 6)
        
        # Entity별 통화 정보 가져오기 (ENTITY_CURRENCY_MAP에서)
        currency = entity_currency_map.get(entity, {}).get('currency', 'Unknown')
        
        # 전처리
        bs_df = preprocess_sheet(bs_df, 'BS')
        
        # 6행(index 5)을 컬럼명으로, 7~231행(index 6~230) 데이터 추출
        bs_data = bs_df.iloc[6:231, [6, 10, 15, 22]]  # G, K, P, W (0-indexed: 6, 10, 15, 22)
        bs_data.columns = bs_df.iloc[5, [6, 10, 15, 22]].values  # 6행을 컬럼명으로
        
        bs_data['Entity'] = entity
        bs_data['Statement'] = 'BS'
        bs_data['Currency'] = currency
        bs_data['Period'] = PERIOD
        
        # 공란 제거 (모든 값이 NaN인 행 제거)
        bs_data = bs_data.dropna(how='all', subset=bs_data.columns[:-4])  # Entity, Statement, Currency, Period 제외
        
        # K열(두 번째 열)에 값이 없는 행 삭제
        k_col_name = bs_data.columns[1]  # K열 컬럼명
        bs_data = bs_data[bs_data[k_col_name].notna()]
        
        # W열만 숫자형으로 변환 (공란을 0으로, G/K/P는 텍스트 유지)
        w_col_name = bs_data.columns[3]  # W열 컬럼명
        bs_data[w_col_name] = pd.to_numeric(bs_data[w_col_name], errors='coerce').fillna(0)
        
        all_data.append(bs_data)
        
        # IS 시트 처리
        is_df = pd.read_excel(file_path, sheet_name='IS', header=None)
        entity = is_df.iloc[0, 6]  # G1
        
        # Entity별 통화 정보 가져오기 (ENTITY_CURRENCY_MAP에서)
        currency = entity_currency_map.get(entity, {}).get('currency', 'Unknown')
        
        # 전처리
        is_df = preprocess_sheet(is_df, 'IS')
        
        # 6행을 컬럼명으로, 7~129행(index 6~128) 데이터 추출
        is_data = is_df.iloc[6:129, [6, 10, 15, 22]]  # G, K, P, W
        is_data.columns = is_df.iloc[5, [6, 10, 15, 22]].values
        
        is_data['Entity'] = entity
        is_data['Statement'] = 'IS'
        is_data['Currency'] = currency
        is_data['Period'] = PERIOD
        
        # 공란 제거
        is_data = is_data.dropna(how='all', subset=is_data.columns[:-4])  # Entity, Statement, Currency, Period 제외
        
        # K열(두 번째 열)에 값이 없는 행 삭제
        k_col_name = is_data.columns[1]  # K열 컬럼명
        is_data = is_data[is_data[k_col_name].notna()]
        
        # W열만 숫자형으로 변환 (공란을 0으로, G/K/P는 텍스트 유지)
        w_col_name = is_data.columns[3]  # W열 컬럼명
        is_data[w_col_name] = pd.to_numeric(is_data[w_col_name], errors='coerce').fillna(0)
        
        all_data.append(is_data)
        
        print(f"✓ {os.path.basename(file_path)} - {entity} ({currency}) 처리 완료")
        
    except Exception as e:
        print(f"✗ {os.path.basename(file_path)} 처리 실패: {str(e)}")
    
    return all_data

def consolidate_financial_data(input_folder, output_file, entity_currency_map):
    """여러 엑셀 파일을 통합"""
    all_results = []
    
    # 폴더 내 모든 엑셀 파일 처리
    excel_files = list(Path(input_folder).glob('*.xlsx'))
    excel_files = [f for f in excel_files if not f.name.startswith('~')]  # 임시파일 제외
    
    print(f"총 {len(excel_files)}개 파일 발견\n")
    
    for file_path in sorted(excel_files):
        data_list = extract_data_from_file(file_path, entity_currency_map)
        all_results.extend(data_list)
    
    # 모든 데이터 통합
    if all_results:
        consolidated_df = pd.concat(all_results, ignore_index=True)
        
        # 컬럼 순서 조정 (Entity, Statement, Currency, Period를 앞으로)
        cols = ['Entity', 'Statement', 'Currency', 'Period'] + [col for col in consolidated_df.columns if col not in ['Entity', 'Statement', 'Currency', 'Period']]
        consolidated_df = consolidated_df[cols]
        
        # 결과 저장
        consolidated_df.to_excel(output_file, index=False)
        
        print(f"\n{'='*60}")
        print(f"[Step 1] 통합 완료!")
        print(f"총 {len(consolidated_df)} 행의 데이터")
        print(f"저장 위치: {output_file}")
        print(f"Entity 개수: {consolidated_df['Entity'].nunique()}")
        print(f"{'='*60}")
        
        # 요약 정보
        print("\n[Entity별 데이터 건수]")
        print(consolidated_df.groupby(['Entity', 'Statement', 'Currency', 'Period']).size())
        
        # Unknown Currency 확인
        unknown_currency = consolidated_df[consolidated_df['Currency'] == 'Unknown']['Entity'].unique()
        if len(unknown_currency) > 0:
            print(f"\n⚠️  통화 정보가 없는 Entity: {list(unknown_currency)}")
            print("   → ENTITY_CURRENCY_MAP에 추가해주세요!")
        
        return consolidated_df
    else:
        print("처리된 데이터가 없습니다.")
        return None

def process_final_file(input_file, output_file, entity_currency_map):
    """
    consolidated_financial_data.xlsx를 읽어서 
    Amount, Exchange_Rate, Amount(KRW) 추가 및 PL 시트 생성
    """
    print("\n" + "="*60)
    print("[Step 2] 통화 및 환율 적용 중...")
    print("="*60)
    
    # 데이터 읽기
    df = pd.read_excel(input_file)
    
    print(f"✓ 데이터 로드 완료: {len(df)} 행")
    print(f"✓ 전체 컬럼: {list(df.columns)}")
    
    # 1. Amount 열 추가 - 각 행의 Currency에 해당하는 컬럼 값
    # 통화 컬럼 매핑 (Currency와 실제 컬럼명이 다를 수 있음)
    통화_컬럼_매핑 = {
        'AUD': 'AUD',
        'KRW': 'KRW',
        'JPY': 'JPY',
        'USD': 'USD',
        'CNY': 'CNH',  # Currency는 CNY지만 컬럼명은 CNH
        'CNH': 'CNH',  # CNH도 허용
        'MXN': 'MXN',
        'EUR': 'EUR',
        'VND': 'VND',
        'MYR': 'MYR',
        'INR': 'INR',
        'TRY': 'TRY'
    }
    
    def get_amount(row):
        currency = row['Currency']
        col_name = 통화_컬럼_매핑.get(currency)
        
        # 매핑된 컬럼이 있고 해당 컬럼이 존재하면
        if col_name and col_name in df.columns:
            value = row[col_name]
            return 0 if pd.isna(value) else float(value)
        else:
            # 통화 컬럼이 없으면 모든 통화 컬럼 확인해서 값이 있는 것 사용
            통화_컬럼들 = ['AUD', 'KRW', 'JPY', 'USD', 'CNH', 'MXN', 'EUR', 'VND', 'MYR', 'INR', 'TRY']
            for col in 통화_컬럼들:
                if col in df.columns and pd.notna(row[col]) and row[col] != 0:
                    return float(row[col])
            return 0
    
    df['Amount'] = df.apply(get_amount, axis=1)
    
    print(f"✓ Amount 생성 완료")
    print(f"  - Amount 합계: {df['Amount'].sum():,.2f}")
    print(f"  - Amount > 0인 행: {(df['Amount'] > 0).sum()} / {len(df)}")
    
    # 2. Entity와 Statement별 Exchange_Rate 매핑
    def get_exchange_rate(row):
        entity = row['Entity']
        statement = row['Statement']
        entity_info = entity_currency_map.get(entity, {})
        
        # BS는 bs_rate, IS(PL)는 pl_rate 사용
        if statement == 'BS':
            return entity_info.get('bs_rate', 1.0)
        else:  # IS
            return entity_info.get('pl_rate', 1.0)
    
    df['Exchange_Rate'] = df.apply(get_exchange_rate, axis=1)
    
    # 3. Amount(KRW) 계산 (소숫점 반올림)
    df['Amount(KRW)'] = (df['Amount'] * df['Exchange_Rate']).round(0).astype('Int64')
    
    print(f"  - Amount(KRW) 합계: {df['Amount(KRW)'].sum():,.0f}")
    
    # 컬럼 순서 조정
    base_cols = ['Entity', 'Statement', 'Currency', 'Period']
    data_cols = [col for col in df.columns if col not in base_cols + ['Amount', 'Exchange_Rate', 'Amount(KRW)']]
    final_cols = base_cols + data_cols + ['Amount', 'Exchange_Rate', 'Amount(KRW)']
    
    df = df[final_cols]
    
    # Sheet1 저장
    with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
        df.to_excel(writer, sheet_name='Sheet1', index=False)
    
    print(f"\n✓ Sheet1 저장 완료: {output_file}")
    
    # 통화별 요약
    print("\n[Entity별 통화 및 총액]")
    summary = df.groupby(['Entity', 'Statement', 'Currency']).agg({
        'Amount': 'sum',
        'Amount(KRW)': 'sum',
        'Exchange_Rate': 'first'  # 환율 표시
    }).round(0)
    print(summary)
    
    print(f"\n{'='*60}")
    print(f"[Step 2] 완료!")
    print(f"총 {len(df)} 행의 데이터")
    print(f"저장 위치: {output_file}")
    print(f"{'='*60}")
    
    # BS와 PL 환율 확인
    print("\n[BS 환율 매핑 상태]")
    bs_data = df[df['Statement'] == 'BS'][['Entity', 'Currency', 'Exchange_Rate']].drop_duplicates()
    for _, row in bs_data.iterrows():
        print(f"✓ {row['Entity']} ({row['Currency']}) - {row['Exchange_Rate']} (기말환율)")
    
    print("\n[PL 환율 매핑 상태]")
    pl_data = df[df['Statement'] == 'IS'][['Entity', 'Currency', 'Exchange_Rate']].drop_duplicates()
    for _, row in pl_data.iterrows():
        print(f"✓ {row['Entity']} ({row['Currency']}) - {row['Exchange_Rate']} (평균환율)")
    
    return df

def create_pl_sheet(final_file):
    """
    final_financial_data.xlsx에 PL 시트 추가
    """
    print("\n" + "="*60)
    print("[Step 3] PL 시트 생성 중...")
    print("="*60)
    
    # final_financial_data.xlsx의 Sheet1 읽기
    df = pd.read_excel(final_file, sheet_name='Sheet1')
    
    # 1. Statement = "IS"만 필터링
    df_pl = df[df['Statement'] == 'IS'].copy()
    print(f"✓ IS 데이터 필터링: {len(df_pl)} 행")
    
    # 2. 특정 컬럼만 선택
    selected_cols = ['Entity', 'Statement', 'Currency', 'Period', 'Code', '계정과목', 'Account', 
                     'Amount', 'Exchange_Rate', 'Amount(KRW)']
    df_pl = df_pl[selected_cols]
    
    # 3. 특정 code 행 삭제
    codes_to_remove = [40000, 50000, 59000, 60000, 69000, 71000, 72000, 73000, 74000, 
                       75000, 79000, 80000, 81000, 82000, 83000, 84000]
    df_pl = df_pl[~df_pl['Code'].isin(codes_to_remove)]
    print(f"✓ 특정 Code 삭제 후: {len(df_pl)} 행")
    
    # 4. Rev_Account 컬럼 생성
    def get_rev_account(code):
        code_str = str(code)
        
        if code_str.startswith('4'):
            return '매출'
        elif code_str.startswith('5'):
            return '매출원가'
        elif code in [60001, 60002, 60003, 60004, 60005, 60006, 60034]:
            return '인건비'
        elif code == 60007:
            return '복리후생비'
        elif code == 60008:
            return '여비교통비'
        elif code == 60015:
            return '광고선전비'
        elif code == 60017:
            return '운반비'
        elif code == 60018:
            return '지급수수료'
        elif code == 60019:
            return '경상연구개발비'
        elif code == 60027:
            return '판매수수료'
        elif code in [60021, 60029]:
            return '감가상각비'
        elif code == 60033:
            return '사용권자산상각비'
        elif code == 60022:
            return '대손상각비'
        elif code_str.startswith('60'):
            return '기타'
        elif code_str.startswith('71'):
            return '영업외수익'
        elif code_str.startswith('72'):
            return '영업외비용'
        elif code_str.startswith('73'):
            return '금융수익'
        elif code_str.startswith('74'):
            return '금융비용'
        elif code_str.startswith('75'):
            return '지분법손익'
        elif code_str.startswith('80'):
            return '법인세비용'
        elif code_str.startswith('81'):
            return '세후중단영업손익'
        elif code_str.startswith('82'):
            return '당기순이익'
        elif code_str.startswith('83'):
            return '포괄손익'
        elif code_str.startswith('84'):
            return '총포괄손익'
        else:
            return '기타'
    
    df_pl['Rev_Account'] = df_pl['Code'].apply(get_rev_account)
    
    # 컬럼 순서 조정 (Rev_Account를 Account 다음에)
    final_cols = ['Entity', 'Statement', 'Currency', 'Period', 'Code', '계정과목', 'Account', 
                  'Rev_Account', 'Amount', 'Exchange_Rate', 'Amount(KRW)']
    df_pl = df_pl[final_cols]
    
    # PL 시트 추가
    with pd.ExcelWriter(final_file, engine='openpyxl', mode='a') as writer:
        df_pl.to_excel(writer, sheet_name='PL', index=False)
    
    print(f"✓ PL 시트 저장 완료: {final_file}")
    
    # Rev_Account별 요약
    print("\n[Rev_Account별 집계]")
    summary = df_pl.groupby('Rev_Account').agg({
        'Amount(KRW)': 'sum',
        'Code': 'count'
    }).round(0)
    summary.columns = ['Amount(KRW)', '행수']
    summary = summary.sort_values('Amount(KRW)', ascending=False)
    print(summary)
    
    print(f"\n{'='*60}")
    print(f"[Step 3] 완료!")
    print(f"{'='*60}")
    
    return df_pl

# 실행
if __name__ == "__main__":
    # 경로 설정
    input_folder = r"C:\Users\user\InBody Co., Ltd\회계팀 - General\★★Accounting Folder★★vF\1. 재무결산\FY2025\2025_06_2Q\2.연결재무제표\3. PKG"
    consolidated_file = r"C:\Users\user\InBody Co., Ltd\회계팀 - General\★★Accounting Folder★★vF\1. 재무결산\FY2025\2025_12_4Q\2. 연결재무제표\Income statement\consolidated_financial_data_25Q2.xlsx"
    final_file = r"C:\Users\user\InBody Co., Ltd\회계팀 - General\★★Accounting Folder★★vF\1. 재무결산\FY2025\2025_12_4Q\2. 연결재무제표\Income statement\final_financial_data_25Q2.xlsx"
    
    print("="*60)
    print("해외법인 재무제표 데이터 통합 시작")
    print("="*60)
    
    # Step 1: 데이터 통합
    result_df = consolidate_financial_data(input_folder, consolidated_file, ENTITY_CURRENCY_MAP)
    
    if result_df is not None:
        # Step 2: 환율 적용
        final_df = process_final_file(consolidated_file, final_file, ENTITY_CURRENCY_MAP)
        
        if final_df is not None:
            # Step 3: PL 시트 생성
            pl_df = create_pl_sheet(final_file)
            
            print("\n" + "="*60)
            print("✅ 모든 작업이 완료되었습니다!")
            print("="*60)
            print(f"\n생성된 파일:")
            print(f"  1. {consolidated_file}")
            print(f"  2. {final_file} (Sheet1 + PL)")
        else:
            print("\n❌ Step 2 실패")
    else:
        print("\n❌ Step 1 실패")

해외법인 재무제표 데이터 통합 시작
총 16개 파일 발견



C:\Users\user\AppData\Local\Temp\ipykernel_21576\2058837659.py:105: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['<Hide>' '계정과목' '매출액' '매출원가' '매출총이익' '판매비와관리비' '영업이익' '기타수익' '기타비용'
 '금융수익' '금융비용' '지분법손익' '법인세비용차감전이익' '법인세비용' '세후중단영업손익' '당기순이익' '포괄손익'
 '총포괄손익']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[mask, 10] = df.loc[mask, col]  # K
C:\Users\user\AppData\Local\Temp\ipykernel_21576\2058837659.py:110: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['Account' 'Sales' 'Cost of Goods Sold ' 'Gross Profit'
 'Selling and Administration Expense' 'Operating Income' 'Other Revenue'
 'Other Expense' 'Financial Revenue' 'Financial Expense'
 'Gain and Loss on Equity Method' 'Net Income before Corporate Income Tax'
 'Corporate Income Tax' 'Gain and Loss from discontinued operations '
 'Net Income '

✓ 1. Inbody K-IFRS Package_HQ_2025_2Q_250801.xlsx - 주식회사 인바디 (KRW) 처리 완료


C:\Users\user\AppData\Local\Temp\ipykernel_21576\2058837659.py:105: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['<Hide>' '계정과목' '매출액' '매출원가' '매출총이익' '판매비와관리비' '영업이익' '기타수익' '기타비용'
 '금융수익' '금융비용' '지분법손익' '법인세비용차감전이익' '법인세비용' '세후중단영업손익' '당기순이익' '포괄손익'
 '총포괄손익']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[mask, 10] = df.loc[mask, col]  # K
C:\Users\user\AppData\Local\Temp\ipykernel_21576\2058837659.py:110: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['Account' 'Sales' 'Cost of Goods Sold ' 'Gross Profit'
 'Selling and Administration Expense' 'Operating Income' 'Other Revenue'
 'Other Expense' 'Financial Revenue' 'Financial Expense'
 'Gain and Loss on Equity Method' 'Net Income before Corporate Income Tax'
 'Corporate Income Tax' 'Gain and Loss from discontinued operations '
 'Net Income '

✓ 10. Inbody K-IFRS Package_KOROT_2025_2Q_Draft(250722).xlsx - ㈜코르트 (KRW) 처리 완료


...

✓ 7. Inbody K-IFRS Package_Europe_2025_2Q(250722).xlsx - Inbody Europe B.V. (EUR) 처리 완료


C:\Users\user\AppData\Local\Temp\ipykernel_21576\2058837659.py:105: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['<Hide>' '계정과목' '매출액' '매출원가' '매출총이익' '판매비와관리비' '영업이익' '기타수익' '기타비용'
 '금융수익' '금융비용' '지분법손익' '법인세비용차감전이익' '법인세비용' '세후중단영업손익' '당기순이익' '포괄손익'
 '총포괄손익']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[mask, 10] = df.loc[mask, col]  # K
C:\Users\user\AppData\Local\Temp\ipykernel_21576\2058837659.py:110: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['Account' 'Sales' 'Cost of Goods Sold ' 'Gross Profit'
 'Selling and Administration Expense' 'Operating Income' 'Other Revenue'
 'Other Expense' 'Financial Revenue' 'Financial Expense'
 'Gain and Loss on Equity Method' 'Net Income before Corporate Income Tax'
 'Corporate Income Tax' 'Gain and Loss from discontinued operations '
 'Net Income '

✓ 8. Inbody K-IFRS Package_삼한정공_2025_2Q_Draft(250728_2).xlsx - (주)삼한정공 (KRW) 처리 완료


C:\Users\user\AppData\Local\Temp\ipykernel_21576\2058837659.py:105: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['<Hide>' '계정과목' '매출액' '매출원가' '매출총이익' '판매비와관리비' '영업이익' '기타수익' '기타비용'
 '금융수익' '금융비용' '지분법손익' '법인세비용차감전이익' '법인세비용' '세후중단영업손익' '당기순이익' '포괄손익'
 '총포괄손익']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[mask, 10] = df.loc[mask, col]  # K
C:\Users\user\AppData\Local\Temp\ipykernel_21576\2058837659.py:110: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['Account' 'Sales' 'Cost of Goods Sold ' 'Gross Profit'
 'Selling and Administration Expense' 'Operating Income' 'Other Revenue'
 'Other Expense' 'Financial Revenue' 'Financial Expense'
 'Gain and Loss on Equity Method' 'Net Income before Corporate Income Tax'
 'Corporate Income Tax' 'Gain and Loss from discontinued operations '
 'Net Income '

✓ 9. Inbody K-IFRS Package_KOCP_2025_2Q_Draft(250720).xlsx - 케이오씨피 프로젝트 제2호 벤처투자조합 (KRW) 처리 완료

[Step 1] 통합 완료!
총 4960 행의 데이터
저장 위치: C:\Users\user\InBody Co., Ltd\회계팀 - General\★★Accounting Folder★★vF\1. 재무결산\FY2025\2025_12_4Q\2. 연결재무제표\Income statement\consolidated_financial_data_25Q2.xlsx
Entity 개수: 16

[Entity별 데이터 건수]
Entity                                   Statement  Currency  Period
(주)삼한정공                                  BS         KRW       20252Q    195
                                         IS         KRW       20252Q    115
ADJ                                      BS         KRW       20252Q    195
                                         IS         KRW       20252Q    115
BIOSPACE LATIN AMERICA S DE RL DE CV(*)  BS         MXN       20252Q    195
                                         IS         MXN       20252Q    115
BWA                                      BS         USD       20252Q    195
                                         IS         USD       20252Q    115

In [6]:
import pandas as pd
import os
from pathlib import Path

# ==================== 여기를 수정하세요! ====================
# 기간 설정
PERIOD = "20251Q"  # 예: "2025_4Q", "2025_12월", "FY2025" 등

# Entity별 통화 및 환율 매핑 (BS: 기말환율, PL: 평균환율)
ENTITY_CURRENCY_MAP = {
    'Biospace Inc. DBA INBODY': {
        'currency': 'USD', 
        'bs_rate': 1466.50,
        'pl_rate': 1452.66
    },
    'InBody Japan Inc.': {
        'currency': 'JPY', 
        'bs_rate': 9.8176,
        'pl_rate': 9.5363
    },
    'Biospace Co.,Ltd.': {
        'currency': 'CNH', 
        'bs_rate': 201.86,
        'pl_rate': 199.33
    },
    'InBody Oceania ': {
        'currency': 'AUD', 
        'bs_rate': 920.67,
        'pl_rate': 911.86
    },
    'BIOSPACE LATIN AMERICA S DE RL DE CV(*)': {
        'currency': 'MXN', 
        'bs_rate': 71.93,
        'pl_rate': 71.15
    },
    'BWA': {
        'currency': 'USD', 
        'bs_rate': 1466.50,
        'pl_rate': 1452.66
    },
    '㈜인바디헬스케어': {
        'currency': 'KRW', 
        'bs_rate': 1.0,
        'pl_rate': 1.0
    },
    'InBody Vietnam': {
        'currency': 'VND', 
        'bs_rate': 0.0573,
        'pl_rate': 0.0571
    },
    'INBODY ASIA SDN. BHD.': {
        'currency': 'MYR', 
        'bs_rate': 330.48,
        'pl_rate': 326.25
    },
    'InBody India Pvt. Ltd': {
        'currency': 'INR', 
        'bs_rate': 17.14,
        'pl_rate': 16.78
    },
    'Inbody Europe B.V.': {
        'currency': 'EUR', 
        'bs_rate': 1587.85,
        'pl_rate': 1529.33
    },
    '㈜코르트': {
        'currency': 'KRW', 
        'bs_rate': 1.0,
        'pl_rate': 1.0
    },
    'INBODY TURKEY MEDİKAL TİCARET LİMİTED ŞİRKETİ': {
        'currency': 'TRY', 
        'bs_rate': 33.40,
        'pl_rate': 36.05
    },
    '(주)삼한정공': {
        'currency': 'KRW', 
        'bs_rate': 1.0,
        'pl_rate': 1.0
    },
    '주식회사 인바디': {
        'currency': 'KRW', 
        'bs_rate': 1.0,
        'pl_rate': 1.0
    },
    '케이오씨피 프로젝트 제2호 벤처투자조합': {
        'currency': 'KRW', 
        'bs_rate': 1.0,
        'pl_rate': 1.0
    },
    'ADJ': {
        'currency': 'KRW', 
        'bs_rate': 1.0,
        'pl_rate': 1.0
    },
}
# ============================================================

def preprocess_sheet(df, sheet_type):
    """데이터 전처리: H~J → K, M~O → P로 이동"""
    # H~J 열 데이터를 K로 이동 (0-indexed: 7,8,9 → 10)
    for col in [7, 8, 9]:  # H, I, J
        mask = df[col].notna()
        df.loc[mask, 10] = df.loc[mask, col]  # K
    
    # M~O 열 데이터를 P로 이동 (0-indexed: 12,13,14 → 15)
    for col in [12, 13, 14]:  # M, N, O
        mask = df[col].notna()
        df.loc[mask, 15] = df.loc[mask, col]  # P
    
    return df

def extract_data_from_file(file_path, entity_currency_map):
    """단일 엑셀 파일에서 BS, IS 데이터 추출"""
    all_data = []
    
    try:
        # BS 시트 처리
        bs_df = pd.read_excel(file_path, sheet_name='BS', header=None)
        entity = bs_df.iloc[0, 6]  # G1 (0-indexed: row 0, col 6)
        
        # Entity별 통화 정보 가져오기 (ENTITY_CURRENCY_MAP에서)
        currency = entity_currency_map.get(entity, {}).get('currency', 'Unknown')
        
        # 전처리
        bs_df = preprocess_sheet(bs_df, 'BS')
        
        # 6행(index 5)을 컬럼명으로, 7~231행(index 6~230) 데이터 추출
        bs_data = bs_df.iloc[6:231, [6, 10, 15, 22]]  # G, K, P, W (0-indexed: 6, 10, 15, 22)
        bs_data.columns = bs_df.iloc[5, [6, 10, 15, 22]].values  # 6행을 컬럼명으로
        
        bs_data['Entity'] = entity
        bs_data['Statement'] = 'BS'
        bs_data['Currency'] = currency
        bs_data['Period'] = PERIOD
        
        # 공란 제거 (모든 값이 NaN인 행 제거)
        bs_data = bs_data.dropna(how='all', subset=bs_data.columns[:-4])  # Entity, Statement, Currency, Period 제외
        
        # K열(두 번째 열)에 값이 없는 행 삭제
        k_col_name = bs_data.columns[1]  # K열 컬럼명
        bs_data = bs_data[bs_data[k_col_name].notna()]
        
        # W열만 숫자형으로 변환 (공란을 0으로, G/K/P는 텍스트 유지)
        w_col_name = bs_data.columns[3]  # W열 컬럼명
        bs_data[w_col_name] = pd.to_numeric(bs_data[w_col_name], errors='coerce').fillna(0)
        
        all_data.append(bs_data)
        
        # IS 시트 처리
        is_df = pd.read_excel(file_path, sheet_name='IS', header=None)
        entity = is_df.iloc[0, 6]  # G1
        
        # Entity별 통화 정보 가져오기 (ENTITY_CURRENCY_MAP에서)
        currency = entity_currency_map.get(entity, {}).get('currency', 'Unknown')
        
        # 전처리
        is_df = preprocess_sheet(is_df, 'IS')
        
        # 6행을 컬럼명으로, 7~129행(index 6~128) 데이터 추출
        is_data = is_df.iloc[6:129, [6, 10, 15, 22]]  # G, K, P, W
        is_data.columns = is_df.iloc[5, [6, 10, 15, 22]].values
        
        is_data['Entity'] = entity
        is_data['Statement'] = 'IS'
        is_data['Currency'] = currency
        is_data['Period'] = PERIOD
        
        # 공란 제거
        is_data = is_data.dropna(how='all', subset=is_data.columns[:-4])  # Entity, Statement, Currency, Period 제외
        
        # K열(두 번째 열)에 값이 없는 행 삭제
        k_col_name = is_data.columns[1]  # K열 컬럼명
        is_data = is_data[is_data[k_col_name].notna()]
        
        # W열만 숫자형으로 변환 (공란을 0으로, G/K/P는 텍스트 유지)
        w_col_name = is_data.columns[3]  # W열 컬럼명
        is_data[w_col_name] = pd.to_numeric(is_data[w_col_name], errors='coerce').fillna(0)
        
        all_data.append(is_data)
        
        print(f"✓ {os.path.basename(file_path)} - {entity} ({currency}) 처리 완료")
        
    except Exception as e:
        print(f"✗ {os.path.basename(file_path)} 처리 실패: {str(e)}")
    
    return all_data

def consolidate_financial_data(input_folder, output_file, entity_currency_map):
    """여러 엑셀 파일을 통합"""
    all_results = []
    
    # 폴더 내 모든 엑셀 파일 처리
    excel_files = list(Path(input_folder).glob('*.xlsx'))
    excel_files = [f for f in excel_files if not f.name.startswith('~')]  # 임시파일 제외
    
    print(f"총 {len(excel_files)}개 파일 발견\n")
    
    for file_path in sorted(excel_files):
        data_list = extract_data_from_file(file_path, entity_currency_map)
        all_results.extend(data_list)
    
    # 모든 데이터 통합
    if all_results:
        consolidated_df = pd.concat(all_results, ignore_index=True)
        
        # 컬럼 순서 조정 (Entity, Statement, Currency, Period를 앞으로)
        cols = ['Entity', 'Statement', 'Currency', 'Period'] + [col for col in consolidated_df.columns if col not in ['Entity', 'Statement', 'Currency', 'Period']]
        consolidated_df = consolidated_df[cols]
        
        # 결과 저장
        consolidated_df.to_excel(output_file, index=False)
        
        print(f"\n{'='*60}")
        print(f"[Step 1] 통합 완료!")
        print(f"총 {len(consolidated_df)} 행의 데이터")
        print(f"저장 위치: {output_file}")
        print(f"Entity 개수: {consolidated_df['Entity'].nunique()}")
        print(f"{'='*60}")
        
        # 요약 정보
        print("\n[Entity별 데이터 건수]")
        print(consolidated_df.groupby(['Entity', 'Statement', 'Currency', 'Period']).size())
        
        # Unknown Currency 확인
        unknown_currency = consolidated_df[consolidated_df['Currency'] == 'Unknown']['Entity'].unique()
        if len(unknown_currency) > 0:
            print(f"\n⚠️  통화 정보가 없는 Entity: {list(unknown_currency)}")
            print("   → ENTITY_CURRENCY_MAP에 추가해주세요!")
        
        return consolidated_df
    else:
        print("처리된 데이터가 없습니다.")
        return None

def process_final_file(input_file, output_file, entity_currency_map):
    """
    consolidated_financial_data.xlsx를 읽어서 
    Amount, Exchange_Rate, Amount(KRW) 추가 및 PL 시트 생성
    """
    print("\n" + "="*60)
    print("[Step 2] 통화 및 환율 적용 중...")
    print("="*60)
    
    # 데이터 읽기
    df = pd.read_excel(input_file)
    
    print(f"✓ 데이터 로드 완료: {len(df)} 행")
    print(f"✓ 전체 컬럼: {list(df.columns)}")
    
    # 1. Amount 열 추가 - 각 행의 Currency에 해당하는 컬럼 값
    # 통화 컬럼 매핑 (Currency와 실제 컬럼명이 다를 수 있음)
    통화_컬럼_매핑 = {
        'AUD': 'AUD',
        'KRW': 'KRW',
        'JPY': 'JPY',
        'USD': 'USD',
        'CNY': 'CNH',  # Currency는 CNY지만 컬럼명은 CNH
        'CNH': 'CNH',  # CNH도 허용
        'MXN': 'MXN',
        'EUR': 'EUR',
        'VND': 'VND',
        'MYR': 'MYR',
        'INR': 'INR',
        'TRY': 'TRY',
    }
    
    def get_amount(row):
        currency = row['Currency']
        col_name = 통화_컬럼_매핑.get(currency)
        
        # 매핑된 컬럼이 있고 해당 컬럼이 존재하면
        if col_name and col_name in df.columns:
            value = row[col_name]
            return 0 if pd.isna(value) else float(value)
        else:
            # 통화 컬럼이 없으면 모든 통화 컬럼 확인해서 값이 있는 것 사용
            통화_컬럼들 = ['AUD', 'KRW', 'JPY', 'USD', 'CNH', 'MXN', 'EUR', 'VND', 'MYR', 'INR', 'TRY']
            for col in 통화_컬럼들:
                if col in df.columns and pd.notna(row[col]) and row[col] != 0:
                    return float(row[col])
            return 0
    
    df['Amount'] = df.apply(get_amount, axis=1)
    
    print(f"✓ Amount 생성 완료")
    print(f"  - Amount 합계: {df['Amount'].sum():,.2f}")
    print(f"  - Amount > 0인 행: {(df['Amount'] > 0).sum()} / {len(df)}")
    
    # 2. Entity와 Statement별 Exchange_Rate 매핑
    def get_exchange_rate(row):
        entity = row['Entity']
        statement = row['Statement']
        entity_info = entity_currency_map.get(entity, {})
        
        # BS는 bs_rate, IS(PL)는 pl_rate 사용
        if statement == 'BS':
            return entity_info.get('bs_rate', 1.0)
        else:  # IS
            return entity_info.get('pl_rate', 1.0)
    
    df['Exchange_Rate'] = df.apply(get_exchange_rate, axis=1)
    
    # 3. Amount(KRW) 계산 (소숫점 반올림)
    df['Amount(KRW)'] = (df['Amount'] * df['Exchange_Rate']).round(0).astype('Int64')
    
    print(f"  - Amount(KRW) 합계: {df['Amount(KRW)'].sum():,.0f}")
    
    # 컬럼 순서 조정
    base_cols = ['Entity', 'Statement', 'Currency', 'Period']
    data_cols = [col for col in df.columns if col not in base_cols + ['Amount', 'Exchange_Rate', 'Amount(KRW)']]
    final_cols = base_cols + data_cols + ['Amount', 'Exchange_Rate', 'Amount(KRW)']
    
    df = df[final_cols]
    
    # Sheet1 저장
    with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
        df.to_excel(writer, sheet_name='Sheet1', index=False)
    
    print(f"\n✓ Sheet1 저장 완료: {output_file}")
    
    # 통화별 요약
    print("\n[Entity별 통화 및 총액]")
    summary = df.groupby(['Entity', 'Statement', 'Currency']).agg({
        'Amount': 'sum',
        'Amount(KRW)': 'sum',
        'Exchange_Rate': 'first'  # 환율 표시
    }).round(0)
    print(summary)
    
    print(f"\n{'='*60}")
    print(f"[Step 2] 완료!")
    print(f"총 {len(df)} 행의 데이터")
    print(f"저장 위치: {output_file}")
    print(f"{'='*60}")
    
    # BS와 PL 환율 확인
    print("\n[BS 환율 매핑 상태]")
    bs_data = df[df['Statement'] == 'BS'][['Entity', 'Currency', 'Exchange_Rate']].drop_duplicates()
    for _, row in bs_data.iterrows():
        print(f"✓ {row['Entity']} ({row['Currency']}) - {row['Exchange_Rate']} (기말환율)")
    
    print("\n[PL 환율 매핑 상태]")
    pl_data = df[df['Statement'] == 'IS'][['Entity', 'Currency', 'Exchange_Rate']].drop_duplicates()
    for _, row in pl_data.iterrows():
        print(f"✓ {row['Entity']} ({row['Currency']}) - {row['Exchange_Rate']} (평균환율)")
    
    return df

def create_pl_sheet(final_file):
    """
    final_financial_data.xlsx에 PL 시트 추가
    """
    print("\n" + "="*60)
    print("[Step 3] PL 시트 생성 중...")
    print("="*60)
    
    # final_financial_data.xlsx의 Sheet1 읽기
    df = pd.read_excel(final_file, sheet_name='Sheet1')
    
    # 1. Statement = "IS"만 필터링
    df_pl = df[df['Statement'] == 'IS'].copy()
    print(f"✓ IS 데이터 필터링: {len(df_pl)} 행")
    
    # 2. 특정 컬럼만 선택
    selected_cols = ['Entity', 'Statement', 'Currency', 'Period', 'Code', '계정과목', 'Account', 
                     'Amount', 'Exchange_Rate', 'Amount(KRW)']
    df_pl = df_pl[selected_cols]
    
    # 3. 특정 code 행 삭제
    codes_to_remove = [40000, 50000, 59000, 60000, 69000, 71000, 72000, 73000, 74000, 
                       75000, 79000, 80000, 81000, 82000, 83000, 84000]
    df_pl = df_pl[~df_pl['Code'].isin(codes_to_remove)]
    print(f"✓ 특정 Code 삭제 후: {len(df_pl)} 행")
    
    # 4. Rev_Account 컬럼 생성
    def get_rev_account(code):
        code_str = str(code)
        
        if code_str.startswith('4'):
            return '매출'
        elif code_str.startswith('5'):
            return '매출원가'
        elif code in [60001, 60002, 60003, 60004, 60005, 60006, 60034]:
            return '인건비'
        elif code == 60007:
            return '복리후생비'
        elif code == 60008:
            return '여비교통비'
        elif code == 60015:
            return '광고선전비'
        elif code == 60017:
            return '운반비'
        elif code == 60018:
            return '지급수수료'
        elif code == 60019:
            return '경상연구개발비'
        elif code == 60027:
            return '판매수수료'
        elif code in [60021, 60029]:
            return '감가상각비'
        elif code == 60033:
            return '사용권자산상각비'
        elif code == 60022:
            return '대손상각비'
        elif code_str.startswith('60'):
            return '기타'
        elif code_str.startswith('71'):
            return '영업외수익'
        elif code_str.startswith('72'):
            return '영업외비용'
        elif code_str.startswith('73'):
            return '금융수익'
        elif code_str.startswith('74'):
            return '금융비용'
        elif code_str.startswith('75'):
            return '지분법손익'
        elif code_str.startswith('80'):
            return '법인세비용'
        elif code_str.startswith('81'):
            return '세후중단영업손익'
        elif code_str.startswith('82'):
            return '당기순이익'
        elif code_str.startswith('83'):
            return '포괄손익'
        elif code_str.startswith('84'):
            return '총포괄손익'
        else:
            return '기타'
    
    df_pl['Rev_Account'] = df_pl['Code'].apply(get_rev_account)
    
    # 컬럼 순서 조정 (Rev_Account를 Account 다음에)
    final_cols = ['Entity', 'Statement', 'Currency', 'Period', 'Code', '계정과목', 'Account', 
                  'Rev_Account', 'Amount', 'Exchange_Rate', 'Amount(KRW)']
    df_pl = df_pl[final_cols]
    
    # PL 시트 추가
    with pd.ExcelWriter(final_file, engine='openpyxl', mode='a') as writer:
        df_pl.to_excel(writer, sheet_name='PL', index=False)
    
    print(f"✓ PL 시트 저장 완료: {final_file}")
    
    # Rev_Account별 요약
    print("\n[Rev_Account별 집계]")
    summary = df_pl.groupby('Rev_Account').agg({
        'Amount(KRW)': 'sum',
        'Code': 'count'
    }).round(0)
    summary.columns = ['Amount(KRW)', '행수']
    summary = summary.sort_values('Amount(KRW)', ascending=False)
    print(summary)
    
    print(f"\n{'='*60}")
    print(f"[Step 3] 완료!")
    print(f"{'='*60}")
    
    return df_pl

# 실행
if __name__ == "__main__":
    # 경로 설정
    input_folder = r"C:\Users\user\InBody Co., Ltd\회계팀 - General\★★Accounting Folder★★vF\1. 재무결산\FY2025\2025_03_1Q\2. 연결재무제표\3. Reporting PKG"
    consolidated_file = r"C:\Users\user\InBody Co., Ltd\회계팀 - General\★★Accounting Folder★★vF\1. 재무결산\FY2025\2025_12_4Q\2. 연결재무제표\Income statement\consolidated_financial_data_25Q1.xlsx"
    final_file = r"C:\Users\user\InBody Co., Ltd\회계팀 - General\★★Accounting Folder★★vF\1. 재무결산\FY2025\2025_12_4Q\2. 연결재무제표\Income statement\final_financial_data_25Q1.xlsx"
    
    print("="*60)
    print("해외법인 재무제표 데이터 통합 시작")
    print("="*60)
    
    # Step 1: 데이터 통합
    result_df = consolidate_financial_data(input_folder, consolidated_file, ENTITY_CURRENCY_MAP)
    
    if result_df is not None:
        # Step 2: 환율 적용
        final_df = process_final_file(consolidated_file, final_file, ENTITY_CURRENCY_MAP)
        
        if final_df is not None:
            # Step 3: PL 시트 생성
            pl_df = create_pl_sheet(final_file)
            
            print("\n" + "="*60)
            print("✅ 모든 작업이 완료되었습니다!")
            print("="*60)
            print(f"\n생성된 파일:")
            print(f"  1. {consolidated_file}")
            print(f"  2. {final_file} (Sheet1 + PL)")
        else:
            print("\n❌ Step 2 실패")
    else:
        print("\n❌ Step 1 실패")

해외법인 재무제표 데이터 통합 시작
총 16개 파일 발견



C:\Users\user\AppData\Local\Temp\ipykernel_21576\2115207701.py:104: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['<Hide>' '계정과목' '매출액' '매출원가' '매출총이익' '판매비와관리비' '영업이익' '기타수익' '기타비용'
 '금융수익' '금융비용' '지분법손익' '법인세비용차감전이익' '법인세비용' '세후중단영업손익' '당기순이익' '포괄손익'
 '총포괄손익']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[mask, 10] = df.loc[mask, col]  # K
C:\Users\user\AppData\Local\Temp\ipykernel_21576\2115207701.py:109: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['Account' 'Sales' 'Cost of Goods Sold ' 'Gross Profit'
 'Selling and Administration Expense' 'Operating Income' 'Other Revenue'
 'Other Expense' 'Financial Revenue' 'Financial Expense'
 'Gain and Loss on Equity Method' 'Net Income before Corporate Income Tax'
 'Corporate Income Tax' 'Gain and Loss from discontinued operations '
 'Net Income '

✓ 1. Inbody K-IFRS Package_HQ_2025_1Q_Draft(250501).xlsx - 주식회사 인바디 (KRW) 처리 완료


C:\Users\user\AppData\Local\Temp\ipykernel_21576\2115207701.py:104: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['<Hide>' '계정과목' '매출액' '매출원가' '매출총이익' '판매비와관리비' '영업이익' '기타수익' '기타비용'
 '금융수익' '금융비용' '지분법손익' '법인세비용차감전이익' '법인세비용' '세후중단영업손익' '당기순이익' '포괄손익'
 '총포괄손익']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[mask, 10] = df.loc[mask, col]  # K
C:\Users\user\AppData\Local\Temp\ipykernel_21576\2115207701.py:109: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['Account' 'Sales' 'Cost of Goods Sold ' 'Gross Profit'
 'Selling and Administration Expense' 'Operating Income' 'Other Revenue'
 'Other Expense' 'Financial Revenue' 'Financial Expense'
 'Gain and Loss on Equity Method' 'Net Income before Corporate Income Tax'
 'Corporate Income Tax' 'Gain and Loss from discontinued operations '
 'Net Income '

✓ 10. Inbody K-IFRS Package_KOROT_2025_1Q_Draft(250501).xlsx - ㈜코르트 (KRW) 처리 완료


...

✓ 7. Inbody K-IFRS Package_Europe_2025_1Q_Draft(250422).xlsx - Inbody Europe B.V. (EUR) 처리 완료


C:\Users\user\AppData\Local\Temp\ipykernel_21576\2115207701.py:104: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['<Hide>' '계정과목' '매출액' '매출원가' '매출총이익' '판매비와관리비' '영업이익' '기타수익' '기타비용'
 '금융수익' '금융비용' '지분법손익' '법인세비용차감전이익' '법인세비용' '세후중단영업손익' '당기순이익' '포괄손익'
 '총포괄손익']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[mask, 10] = df.loc[mask, col]  # K
C:\Users\user\AppData\Local\Temp\ipykernel_21576\2115207701.py:109: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['Account' 'Sales' 'Cost of Goods Sold ' 'Gross Profit'
 'Selling and Administration Expense' 'Operating Income' 'Other Revenue'
 'Other Expense' 'Financial Revenue' 'Financial Expense'
 'Gain and Loss on Equity Method' 'Net Income before Corporate Income Tax'
 'Corporate Income Tax' 'Gain and Loss from discontinued operations '
 'Net Income '

✓ 8. Inbody K-IFRS Package_삼한정공_2025_1Q_Draft(250429).xlsx - (주)삼한정공 (KRW) 처리 완료


C:\Users\user\AppData\Local\Temp\ipykernel_21576\2115207701.py:104: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['<Hide>' '계정과목' '매출액' '매출원가' '매출총이익' '판매비와관리비' '영업이익' '기타수익' '기타비용'
 '금융수익' '금융비용' '지분법손익' '법인세비용차감전이익' '법인세비용' '세후중단영업손익' '당기순이익' '포괄손익'
 '총포괄손익']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[mask, 10] = df.loc[mask, col]  # K
C:\Users\user\AppData\Local\Temp\ipykernel_21576\2115207701.py:109: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['Account' 'Sales' 'Cost of Goods Sold ' 'Gross Profit'
 'Selling and Administration Expense' 'Operating Income' 'Other Revenue'
 'Other Expense' 'Financial Revenue' 'Financial Expense'
 'Gain and Loss on Equity Method' 'Net Income before Corporate Income Tax'
 'Corporate Income Tax' 'Gain and Loss from discontinued operations '
 'Net Income '

✓ 9. Inbody K-IFRS Package_KOCP_2025_1Q_Draft(250421).xlsx - 케이오씨피 프로젝트 제2호 벤처투자조합 (KRW) 처리 완료

[Step 1] 통합 완료!
총 4960 행의 데이터
저장 위치: C:\Users\user\InBody Co., Ltd\회계팀 - General\★★Accounting Folder★★vF\1. 재무결산\FY2025\2025_12_4Q\2. 연결재무제표\Income statement\consolidated_financial_data_25Q1.xlsx
Entity 개수: 16

[Entity별 데이터 건수]
Entity                                   Statement  Currency  Period
(주)삼한정공                                  BS         KRW       20251Q    195
                                         IS         KRW       20251Q    115
ADJ                                      BS         KRW       20251Q    195
                                         IS         KRW       20251Q    115
BIOSPACE LATIN AMERICA S DE RL DE CV(*)  BS         MXN       20251Q    195
                                         IS         MXN       20251Q    115
BWA                                      BS         USD       20251Q    195
                                         IS         USD       20251Q    115

In [7]:
import pandas as pd
import os
from pathlib import Path

# ==================== 여기를 수정하세요! ====================
# 기간 설정
PERIOD = "20244Q"  # 예: "2025_4Q", "2025_12월", "FY2025" 등

# Entity별 통화 및 환율 매핑 (BS: 기말환율, PL: 평균환율)
ENTITY_CURRENCY_MAP = {
    'Biospace Inc. DBA INBODY': {
        'currency': 'USD', 
        'bs_rate': 1470,  # BS 기말환율
        'pl_rate': 1363.98   # PL 평균환율
    },
    'InBody Japan Inc.': {
        'currency': 'JPY', 
        'bs_rate': 9.3648,
        'pl_rate': 9.0036
    },
    'Biospace Co.,Ltd.': {
        'currency': 'CNH', 
        'bs_rate': 201.27,
        'pl_rate': 189.20
    },
    'InBody Oceania ': {
        'currency': 'AUD', 
        'bs_rate': 913.68,
        'pl_rate': 899.64
    },
    'BIOSPACE LATIN AMERICA S DE RL DE CV(*)': {
        'currency': 'MXN', 
        'bs_rate': 71.17,
        'pl_rate': 74.79
    },
    'BWA': {
        'currency': 'USD', 
        'bs_rate': 1470,
        'pl_rate': 1363.98
    },
    '㈜인바디헬스케어': {
        'currency': 'KRW', 
        'bs_rate': 1.0,
        'pl_rate': 1.0
    },
    'InBody Vietnam': {
        'currency': 'VND', 
        'bs_rate': 0.0577,
        'pl_rate': 0.0544
    },
    'INBODY ASIA SDN. BHD.': {
        'currency': 'MYR', 
        'bs_rate': 329.23,
        'pl_rate': 298.33
    },
    'InBody India Pvt. Ltd': {
        'currency': 'INR', 
        'bs_rate': 17.19,
        'pl_rate': 16.30
    },
    'Inbody Europe B.V.': {
        'currency': 'EUR', 
        'bs_rate': 1528.73,
        'pl_rate': 1475.05
    },
    '㈜코르트': {
        'currency': 'KRW', 
        'bs_rate': 1.0,
        'pl_rate': 1.0
    },
    'INBODY TURKEY MEDİKAL TİCARET LİMİTED ŞİRKETİ': {
        'currency': 'TRY', 
        'bs_rate': 33.40,
        'pl_rate': 36.05
    },
    '(주)삼한정공': {
        'currency': 'KRW', 
        'bs_rate': 1.0,
        'pl_rate': 1.0
    },
    '주식회사 인바디': {
        'currency': 'KRW', 
        'bs_rate': 1.0,
        'pl_rate': 1.0
    },
    '케이오씨피 프로젝트 제2호 벤처투자조합': {
        'currency': 'KRW', 
        'bs_rate': 1.0,
        'pl_rate': 1.0
    },
    'ADJ': {
        'currency': 'KRW', 
        'bs_rate': 1.0,
        'pl_rate': 1.0
    },    
}
# ============================================================

def preprocess_sheet(df, sheet_type):
    """데이터 전처리: H~J → K, M~O → P로 이동"""
    # H~J 열 데이터를 K로 이동 (0-indexed: 7,8,9 → 10)
    for col in [7, 8, 9]:  # H, I, J
        mask = df[col].notna()
        df.loc[mask, 10] = df.loc[mask, col]  # K
    
    # M~O 열 데이터를 P로 이동 (0-indexed: 12,13,14 → 15)
    for col in [12, 13, 14]:  # M, N, O
        mask = df[col].notna()
        df.loc[mask, 15] = df.loc[mask, col]  # P
    
    return df

def extract_data_from_file(file_path, entity_currency_map):
    """단일 엑셀 파일에서 BS, IS 데이터 추출"""
    all_data = []
    
    try:
        # BS 시트 처리
        bs_df = pd.read_excel(file_path, sheet_name='BS', header=None)
        entity = bs_df.iloc[0, 6]  # G1 (0-indexed: row 0, col 6)
        
        # Entity별 통화 정보 가져오기 (ENTITY_CURRENCY_MAP에서)
        currency = entity_currency_map.get(entity, {}).get('currency', 'Unknown')
        
        # 전처리
        bs_df = preprocess_sheet(bs_df, 'BS')
        
        # 6행(index 5)을 컬럼명으로, 7~231행(index 6~230) 데이터 추출
        bs_data = bs_df.iloc[6:231, [6, 10, 15, 22]]  # G, K, P, W (0-indexed: 6, 10, 15, 22)
        bs_data.columns = bs_df.iloc[5, [6, 10, 15, 22]].values  # 6행을 컬럼명으로
        
        bs_data['Entity'] = entity
        bs_data['Statement'] = 'BS'
        bs_data['Currency'] = currency
        bs_data['Period'] = PERIOD
        
        # 공란 제거 (모든 값이 NaN인 행 제거)
        bs_data = bs_data.dropna(how='all', subset=bs_data.columns[:-4])  # Entity, Statement, Currency, Period 제외
        
        # K열(두 번째 열)에 값이 없는 행 삭제
        k_col_name = bs_data.columns[1]  # K열 컬럼명
        bs_data = bs_data[bs_data[k_col_name].notna()]
        
        # W열만 숫자형으로 변환 (공란을 0으로, G/K/P는 텍스트 유지)
        w_col_name = bs_data.columns[3]  # W열 컬럼명
        bs_data[w_col_name] = pd.to_numeric(bs_data[w_col_name], errors='coerce').fillna(0)
        
        all_data.append(bs_data)
        
        # IS 시트 처리
        is_df = pd.read_excel(file_path, sheet_name='IS', header=None)
        entity = is_df.iloc[0, 6]  # G1
        
        # Entity별 통화 정보 가져오기 (ENTITY_CURRENCY_MAP에서)
        currency = entity_currency_map.get(entity, {}).get('currency', 'Unknown')
        
        # 전처리
        is_df = preprocess_sheet(is_df, 'IS')
        
        # 6행을 컬럼명으로, 7~129행(index 6~128) 데이터 추출
        is_data = is_df.iloc[6:129, [6, 10, 15, 22]]  # G, K, P, W
        is_data.columns = is_df.iloc[5, [6, 10, 15, 22]].values
        
        is_data['Entity'] = entity
        is_data['Statement'] = 'IS'
        is_data['Currency'] = currency
        is_data['Period'] = PERIOD
        
        # 공란 제거
        is_data = is_data.dropna(how='all', subset=is_data.columns[:-4])  # Entity, Statement, Currency, Period 제외
        
        # K열(두 번째 열)에 값이 없는 행 삭제
        k_col_name = is_data.columns[1]  # K열 컬럼명
        is_data = is_data[is_data[k_col_name].notna()]
        
        # W열만 숫자형으로 변환 (공란을 0으로, G/K/P는 텍스트 유지)
        w_col_name = is_data.columns[3]  # W열 컬럼명
        is_data[w_col_name] = pd.to_numeric(is_data[w_col_name], errors='coerce').fillna(0)
        
        all_data.append(is_data)
        
        print(f"✓ {os.path.basename(file_path)} - {entity} ({currency}) 처리 완료")
        
    except Exception as e:
        print(f"✗ {os.path.basename(file_path)} 처리 실패: {str(e)}")
    
    return all_data

def consolidate_financial_data(input_folder, output_file, entity_currency_map):
    """여러 엑셀 파일을 통합"""
    all_results = []
    
    # 폴더 내 모든 엑셀 파일 처리
    excel_files = list(Path(input_folder).glob('*.xlsx'))
    excel_files = [f for f in excel_files if not f.name.startswith('~')]  # 임시파일 제외
    
    print(f"총 {len(excel_files)}개 파일 발견\n")
    
    for file_path in sorted(excel_files):
        data_list = extract_data_from_file(file_path, entity_currency_map)
        all_results.extend(data_list)
    
    # 모든 데이터 통합
    if all_results:
        consolidated_df = pd.concat(all_results, ignore_index=True)
        
        # 컬럼 순서 조정 (Entity, Statement, Currency, Period를 앞으로)
        cols = ['Entity', 'Statement', 'Currency', 'Period'] + [col for col in consolidated_df.columns if col not in ['Entity', 'Statement', 'Currency', 'Period']]
        consolidated_df = consolidated_df[cols]
        
        # 결과 저장
        consolidated_df.to_excel(output_file, index=False)
        
        print(f"\n{'='*60}")
        print(f"[Step 1] 통합 완료!")
        print(f"총 {len(consolidated_df)} 행의 데이터")
        print(f"저장 위치: {output_file}")
        print(f"Entity 개수: {consolidated_df['Entity'].nunique()}")
        print(f"{'='*60}")
        
        # 요약 정보
        print("\n[Entity별 데이터 건수]")
        print(consolidated_df.groupby(['Entity', 'Statement', 'Currency', 'Period']).size())
        
        # Unknown Currency 확인
        unknown_currency = consolidated_df[consolidated_df['Currency'] == 'Unknown']['Entity'].unique()
        if len(unknown_currency) > 0:
            print(f"\n⚠️  통화 정보가 없는 Entity: {list(unknown_currency)}")
            print("   → ENTITY_CURRENCY_MAP에 추가해주세요!")
        
        return consolidated_df
    else:
        print("처리된 데이터가 없습니다.")
        return None

def process_final_file(input_file, output_file, entity_currency_map):
    """
    consolidated_financial_data.xlsx를 읽어서 
    Amount, Exchange_Rate, Amount(KRW) 추가 및 PL 시트 생성
    """
    print("\n" + "="*60)
    print("[Step 2] 통화 및 환율 적용 중...")
    print("="*60)
    
    # 데이터 읽기
    df = pd.read_excel(input_file)
    
    print(f"✓ 데이터 로드 완료: {len(df)} 행")
    print(f"✓ 전체 컬럼: {list(df.columns)}")
    
    # 1. Amount 열 추가 - 각 행의 Currency에 해당하는 컬럼 값
    # 통화 컬럼 매핑 (Currency와 실제 컬럼명이 다를 수 있음)
    통화_컬럼_매핑 = {
        'AUD': 'AUD',
        'KRW': 'KRW',
        'JPY': 'JPY',
        'USD': 'USD',
        'CNY': 'CNH',  # Currency는 CNY지만 컬럼명은 CNH
        'CNH': 'CNH',  # CNH도 허용
        'MXN': 'MXN',
        'EUR': 'EUR',
        'VND': 'VND',
        'MYR': 'MYR',
        'INR': 'INR',
        'TRY': 'TRY',
    }
    
    def get_amount(row):
        currency = row['Currency']
        col_name = 통화_컬럼_매핑.get(currency)
        
        # 매핑된 컬럼이 있고 해당 컬럼이 존재하면
        if col_name and col_name in df.columns:
            value = row[col_name]
            return 0 if pd.isna(value) else float(value)
        else:
            # 통화 컬럼이 없으면 모든 통화 컬럼 확인해서 값이 있는 것 사용
            통화_컬럼들 = ['AUD', 'KRW', 'JPY', 'USD', 'CNH', 'MXN', 'EUR', 'VND', 'MYR', 'INR', 'TRY']
            for col in 통화_컬럼들:
                if col in df.columns and pd.notna(row[col]) and row[col] != 0:
                    return float(row[col])
            return 0
    
    df['Amount'] = df.apply(get_amount, axis=1)
    
    print(f"✓ Amount 생성 완료")
    print(f"  - Amount 합계: {df['Amount'].sum():,.2f}")
    print(f"  - Amount > 0인 행: {(df['Amount'] > 0).sum()} / {len(df)}")
    
    # 2. Entity와 Statement별 Exchange_Rate 매핑
    def get_exchange_rate(row):
        entity = row['Entity']
        statement = row['Statement']
        entity_info = entity_currency_map.get(entity, {})
        
        # BS는 bs_rate, IS(PL)는 pl_rate 사용
        if statement == 'BS':
            return entity_info.get('bs_rate', 1.0)
        else:  # IS
            return entity_info.get('pl_rate', 1.0)
    
    df['Exchange_Rate'] = df.apply(get_exchange_rate, axis=1)
    
    # 3. Amount(KRW) 계산 (소숫점 반올림)
    df['Amount(KRW)'] = (df['Amount'] * df['Exchange_Rate']).round(0).astype('Int64')
    
    print(f"  - Amount(KRW) 합계: {df['Amount(KRW)'].sum():,.0f}")
    
    # 컬럼 순서 조정
    base_cols = ['Entity', 'Statement', 'Currency', 'Period']
    data_cols = [col for col in df.columns if col not in base_cols + ['Amount', 'Exchange_Rate', 'Amount(KRW)']]
    final_cols = base_cols + data_cols + ['Amount', 'Exchange_Rate', 'Amount(KRW)']
    
    df = df[final_cols]
    
    # Sheet1 저장
    with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
        df.to_excel(writer, sheet_name='Sheet1', index=False)
    
    print(f"\n✓ Sheet1 저장 완료: {output_file}")
    
    # 통화별 요약
    print("\n[Entity별 통화 및 총액]")
    summary = df.groupby(['Entity', 'Statement', 'Currency']).agg({
        'Amount': 'sum',
        'Amount(KRW)': 'sum',
        'Exchange_Rate': 'first'  # 환율 표시
    }).round(0)
    print(summary)
    
    print(f"\n{'='*60}")
    print(f"[Step 2] 완료!")
    print(f"총 {len(df)} 행의 데이터")
    print(f"저장 위치: {output_file}")
    print(f"{'='*60}")
    
    # BS와 PL 환율 확인
    print("\n[BS 환율 매핑 상태]")
    bs_data = df[df['Statement'] == 'BS'][['Entity', 'Currency', 'Exchange_Rate']].drop_duplicates()
    for _, row in bs_data.iterrows():
        print(f"✓ {row['Entity']} ({row['Currency']}) - {row['Exchange_Rate']} (기말환율)")
    
    print("\n[PL 환율 매핑 상태]")
    pl_data = df[df['Statement'] == 'IS'][['Entity', 'Currency', 'Exchange_Rate']].drop_duplicates()
    for _, row in pl_data.iterrows():
        print(f"✓ {row['Entity']} ({row['Currency']}) - {row['Exchange_Rate']} (평균환율)")
    
    return df

def create_pl_sheet(final_file):
    """
    final_financial_data.xlsx에 PL 시트 추가
    """
    print("\n" + "="*60)
    print("[Step 3] PL 시트 생성 중...")
    print("="*60)
    
    # final_financial_data.xlsx의 Sheet1 읽기
    df = pd.read_excel(final_file, sheet_name='Sheet1')
    
    # 1. Statement = "IS"만 필터링
    df_pl = df[df['Statement'] == 'IS'].copy()
    print(f"✓ IS 데이터 필터링: {len(df_pl)} 행")
    
    # 2. 특정 컬럼만 선택
    selected_cols = ['Entity', 'Statement', 'Currency', 'Period', 'Code', '계정과목', 'Account', 
                     'Amount', 'Exchange_Rate', 'Amount(KRW)']
    df_pl = df_pl[selected_cols]
    
    # 3. 특정 code 행 삭제
    codes_to_remove = [40000, 50000, 59000, 60000, 69000, 71000, 72000, 73000, 74000, 
                       75000, 79000, 80000, 81000, 82000, 83000, 84000]
    df_pl = df_pl[~df_pl['Code'].isin(codes_to_remove)]
    print(f"✓ 특정 Code 삭제 후: {len(df_pl)} 행")
    
    # 4. Rev_Account 컬럼 생성
    def get_rev_account(code):
        code_str = str(code)
        
        if code_str.startswith('4'):
            return '매출'
        elif code_str.startswith('5'):
            return '매출원가'
        elif code in [60001, 60002, 60003, 60004, 60005, 60006, 60034]:
            return '인건비'
        elif code == 60007:
            return '복리후생비'
        elif code == 60008:
            return '여비교통비'
        elif code == 60015:
            return '광고선전비'
        elif code == 60017:
            return '운반비'
        elif code == 60018:
            return '지급수수료'
        elif code == 60019:
            return '경상연구개발비'
        elif code == 60027:
            return '판매수수료'
        elif code in [60021, 60029]:
            return '감가상각비'
        elif code == 60033:
            return '사용권자산상각비'
        elif code == 60022:
            return '대손상각비'
        elif code_str.startswith('60'):
            return '기타'
        elif code_str.startswith('71'):
            return '영업외수익'
        elif code_str.startswith('72'):
            return '영업외비용'
        elif code_str.startswith('73'):
            return '금융수익'
        elif code_str.startswith('74'):
            return '금융비용'
        elif code_str.startswith('75'):
            return '지분법손익'
        elif code_str.startswith('80'):
            return '법인세비용'
        elif code_str.startswith('81'):
            return '세후중단영업손익'
        elif code_str.startswith('82'):
            return '당기순이익'
        elif code_str.startswith('83'):
            return '포괄손익'
        elif code_str.startswith('84'):
            return '총포괄손익'
        else:
            return '기타'
    
    df_pl['Rev_Account'] = df_pl['Code'].apply(get_rev_account)
    
    # 컬럼 순서 조정 (Rev_Account를 Account 다음에)
    final_cols = ['Entity', 'Statement', 'Currency', 'Period', 'Code', '계정과목', 'Account', 
                  'Rev_Account', 'Amount', 'Exchange_Rate', 'Amount(KRW)']
    df_pl = df_pl[final_cols]
    
    # PL 시트 추가
    with pd.ExcelWriter(final_file, engine='openpyxl', mode='a') as writer:
        df_pl.to_excel(writer, sheet_name='PL', index=False)
    
    print(f"✓ PL 시트 저장 완료: {final_file}")
    
    # Rev_Account별 요약
    print("\n[Rev_Account별 집계]")
    summary = df_pl.groupby('Rev_Account').agg({
        'Amount(KRW)': 'sum',
        'Code': 'count'
    }).round(0)
    summary.columns = ['Amount(KRW)', '행수']
    summary = summary.sort_values('Amount(KRW)', ascending=False)
    print(summary)
    
    print(f"\n{'='*60}")
    print(f"[Step 3] 완료!")
    print(f"{'='*60}")
    
    return df_pl

# 실행
if __name__ == "__main__":
    # 경로 설정
    input_folder = r"C:\Users\user\InBody Co., Ltd\회계팀 - General\★★Accounting Folder★★vF\1. 재무결산\FY2024\2024_12_4Q\2. 연결재무제표\3. Reporting PKG"
    consolidated_file = r"C:\Users\user\InBody Co., Ltd\회계팀 - General\★★Accounting Folder★★vF\1. 재무결산\FY2025\2025_12_4Q\2. 연결재무제표\Income statement\consolidated_financial_data_24Q4.xlsx"
    final_file = r"C:\Users\user\InBody Co., Ltd\회계팀 - General\★★Accounting Folder★★vF\1. 재무결산\FY2025\2025_12_4Q\2. 연결재무제표\Income statement\final_financial_data_24Q4.xlsx"
    
    print("="*60)
    print("해외법인 재무제표 데이터 통합 시작")
    print("="*60)
    
    # Step 1: 데이터 통합
    result_df = consolidate_financial_data(input_folder, consolidated_file, ENTITY_CURRENCY_MAP)
    
    if result_df is not None:
        # Step 2: 환율 적용
        final_df = process_final_file(consolidated_file, final_file, ENTITY_CURRENCY_MAP)
        
        if final_df is not None:
            # Step 3: PL 시트 생성
            pl_df = create_pl_sheet(final_file)
            
            print("\n" + "="*60)
            print("✅ 모든 작업이 완료되었습니다!")
            print("="*60)
            print(f"\n생성된 파일:")
            print(f"  1. {consolidated_file}")
            print(f"  2. {final_file} (Sheet1 + PL)")
        else:
            print("\n❌ Step 2 실패")
    else:
        print("\n❌ Step 1 실패")

해외법인 재무제표 데이터 통합 시작
총 16개 파일 발견



C:\Users\user\AppData\Local\Temp\ipykernel_21576\2084618191.py:104: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['<Hide>' '계정과목' '매출액' '매출원가' '매출총이익' '판매비와관리비' '영업이익' '기타수익' '기타비용'
 '금융수익' '금융비용' '지분법손익' '법인세비용차감전이익' '법인세비용' '세후중단영업손익' '당기순이익' '포괄손익'
 '총포괄손익']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[mask, 10] = df.loc[mask, col]  # K
C:\Users\user\AppData\Local\Temp\ipykernel_21576\2084618191.py:109: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['Account' 'Sales' 'Cost of Goods Sold ' 'Gross Profit'
 'Selling and Administration Expense' 'Operating Income' 'Other Revenue'
 'Other Expense' 'Financial Revenue' 'Financial Expense'
 'Gain and Loss on Equity Method' 'Net Income before Corporate Income Tax'
 'Corporate Income Tax' 'Gain and Loss from discontinued operations '
 'Net Income '

✓ 1. Inbody K-IFRS Package_HQ_2024_4Q_Draft(260107).xlsx - 주식회사 인바디 (KRW) 처리 완료


C:\Users\user\AppData\Local\Temp\ipykernel_21576\2084618191.py:104: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['<Hide>' '계정과목' '매출액' '매출원가' '매출총이익' '판매비와관리비' '영업이익' '기타수익' '기타비용'
 '금융수익' '금융비용' '지분법손익' '법인세비용차감전이익' '법인세비용' '세후중단영업손익' '당기순이익' '포괄손익'
 '총포괄손익']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[mask, 10] = df.loc[mask, col]  # K
C:\Users\user\AppData\Local\Temp\ipykernel_21576\2084618191.py:109: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['Account' 'Sales' 'Cost of Goods Sold ' 'Gross Profit'
 'Selling and Administration Expense' 'Operating Income' 'Other Revenue'
 'Other Expense' 'Financial Revenue' 'Financial Expense'
 'Gain and Loss on Equity Method' 'Net Income before Corporate Income Tax'
 'Corporate Income Tax' 'Gain and Loss from discontinued operations '
 'Net Income '

✓ 10. Inbody K-IFRS Package_코르트_2024_4Q_Final(250218).xlsx - ㈜코르트 (KRW) 처리 완료


...

✓ 7. Inbody K-IFRS Package_Europe_2024_4Q_Final(250215).xlsx - Inbody Europe B.V. (EUR) 처리 완료


C:\Users\user\AppData\Local\Temp\ipykernel_21576\2084618191.py:104: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['<Hide>' '계정과목' '매출액' '매출원가' '매출총이익' '판매비와관리비' '영업이익' '기타수익' '기타비용'
 '금융수익' '금융비용' '지분법손익' '법인세비용차감전이익' '법인세비용' '세후중단영업손익' '당기순이익' '포괄손익'
 '총포괄손익']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[mask, 10] = df.loc[mask, col]  # K
C:\Users\user\AppData\Local\Temp\ipykernel_21576\2084618191.py:109: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['Account' 'Sales' 'Cost of Goods Sold ' 'Gross Profit'
 'Selling and Administration Expense' 'Operating Income' 'Other Revenue'
 'Other Expense' 'Financial Revenue' 'Financial Expense'
 'Gain and Loss on Equity Method' 'Net Income before Corporate Income Tax'
 'Corporate Income Tax' 'Gain and Loss from discontinued operations '
 'Net Income '

✓ 8. Inbody K-IFRS Package_삼한정공_2024_4Q_Final(250127).xlsx - (주)삼한정공 (KRW) 처리 완료


C:\Users\user\AppData\Local\Temp\ipykernel_21576\2084618191.py:104: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['<Hide>' '계정과목' '매출액' '매출원가' '매출총이익' '판매비와관리비' '영업이익' '기타수익' '기타비용'
 '금융수익' '금융비용' '지분법손익' '법인세비용차감전이익' '법인세비용' '세후중단영업손익' '당기순이익' '포괄손익'
 '총포괄손익']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[mask, 10] = df.loc[mask, col]  # K
C:\Users\user\AppData\Local\Temp\ipykernel_21576\2084618191.py:109: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['Account' 'Sales' 'Cost of Goods Sold ' 'Gross Profit'
 'Selling and Administration Expense' 'Operating Income' 'Other Revenue'
 'Other Expense' 'Financial Revenue' 'Financial Expense'
 'Gain and Loss on Equity Method' 'Net Income before Corporate Income Tax'
 'Corporate Income Tax' 'Gain and Loss from discontinued operations '
 'Net Income '

✓ 9. Inbody K-IFRS Package_KOCP_2024_4Q_Final(250204).xlsx - 케이오씨피 프로젝트 제2호 벤처투자조합 (KRW) 처리 완료

[Step 1] 통합 완료!
총 4960 행의 데이터
저장 위치: C:\Users\user\InBody Co., Ltd\회계팀 - General\★★Accounting Folder★★vF\1. 재무결산\FY2025\2025_12_4Q\2. 연결재무제표\Income statement\consolidated_financial_data_24Q4.xlsx
Entity 개수: 16

[Entity별 데이터 건수]
Entity                                   Statement  Currency  Period
(주)삼한정공                                  BS         KRW       20244Q    195
                                         IS         KRW       20244Q    115
ADJ                                      BS         KRW       20244Q    195
                                         IS         KRW       20244Q    115
BIOSPACE LATIN AMERICA S DE RL DE CV(*)  BS         MXN       20244Q    195
                                         IS         MXN       20244Q    115
BWA                                      BS         USD       20244Q    195
                                         IS         USD       20244Q    115

In [3]:
import pandas as pd
import os
from pathlib import Path

# ==================== 여기를 수정하세요! ====================
# 기간 설정
PERIOD = "20243Q"  # 예: "2025_4Q", "2025_12월", "FY2025" 등

# Entity별 통화 및 환율 매핑 (BS: 기말환율, PL: 평균환율)
ENTITY_CURRENCY_MAP = {
    'Biospace Inc. DBA INBODY': {
        'currency': 'USD', 
        'bs_rate': 1319.60,
        'pl_rate': 1352.85 
    },
    'InBody Japan Inc.': {
        'currency': 'JPY', 
        'bs_rate': 9.2451,
        'pl_rate': 8.9499
    },
    'Biospace Co.,Ltd.': {
        'currency': 'CNH', 
        'bs_rate': 188.74,
        'pl_rate': 187.57
    },
    'InBody Oceania ': {
        'currency': 'AUD', 
        'bs_rate': 912.44,
        'pl_rate': 895.91
    },
    'BIOSPACE LATIN AMERICA S DE RL DE CV(*)': {
        'currency': 'MXN', 
        'bs_rate': 67.07,
        'pl_rate': 76.58
    },
    'BWA': {
        'currency': 'USD', 
        'bs_rate': 1319.60,
        'pl_rate': 1352.85
    },
    '㈜인바디헬스케어': {
        'currency': 'KRW', 
        'bs_rate': 1.0,
        'pl_rate': 1.0
    },
    'InBody Vietnam': {
        'currency': 'VND', 
        'bs_rate': 0.0536,
        'pl_rate': 0.0541
    },
    'INBODY ASIA SDN. BHD.': {
        'currency': 'MYR', 
        'bs_rate': 319.90,
        'pl_rate': 291.83
    },
    'InBody India Pvt. Ltd': {
        'currency': 'INR', 
        'bs_rate': 15.76,
        'pl_rate': 16.22
    },
    'Inbody Europe B.V.': {
        'currency': 'EUR', 
        'bs_rate': 1474.06,
        'pl_rate': 1470.30
    },
    '㈜코르트': {
        'currency': 'KRW', 
        'bs_rate': 1.0,
        'pl_rate': 1.0
    },
    'INBODY TURKEY MEDİKAL TİCARET LİMİTED ŞİRKETİ': {
        'currency': 'TRY', 
        'bs_rate': 33.40,
        'pl_rate': 36.05
    },
    '(주)삼한정공': {
        'currency': 'KRW', 
        'bs_rate': 1.0,
        'pl_rate': 1.0
    },
    '주식회사 인바디': {
        'currency': 'KRW', 
        'bs_rate': 1.0,
        'pl_rate': 1.0
    },
    '케이오씨피 프로젝트 제2호 벤처투자조합': {
        'currency': 'KRW', 
        'bs_rate': 1.0,
        'pl_rate': 1.0
    },
    'ADJ': {
        'currency': 'KRW', 
        'bs_rate': 1.0,
        'pl_rate': 1.0
    },
}
# ============================================================

def preprocess_sheet(df, sheet_type):
    """데이터 전처리: H~J → K, M~O → P로 이동"""
    # H~J 열 데이터를 K로 이동 (0-indexed: 7,8,9 → 10)
    for col in [7, 8, 9]:  # H, I, J
        mask = df[col].notna()
        df.loc[mask, 10] = df.loc[mask, col]  # K
    
    # M~O 열 데이터를 P로 이동 (0-indexed: 12,13,14 → 15)
    for col in [12, 13, 14]:  # M, N, O
        mask = df[col].notna()
        df.loc[mask, 15] = df.loc[mask, col]  # P
    
    return df

def extract_data_from_file(file_path, entity_currency_map):
    """단일 엑셀 파일에서 BS, IS 데이터 추출"""
    all_data = []
    
    try:
        # BS 시트 처리
        bs_df = pd.read_excel(file_path, sheet_name='BS', header=None)
        entity = bs_df.iloc[0, 6]  # G1 (0-indexed: row 0, col 6)
        
        # Entity별 통화 정보 가져오기 (ENTITY_CURRENCY_MAP에서)
        currency = entity_currency_map.get(entity, {}).get('currency', 'Unknown')
        
        # 전처리
        bs_df = preprocess_sheet(bs_df, 'BS')
        
        # 6행(index 5)을 컬럼명으로, 7~231행(index 6~230) 데이터 추출
        bs_data = bs_df.iloc[6:231, [6, 10, 15, 22]]  # G, K, P, W (0-indexed: 6, 10, 15, 22)
        bs_data.columns = bs_df.iloc[5, [6, 10, 15, 22]].values  # 6행을 컬럼명으로
        
        bs_data['Entity'] = entity
        bs_data['Statement'] = 'BS'
        bs_data['Currency'] = currency
        bs_data['Period'] = PERIOD
        
        # 공란 제거 (모든 값이 NaN인 행 제거)
        bs_data = bs_data.dropna(how='all', subset=bs_data.columns[:-4])  # Entity, Statement, Currency, Period 제외
        
        # K열(두 번째 열)에 값이 없는 행 삭제
        k_col_name = bs_data.columns[1]  # K열 컬럼명
        bs_data = bs_data[bs_data[k_col_name].notna()]
        
        # W열만 숫자형으로 변환 (공란을 0으로, G/K/P는 텍스트 유지)
        w_col_name = bs_data.columns[3]  # W열 컬럼명
        bs_data[w_col_name] = pd.to_numeric(bs_data[w_col_name], errors='coerce').fillna(0)
        
        all_data.append(bs_data)
        
        # IS 시트 처리
        is_df = pd.read_excel(file_path, sheet_name='IS', header=None)
        entity = is_df.iloc[0, 6]  # G1
        
        # Entity별 통화 정보 가져오기 (ENTITY_CURRENCY_MAP에서)
        currency = entity_currency_map.get(entity, {}).get('currency', 'Unknown')
        
        # 전처리
        is_df = preprocess_sheet(is_df, 'IS')
        
        # 6행을 컬럼명으로, 7~129행(index 6~128) 데이터 추출
        is_data = is_df.iloc[6:129, [6, 10, 15, 22]]  # G, K, P, W
        is_data.columns = is_df.iloc[5, [6, 10, 15, 22]].values
        
        is_data['Entity'] = entity
        is_data['Statement'] = 'IS'
        is_data['Currency'] = currency
        is_data['Period'] = PERIOD
        
        # 공란 제거
        is_data = is_data.dropna(how='all', subset=is_data.columns[:-4])  # Entity, Statement, Currency, Period 제외
        
        # K열(두 번째 열)에 값이 없는 행 삭제
        k_col_name = is_data.columns[1]  # K열 컬럼명
        is_data = is_data[is_data[k_col_name].notna()]
        
        # W열만 숫자형으로 변환 (공란을 0으로, G/K/P는 텍스트 유지)
        w_col_name = is_data.columns[3]  # W열 컬럼명
        is_data[w_col_name] = pd.to_numeric(is_data[w_col_name], errors='coerce').fillna(0)
        
        all_data.append(is_data)
        
        print(f"✓ {os.path.basename(file_path)} - {entity} ({currency}) 처리 완료")
        
    except Exception as e:
        print(f"✗ {os.path.basename(file_path)} 처리 실패: {str(e)}")
    
    return all_data

def consolidate_financial_data(input_folder, output_file, entity_currency_map):
    """여러 엑셀 파일을 통합"""
    all_results = []
    
    # 폴더 내 모든 엑셀 파일 처리
    excel_files = list(Path(input_folder).glob('*.xlsx'))
    excel_files = [f for f in excel_files if not f.name.startswith('~')]  # 임시파일 제외
    
    print(f"총 {len(excel_files)}개 파일 발견\n")
    
    for file_path in sorted(excel_files):
        data_list = extract_data_from_file(file_path, entity_currency_map)
        all_results.extend(data_list)
    
    # 모든 데이터 통합
    if all_results:
        consolidated_df = pd.concat(all_results, ignore_index=True)
        
        # 컬럼 순서 조정 (Entity, Statement, Currency, Period를 앞으로)
        cols = ['Entity', 'Statement', 'Currency', 'Period'] + [col for col in consolidated_df.columns if col not in ['Entity', 'Statement', 'Currency', 'Period']]
        consolidated_df = consolidated_df[cols]
        
        # 결과 저장
        consolidated_df.to_excel(output_file, index=False)
        
        print(f"\n{'='*60}")
        print(f"[Step 1] 통합 완료!")
        print(f"총 {len(consolidated_df)} 행의 데이터")
        print(f"저장 위치: {output_file}")
        print(f"Entity 개수: {consolidated_df['Entity'].nunique()}")
        print(f"{'='*60}")
        
        # 요약 정보
        print("\n[Entity별 데이터 건수]")
        print(consolidated_df.groupby(['Entity', 'Statement', 'Currency', 'Period']).size())
        
        # Unknown Currency 확인
        unknown_currency = consolidated_df[consolidated_df['Currency'] == 'Unknown']['Entity'].unique()
        if len(unknown_currency) > 0:
            print(f"\n⚠️  통화 정보가 없는 Entity: {list(unknown_currency)}")
            print("   → ENTITY_CURRENCY_MAP에 추가해주세요!")
        
        return consolidated_df
    else:
        print("처리된 데이터가 없습니다.")
        return None

def process_final_file(input_file, output_file, entity_currency_map):
    """
    consolidated_financial_data.xlsx를 읽어서 
    Amount, Exchange_Rate, Amount(KRW) 추가 및 PL 시트 생성
    """
    print("\n" + "="*60)
    print("[Step 2] 통화 및 환율 적용 중...")
    print("="*60)
    
    # 데이터 읽기
    df = pd.read_excel(input_file)
    
    print(f"✓ 데이터 로드 완료: {len(df)} 행")
    print(f"✓ 전체 컬럼: {list(df.columns)}")
    
    # 1. Amount 열 추가 - 각 행의 Currency에 해당하는 컬럼 값
    # 통화 컬럼 매핑 (Currency와 실제 컬럼명이 다를 수 있음)
    통화_컬럼_매핑 = {
        'AUD': 'AUD',
        'KRW': 'KRW',
        'JPY': 'JPY',
        'USD': 'USD',
        'CNY': 'CNH',  # Currency는 CNY지만 컬럼명은 CNH
        'CNH': 'CNH',  # CNH도 허용
        'MXN': 'MXN',
        'EUR': 'EUR',
        'VND': 'VND',
        'MYR': 'MYR',
        'INR': 'INR',
        'TRY': 'TRY',
    }
    
    def get_amount(row):
        currency = row['Currency']
        col_name = 통화_컬럼_매핑.get(currency)
        
        # 매핑된 컬럼이 있고 해당 컬럼이 존재하면
        if col_name and col_name in df.columns:
            value = row[col_name]
            return 0 if pd.isna(value) else float(value)
        else:
            # 통화 컬럼이 없으면 모든 통화 컬럼 확인해서 값이 있는 것 사용
            통화_컬럼들 = ['AUD', 'KRW', 'JPY', 'USD', 'CNH', 'MXN', 'EUR', 'VND', 'MYR', 'INR', 'TRY']
            for col in 통화_컬럼들:
                if col in df.columns and pd.notna(row[col]) and row[col] != 0:
                    return float(row[col])
            return 0
    
    df['Amount'] = df.apply(get_amount, axis=1)
    
    print(f"✓ Amount 생성 완료")
    print(f"  - Amount 합계: {df['Amount'].sum():,.2f}")
    print(f"  - Amount > 0인 행: {(df['Amount'] > 0).sum()} / {len(df)}")
    
    # 2. Entity와 Statement별 Exchange_Rate 매핑
    def get_exchange_rate(row):
        entity = row['Entity']
        statement = row['Statement']
        entity_info = entity_currency_map.get(entity, {})
        
        # BS는 bs_rate, IS(PL)는 pl_rate 사용
        if statement == 'BS':
            return entity_info.get('bs_rate', 1.0)
        else:  # IS
            return entity_info.get('pl_rate', 1.0)
    
    df['Exchange_Rate'] = df.apply(get_exchange_rate, axis=1)
    
    # 3. Amount(KRW) 계산 (소숫점 반올림)
    df['Amount(KRW)'] = (df['Amount'] * df['Exchange_Rate']).round(0).astype('Int64')
    
    print(f"  - Amount(KRW) 합계: {df['Amount(KRW)'].sum():,.0f}")
    
    # 컬럼 순서 조정
    base_cols = ['Entity', 'Statement', 'Currency', 'Period']
    data_cols = [col for col in df.columns if col not in base_cols + ['Amount', 'Exchange_Rate', 'Amount(KRW)']]
    final_cols = base_cols + data_cols + ['Amount', 'Exchange_Rate', 'Amount(KRW)']
    
    df = df[final_cols]
    
    # Sheet1 저장
    with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
        df.to_excel(writer, sheet_name='Sheet1', index=False)
    
    print(f"\n✓ Sheet1 저장 완료: {output_file}")
    
    # 통화별 요약
    print("\n[Entity별 통화 및 총액]")
    summary = df.groupby(['Entity', 'Statement', 'Currency']).agg({
        'Amount': 'sum',
        'Amount(KRW)': 'sum',
        'Exchange_Rate': 'first'  # 환율 표시
    }).round(0)
    print(summary)
    
    print(f"\n{'='*60}")
    print(f"[Step 2] 완료!")
    print(f"총 {len(df)} 행의 데이터")
    print(f"저장 위치: {output_file}")
    print(f"{'='*60}")
    
    # BS와 PL 환율 확인
    print("\n[BS 환율 매핑 상태]")
    bs_data = df[df['Statement'] == 'BS'][['Entity', 'Currency', 'Exchange_Rate']].drop_duplicates()
    for _, row in bs_data.iterrows():
        print(f"✓ {row['Entity']} ({row['Currency']}) - {row['Exchange_Rate']} (기말환율)")
    
    print("\n[PL 환율 매핑 상태]")
    pl_data = df[df['Statement'] == 'IS'][['Entity', 'Currency', 'Exchange_Rate']].drop_duplicates()
    for _, row in pl_data.iterrows():
        print(f"✓ {row['Entity']} ({row['Currency']}) - {row['Exchange_Rate']} (평균환율)")
    
    return df

def create_pl_sheet(final_file):
    """
    final_financial_data.xlsx에 PL 시트 추가
    """
    print("\n" + "="*60)
    print("[Step 3] PL 시트 생성 중...")
    print("="*60)
    
    # final_financial_data.xlsx의 Sheet1 읽기
    df = pd.read_excel(final_file, sheet_name='Sheet1')
    
    # 1. Statement = "IS"만 필터링
    df_pl = df[df['Statement'] == 'IS'].copy()
    print(f"✓ IS 데이터 필터링: {len(df_pl)} 행")
    
    # 2. 특정 컬럼만 선택
    selected_cols = ['Entity', 'Statement', 'Currency', 'Period', 'Code', '계정과목', 'Account', 
                     'Amount', 'Exchange_Rate', 'Amount(KRW)']
    df_pl = df_pl[selected_cols]
    
    # 3. 특정 code 행 삭제
    codes_to_remove = [40000, 50000, 59000, 60000, 69000, 71000, 72000, 73000, 74000, 
                       75000, 79000, 80000, 81000, 82000, 83000, 84000]
    df_pl = df_pl[~df_pl['Code'].isin(codes_to_remove)]
    print(f"✓ 특정 Code 삭제 후: {len(df_pl)} 행")
    
    # 4. Rev_Account 컬럼 생성
    def get_rev_account(code):
        code_str = str(code)
        
        if code_str.startswith('4'):
            return '매출'
        elif code_str.startswith('5'):
            return '매출원가'
        elif code in [60001, 60002, 60003, 60004, 60005, 60006, 60034]:
            return '인건비'
        elif code == 60007:
            return '복리후생비'
        elif code == 60008:
            return '여비교통비'
        elif code == 60015:
            return '광고선전비'
        elif code == 60017:
            return '운반비'
        elif code == 60018:
            return '지급수수료'
        elif code == 60019:
            return '경상연구개발비'
        elif code == 60027:
            return '판매수수료'
        elif code in [60021, 60029]:
            return '감가상각비'
        elif code == 60033:
            return '사용권자산상각비'
        elif code == 60022:
            return '대손상각비'
        elif code_str.startswith('60'):
            return '기타'
        elif code_str.startswith('71'):
            return '영업외수익'
        elif code_str.startswith('72'):
            return '영업외비용'
        elif code_str.startswith('73'):
            return '금융수익'
        elif code_str.startswith('74'):
            return '금융비용'
        elif code_str.startswith('75'):
            return '지분법손익'
        elif code_str.startswith('80'):
            return '법인세비용'
        elif code_str.startswith('81'):
            return '세후중단영업손익'
        elif code_str.startswith('82'):
            return '당기순이익'
        elif code_str.startswith('83'):
            return '포괄손익'
        elif code_str.startswith('84'):
            return '총포괄손익'
        else:
            return '기타'
    
    df_pl['Rev_Account'] = df_pl['Code'].apply(get_rev_account)
    
    # 컬럼 순서 조정 (Rev_Account를 Account 다음에)
    final_cols = ['Entity', 'Statement', 'Currency', 'Period', 'Code', '계정과목', 'Account', 
                  'Rev_Account', 'Amount', 'Exchange_Rate', 'Amount(KRW)']
    df_pl = df_pl[final_cols]
    
    # PL 시트 추가
    with pd.ExcelWriter(final_file, engine='openpyxl', mode='a') as writer:
        df_pl.to_excel(writer, sheet_name='PL', index=False)
    
    print(f"✓ PL 시트 저장 완료: {final_file}")
    
    # Rev_Account별 요약
    print("\n[Rev_Account별 집계]")
    summary = df_pl.groupby('Rev_Account').agg({
        'Amount(KRW)': 'sum',
        'Code': 'count'
    }).round(0)
    summary.columns = ['Amount(KRW)', '행수']
    summary = summary.sort_values('Amount(KRW)', ascending=False)
    print(summary)
    
    print(f"\n{'='*60}")
    print(f"[Step 3] 완료!")
    print(f"{'='*60}")
    
    return df_pl

# 실행
if __name__ == "__main__":
    # 경로 설정
    input_folder = r"C:\Users\user\InBody Co., Ltd\회계팀 - General\★★Accounting Folder★★vF\1. 재무결산\FY2024\2024_09_3Q\2. 연결재무제표\3. Reporting Package"
    consolidated_file = r"C:\Users\user\InBody Co., Ltd\회계팀 - General\★★Accounting Folder★★vF\1. 재무결산\FY2025\2025_12_4Q\2. 연결재무제표\Income statement\consolidated_financial_data_24Q3.xlsx"
    final_file = r"C:\Users\user\InBody Co., Ltd\회계팀 - General\★★Accounting Folder★★vF\1. 재무결산\FY2025\2025_12_4Q\2. 연결재무제표\Income statement\final_financial_data_24Q3.xlsx"
    
    print("="*60)
    print("해외법인 재무제표 데이터 통합 시작")
    print("="*60)
    
    # Step 1: 데이터 통합
    result_df = consolidate_financial_data(input_folder, consolidated_file, ENTITY_CURRENCY_MAP)
    
    if result_df is not None:
        # Step 2: 환율 적용
        final_df = process_final_file(consolidated_file, final_file, ENTITY_CURRENCY_MAP)
        
        if final_df is not None:
            # Step 3: PL 시트 생성
            pl_df = create_pl_sheet(final_file)
            
            print("\n" + "="*60)
            print("✅ 모든 작업이 완료되었습니다!")
            print("="*60)
            print(f"\n생성된 파일:")
            print(f"  1. {consolidated_file}")
            print(f"  2. {final_file} (Sheet1 + PL)")
        else:
            print("\n❌ Step 2 실패")
    else:
        print("\n❌ Step 1 실패")

해외법인 재무제표 데이터 통합 시작
총 16개 파일 발견



C:\Users\user\AppData\Local\Temp\ipykernel_21576\3789422268.py:104: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['<Hide>' '계정과목' '매출액' '매출원가' '매출총이익' '판매비와관리비' '영업이익' '기타수익' '기타비용'
 '금융수익' '금융비용' '지분법손익' '법인세비용차감전이익' '법인세비용' '세후중단영업손익' '당기순이익' '포괄손익'
 '총포괄손익']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[mask, 10] = df.loc[mask, col]  # K
C:\Users\user\AppData\Local\Temp\ipykernel_21576\3789422268.py:109: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['Account' 'Sales' 'Cost of Goods Sold ' 'Gross Profit'
 'Selling and Administration Expense' 'Operating Income' 'Other Revenue'
 'Other Expense' 'Financial Revenue' 'Financial Expense'
 'Gain and Loss on Equity Method' 'Net Income before Corporate Income Tax'
 'Corporate Income Tax' 'Gain and Loss from discontinued operations '
 'Net Income '

✓ 1. Inbody K-IFRS Package_HQ_2024_3Q_Draft(241028).xlsx - 주식회사 인바디 (KRW) 처리 완료


C:\Users\user\AppData\Local\Temp\ipykernel_21576\3789422268.py:104: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['<Hide>' '계정과목' '매출액' '매출원가' '매출총이익' '판매비와관리비' '영업이익' '기타수익' '기타비용'
 '금융수익' '금융비용' '지분법손익' '법인세비용차감전이익' '법인세비용' '세후중단영업손익' '당기순이익' '포괄손익'
 '총포괄손익']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[mask, 10] = df.loc[mask, col]  # K
C:\Users\user\AppData\Local\Temp\ipykernel_21576\3789422268.py:109: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['Account' 'Sales' 'Cost of Goods Sold ' 'Gross Profit'
 'Selling and Administration Expense' 'Operating Income' 'Other Revenue'
 'Other Expense' 'Financial Revenue' 'Financial Expense'
 'Gain and Loss on Equity Method' 'Net Income before Corporate Income Tax'
 'Corporate Income Tax' 'Gain and Loss from discontinued operations '
 'Net Income '

✓ 10. Inbody K-IFRS Package_코르트_2024_3Q_Draft(241022).xlsx - ㈜코르트 (KRW) 처리 완료


...

✓ 7. Inbody K-IFRS Package_Europe_2024_3Q_Draft(241022).xlsx - Inbody Europe B.V. (EUR) 처리 완료


C:\Users\user\AppData\Local\Temp\ipykernel_21576\3789422268.py:104: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['<Hide>' '계정과목' '매출액' '매출원가' '매출총이익' '판매비와관리비' '영업이익' '기타수익' '기타비용'
 '금융수익' '금융비용' '지분법손익' '법인세비용차감전이익' '법인세비용' '세후중단영업손익' '당기순이익' '포괄손익'
 '총포괄손익']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[mask, 10] = df.loc[mask, col]  # K
C:\Users\user\AppData\Local\Temp\ipykernel_21576\3789422268.py:109: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['Account' 'Sales' 'Cost of Goods Sold ' 'Gross Profit'
 'Selling and Administration Expense' 'Operating Income' 'Other Revenue'
 'Other Expense' 'Financial Revenue' 'Financial Expense'
 'Gain and Loss on Equity Method' 'Net Income before Corporate Income Tax'
 'Corporate Income Tax' 'Gain and Loss from discontinued operations '
 'Net Income '

✓ 8. Inbody K-IFRS Package_삼한정공_2024_3Q_Draft(241022).xlsx - (주)삼한정공 (KRW) 처리 완료


C:\Users\user\AppData\Local\Temp\ipykernel_21576\3789422268.py:104: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['<Hide>' '계정과목' '매출액' '매출원가' '매출총이익' '판매비와관리비' '영업이익' '기타수익' '기타비용'
 '금융수익' '금융비용' '지분법손익' '법인세비용차감전이익' '법인세비용' '세후중단영업손익' '당기순이익' '포괄손익'
 '총포괄손익']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[mask, 10] = df.loc[mask, col]  # K
C:\Users\user\AppData\Local\Temp\ipykernel_21576\3789422268.py:109: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['Account' 'Sales' 'Cost of Goods Sold ' 'Gross Profit'
 'Selling and Administration Expense' 'Operating Income' 'Other Revenue'
 'Other Expense' 'Financial Revenue' 'Financial Expense'
 'Gain and Loss on Equity Method' 'Net Income before Corporate Income Tax'
 'Corporate Income Tax' 'Gain and Loss from discontinued operations '
 'Net Income '

✓ 9. Inbody K-IFRS Package_KOCP_2024_3Q_Draft(241023).xlsx - 케이오씨피 프로젝트 제2호 벤처투자조합 (KRW) 처리 완료

[Step 1] 통합 완료!
총 4957 행의 데이터
저장 위치: C:\Users\user\InBody Co., Ltd\회계팀 - General\★★Accounting Folder★★vF\1. 재무결산\FY2025\2025_12_4Q\2. 연결재무제표\Income statement\consolidated_financial_data_24Q3.xlsx
Entity 개수: 16

[Entity별 데이터 건수]
Entity                                   Statement  Currency  Period
(주)삼한정공                                  BS         KRW       20243Q    195
                                         IS         KRW       20243Q    115
ADJ                                      BS         KRW       20243Q    195
                                         IS         KRW       20243Q    115
BIOSPACE LATIN AMERICA S DE RL DE CV(*)  BS         MXN       20243Q    195
                                         IS         MXN       20243Q    115
BWA                                      BS         USD       20243Q    192
                                         IS         USD       20243Q    115